In [1]:
!rm -rf /kaggle/working/Streaming_TTS
!rm -rf /kaggle/working/streaming_speech

In [2]:
!git clone https://github.com/Aakash-JavaDev/Streaming_TTS.git

Cloning into 'Streaming_TTS'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 33 (delta 6), reused 29 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 57.22 KiB | 1.10 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
from pathlib import Path
import json
import os

# ---------------------------------------------------------------------------
# EDIT ONLY THIS CELL BEFORE RUNNING THE NOTEBOOK.
# ---------------------------------------------------------------------------
RUN_STAGE = "prepare"  # preflight | prepare | qualify | train_full | evaluate | export
CONFIRM_FULL_TRAINING = ""  # set exactly to TRAIN_FULL_DATASET only for train_full
# First prove the adapter with the answer-only hidden states already stored in
# the encoded shards. Switch to "deployment_context" only in a later ablation.
FEATURE_MODE = "legacy_answer"  # legacy_answer | answer_only_recompute | deployment_context


INPUT_ROOT = Path("/kaggle/input/datasets/raja25/mimi-codes/dataset_question_answer_codes")
WORK_DIR = Path("/kaggle/working/streaming_speech")

def first_match(patterns, *, directory=False, required=True):
    for pattern in patterns:
        matches = sorted(
            path.resolve() for path in INPUT_ROOT.glob(pattern)
            if path.is_dir() if directory
        ) if directory else sorted(
            path.resolve() for path in INPUT_ROOT.glob(pattern) if path.is_file()
        )
        if matches:
            return str(matches[0])
    if required:
        raise FileNotFoundError(
            "Could not auto-discover a required Kaggle input. Attach the dataset, "
            f"then edit the matching path below. Patterns: {patterns}"
        )
    return ""

def discover_complete_data_dir():
    candidates = []
    for shard in INPUT_ROOT.glob("**/shard_*"):
        if shard.is_dir() and all((shard / name).is_file() for name in (
            "codes.npy", "offsets.npy", "hidden.npy", "hidden_offsets.npy",
            "token_ids.npy", "token_char_offsets.npy", "ids.jsonl",
            "alignment.jsonl", "pieces.jsonl",
        )):
            candidates.append(shard.parent.resolve())
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            "No complete encoded shard dataset found below /kaggle/input. "
            "Codes-only shards are not sufficient."
        )
    return str(candidates[0])

# Auto-discovery is intentionally deterministic. If more than one candidate exists,
# replace a value here with the exact /kaggle/input/... path shown by Kaggle.
DATA_DIR = discover_complete_data_dir()
SOURCE_JSONL = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/dataset_question_answer_pair.jsonl"
QWEN_MODEL = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Qwen2.5-1.5B-Instruct"
KOKORO_MODEL = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/onnx/model.onnx"
KOKORO_VOICES = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/voices/af_jessica.bin"
MIMI_ENCODER = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/encoder_model_8cb.onnx"
MIMI_DECODER = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/decoder_model_8cb.onnx"
MIMI_STREAMING_WEIGHTS = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/moshiko_pt_model/moshiko-pytorch-bf16/tokenizer-e351c8d8-checkpoint125.safetensors"


CONFIG = {
    "run_stage": RUN_STAGE,
    "confirm_full_training": CONFIRM_FULL_TRAINING,
    "feature_mode": FEATURE_MODE,
    "runtime": {
        "profile": "kaggle",
        "device": "auto",
        "require_two_gpus": True,
        "manifest_record_limit": 0,
        "cpu_threads": 2,
    },
    "paths": {
        "data_dir": DATA_DIR,
        "source_jsonl": SOURCE_JSONL,
        "qwen_model": QWEN_MODEL,
        "kokoro_model": KOKORO_MODEL,
        "kokoro_voices": KOKORO_VOICES,
        "mimi_encoder": MIMI_ENCODER,
        "mimi_decoder": MIMI_DECODER,
        "mimi_weights": MIMI_STREAMING_WEIGHTS,
        "work_dir": str(WORK_DIR),
    },
    "model": {
        "model_dim": 256,
        "temporal_layers": 2,
        "attention_heads": 4,
        "ffn_dim": 512,
        "text_context_size": 8,
        "audio_cache_frames": 64,
        "num_codebooks": 8,
        "codec_vocab": 2048,
        "dropout": 0.0,
    },
    "train": {
        "seed": 42,
        "epochs": 20,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "warmup_fraction": 0.05,
        "min_lr_ratio": 0.05,
        "grad_clip": 1.0,
        "duration_weight": 0.1,
        "cb0_weight": 4.0,
        "residual_weight": 1.0,
        "max_frames_per_gpu": 2500,
        "max_batch_size": 8,
        "grad_accum_steps": 1,
        "num_workers": 2,
        "val_fraction": 0.1,
        "eval_samples": 64,
        "eval_audio_samples": 2,
        "save_every": 1,
        "log_every": 25,
        "resume": True,
    },
    "gates": {
        "codec_samples": 20,
        "alignment_samples": 50,
        "alignment_pass_fraction": 0.95,
        "overfit_teacher_forced_ce": 0.05,
        "overfit_rollout_cb0_accuracy": 0.99,
        "text_control_margin": 0.05,
        "min_code_usage_ratio": 0.50,
        "min_entropy_ratio": 0.75,
        "max_duration_median_ape": 0.20,
        "max_device_p50_ms": 20.0,
        "max_device_p95_ms": 40.0,
        "max_device_worst_ms": 80.0,
        "max_first_audio_ms": 300.0,
        "max_int8_weight_bytes": 12582912,
        "max_working_memory_bytes": 33554432,
        "max_cache_bytes": 1048576,
    },
}

WORK_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK_DIR / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(json.dumps(CONFIG, indent=2))

{
  "run_stage": "prepare",
  "confirm_full_training": "",
  "feature_mode": "legacy_answer",
  "runtime": {
    "profile": "kaggle",
    "device": "auto",
    "require_two_gpus": true,
    "manifest_record_limit": 0,
    "cpu_threads": 2
  },
  "paths": {
    "data_dir": "/kaggle/input/datasets/raja25/mimi-codes/dataset_question_answer_codes",
    "source_jsonl": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/dataset_question_answer_pair.jsonl",
    "qwen_model": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Qwen2.5-1.5B-Instruct",
    "kokoro_model": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/onnx/model.onnx",
    "kokoro_voices": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/voices/af_jessica.bin",
    "mimi_encoder": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/encoder_model_8cb.onnx",
    "mimi_decoder": "/kaggle/input/datasets/raja25/lllama-q

In [4]:
import importlib
import importlib.metadata
import subprocess
import sys

# Keep Kaggle's preinstalled torch build: replacing it can break CUDA support.
try:
    import torch
except ImportError as exc:
    raise RuntimeError("Kaggle PyTorch is missing; use the standard Kaggle GPU image") from exc

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Notebook options select Accelerator: GPU T4 x2.")

BASE_REQUIREMENTS = {
    "safetensors": "safetensors==0.6.2",
    "transformers": "transformers==5.18.0",
    "soundfile": "soundfile==0.13.1",
    "matplotlib": "matplotlib==3.10.7",
}
OPTIONAL_BY_STAGE = {
    "qualify": {
        "onnx": "onnx==1.19.1",
        "onnxruntime": "onnxruntime-gpu==1.22.0",
        "kokoro_onnx": "kokoro-onnx==0.6.1",
        "moshi": "moshi==0.2.13",
    },
    "evaluate": {"onnxruntime": "onnxruntime-gpu==1.22.0"},
    "export": {
        "onnx": "onnx==1.19.1",
        "onnxruntime": "onnxruntime-gpu==1.22.0",
    },
}
requirements = dict(BASE_REQUIREMENTS)
requirements.update(OPTIONAL_BY_STAGE.get(RUN_STAGE, {}))
missing = []
for module, requirement in requirements.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(requirement)
if missing:
    print("Installing missing packages only:", missing)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing], text=True
    )
    if result.returncode:
        raise RuntimeError(
            "Dependency installation failed. Enable Internet in Kaggle Notebook options "
            "or attach a wheel dataset containing: " + " ".join(missing)
        )

print("Python", sys.version)
print("PyTorch", torch.__version__, "CUDA", torch.version.cuda)
for module in requirements:
    imported = importlib.import_module(module)
    distribution = "onnxruntime-gpu" if module == "onnxruntime" else module.replace("_", "-")
    try:
        version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        version = getattr(imported, "__version__", "unknown")
    print(f"{module}: {version}")


Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch 2.9.1+cu128 CUDA 12.8
safetensors: 0.7.0
transformers: 4.57.6
soundfile: 0.13.1
matplotlib: 3.10.0


In [5]:
from pathlib import Path
import json

SOURCES = {"__init__.py": "\"\"\"Streaming Qwen-to-Mimi adapter used by the Kaggle production notebook.\"\"\"\n\n__version__ = \"1.0.0\"\n\n", "__main__.py": "from .cli import main\n\nmain()\n", "cli.py": "from __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\n\nfrom .config import AppConfig, atomic_json\n\n\ndef _config(path: str, context_size: int = 0) -> AppConfig:\n    cfg = AppConfig.load(path)\n    if context_size:\n        cfg.model.text_context_size = int(context_size)\n        cfg.validate()\n    return cfg\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(\n        prog=\"speech-adapter\",\n        description=\"Local/Kaggle runner for the causal Qwen-to-Mimi speech adapter\",\n    )\n    parser.add_argument(\"--config\", required=True, help=\"resolved notebook JSON configuration\")\n    commands = parser.add_subparsers(dest=\"command\", required=True)\n    commands.add_parser(\"doctor\", help=\"validate configured hardware and data/model inputs\")\n    manifest = commands.add_parser(\"prepare-manifest\", help=\"validate shards and build manifest\")\n    manifest.add_argument(\"--force\", action=\"store_true\")\n    commands.add_parser(\"prepare-features\", help=\"prepare indexed or recomputed Qwen features\")\n    commands.add_parser(\"structural-checks\", help=\"run causal and CB0-gradient tests\")\n    commands.add_parser(\"smoke\", help=\"run one bounded CPU/CUDA forward/backward/cache check\")\n    commands.add_parser(\"local-smoke\", help=\"alias for smoke (device comes from runtime config)\")\n    commands.add_parser(\"codec-contract\", help=\"verify Mimi encoder/decoder codebook compatibility\")\n    commands.add_parser(\"gate-a\", help=\"run codec oracle diagnostics\")\n    commands.add_parser(\"gate-b\", help=\"render alignment review panels\")\n    approve = commands.add_parser(\"approve-gate\", help=\"record a manual gate review\")\n    approve.add_argument(\"--gate\", required=True, choices=list(\"ABCDEF\"))\n    approve.add_argument(\"--manual-pass\", required=True, choices=(\"yes\", \"no\"))\n    approve.add_argument(\"--reviewed\", type=int, default=0)\n    approve.add_argument(\"--passed\", type=int, default=0)\n    approve.add_argument(\"--note\", default=\"\")\n    train = commands.add_parser(\"train\", help=\"run DDP training or a qualification experiment\")\n    train.add_argument(\"--experiment\", default=\"full\")\n    train.add_argument(\"--limit\", type=int, default=0)\n    train.add_argument(\"--epochs\", type=int, default=0)\n    train.add_argument(\"--context-size\", type=int, default=0)\n    train.add_argument(\"--init-weights\", default=\"\")\n    evaluate = commands.add_parser(\"evaluate\", help=\"generate four-mode audio and metrics\")\n    evaluate.add_argument(\"--limit\", type=int, default=20)\n    export = commands.add_parser(\"export\", help=\"export and verify incremental ONNX graphs\")\n    export.add_argument(\"--parity-steps\", type=int, default=100)\n    export.add_argument(\"--context-size\", type=int, default=0)\n    export.add_argument(\"--checkpoint\", default=\"\")\n    export.add_argument(\"--output-dir\", default=\"\")\n    context = commands.add_parser(\"compare-context\", help=\"select K from completed context experiments\")\n    context.add_argument(\"--small\", default=\"E_k3\")\n    context.add_argument(\"--large\", default=\"E_k8\")\n    context.add_argument(\"--small-k\", type=int, default=3)\n    context.add_argument(\"--large-k\", type=int, default=8)\n    device = commands.add_parser(\"import-device-report\", help=\"validate an Android benchmark JSON\")\n    device.add_argument(\"--path\", required=True)\n    commands.add_parser(\"status\", help=\"show gate and run status\")\n    package = commands.add_parser(\"package\", help=\"zip final run artifacts\")\n    package.add_argument(\"--name\", default=\"streaming_speech_artifacts\")\n    return parser\n\n\ndef main(argv: list[str] | None = None) -> None:\n    args = build_parser().parse_args(argv)\n    cfg = _config(args.config, getattr(args, \"context_size\", 0))\n    if args.command == \"doctor\":\n        from .diagnostics import preflight\n\n        preflight(cfg, require_two_gpus=cfg.runtime.require_two_gpus)\n    elif args.command == \"prepare-manifest\":\n        from .data import build_manifest\n\n        print(json.dumps(build_manifest(cfg, force=args.force), indent=2))\n    elif args.command == \"prepare-features\":\n        from .data import prepare_qwen_features\n\n        prepare_qwen_features(cfg)\n    elif args.command == \"structural-checks\":\n        from .diagnostics import model_structural_checks\n\n        report = model_structural_checks(cfg)\n        print(json.dumps(report, indent=2))\n        if not report[\"automatic_pass\"]:\n            raise SystemExit(2)\n    elif args.command in {\"smoke\", \"local-smoke\"}:\n        from .smoke import run_smoke\n\n        report = run_smoke(cfg)\n        print(json.dumps(report, indent=2))\n        if not report[\"automatic_pass\"]:\n            raise SystemExit(2)\n    elif args.command == \"codec-contract\":\n        from .diagnostics import inspect_mimi_contract\n\n        report = inspect_mimi_contract(cfg)\n        print(json.dumps(report, indent=2))\n        if not report[\"automatic_pass\"]:\n            raise SystemExit(2)\n    elif args.command == \"gate-a\":\n        from .diagnostics import run_gate_a, update_gate\n\n        report = run_gate_a(cfg)\n        update_gate(cfg, \"A\", report[\"automatic_pass\"], None, note=\"review gates/A_codec\")\n        print(json.dumps(report, indent=2))\n    elif args.command == \"gate-b\":\n        from .diagnostics import run_gate_b, update_gate\n\n        report = run_gate_b(cfg)\n        update_gate(cfg, \"B\", report[\"automatic_pass\"], None, note=\"review gates/B_alignment/index.html\")\n        print(json.dumps(report, indent=2))\n    elif args.command == \"approve-gate\":\n        from .diagnostics import update_gate\n\n        path = cfg.work_dir / \"gates\" / \"gate_report.json\"\n        if not path.is_file():\n            raise FileNotFoundError(\"run the automatic gate before approving it\")\n        old = json.loads(path.read_text()).get(\"gates\", {}).get(args.gate)\n        if not old:\n            raise RuntimeError(f\"Gate {args.gate} has no automatic result\")\n        report = update_gate(\n            cfg,\n            args.gate,\n            bool(old[\"automatic_pass\"]),\n            args.manual_pass == \"yes\",\n            args.reviewed,\n            args.passed,\n            args.note,\n        )\n        print(json.dumps(report[\"gates\"][args.gate], indent=2))\n    elif args.command == \"train\":\n        from .train import run_training\n\n        result = run_training(\n            cfg,\n            limit=args.limit,\n            epochs=args.epochs or None,\n            experiment=args.experiment,\n            init_weights=args.init_weights,\n        )\n        if result:\n            print(json.dumps(result, indent=2))\n    elif args.command == \"evaluate\":\n        from .evaluation import generate_artifacts\n\n        result = generate_artifacts(cfg, args.limit)\n        print(json.dumps({key: value[\"rollout_cb0_ce\"] for key, value in result.items()}, indent=2))\n    elif args.command == \"export\":\n        from .export import export_adapter\n\n        print(json.dumps(export_adapter(\n            cfg, args.parity_steps, args.checkpoint, args.output_dir\n        ), indent=2))\n    elif args.command == \"compare-context\":\n        from .diagnostics import update_gate\n\n        small_path = cfg.work_dir / \"experiments\" / args.small / \"gate_metrics.json\"\n        large_path = cfg.work_dir / \"experiments\" / args.large / \"gate_metrics.json\"\n        small = json.loads(small_path.read_text())\n        large = json.loads(large_path.read_text())\n        small_ce = float(small[\"metrics\"][\"rollout_cb0_ce\"])\n        large_ce = float(large[\"metrics\"][\"rollout_cb0_ce\"])\n        improvement = (small_ce - large_ce) / max(small_ce, 1e-12)\n        selected = args.large_k if improvement >= 0.01 else args.small_k\n        report = {\n            \"small_experiment\": args.small,\n            \"large_experiment\": args.large,\n            \"small_cb0_ce\": small_ce,\n            \"large_cb0_ce\": large_ce,\n            \"relative_improvement\": improvement,\n            \"selected_context_size\": selected,\n            \"test_larger_context\": bool(improvement >= 0.01),\n        }\n        atomic_json(cfg.work_dir / \"gates\" / \"context_selection.json\", report)\n        chosen_report = large if selected == args.large_k else small\n        update_gate(\n            cfg,\n            \"E\",\n            bool(chosen_report[\"automatic_pass\"]),\n            None,\n            note=f\"selected context experiment: {args.large if selected == args.large_k else args.small}\",\n        )\n        print(json.dumps(report, indent=2))\n    elif args.command == \"import-device-report\":\n        source = json.loads(Path(args.path).read_text())\n        required = (\n            \"p50_ms\", \"p95_ms\", \"worst_ms\", \"first_audio_ms\", \"int8_weight_bytes\",\n            \"working_memory_bytes\", \"cache_bytes\", \"continuous_minutes\", \"thermal_throttling\",\n        )\n        missing = [key for key in required if key not in source]\n        if missing:\n            raise ValueError(f\"device report is missing fields: {missing}\")\n        checks = {\n            \"p50\": float(source[\"p50_ms\"]) < cfg.gates.max_device_p50_ms,\n            \"p95\": float(source[\"p95_ms\"]) < cfg.gates.max_device_p95_ms,\n            \"worst\": float(source[\"worst_ms\"]) <= cfg.gates.max_device_worst_ms,\n            \"first_audio\": float(source[\"first_audio_ms\"]) <= cfg.gates.max_first_audio_ms,\n            \"weights\": int(source[\"int8_weight_bytes\"]) <= cfg.gates.max_int8_weight_bytes,\n            \"working_memory\": int(source[\"working_memory_bytes\"]) <= cfg.gates.max_working_memory_bytes,\n            \"cache\": int(source[\"cache_bytes\"]) < cfg.gates.max_cache_bytes,\n            \"thermal\": float(source[\"continuous_minutes\"]) >= 5.0 and not bool(source[\"thermal_throttling\"]),\n        }\n        report = {**source, \"checks\": checks, \"automatic_pass\": all(checks.values())}\n        atomic_json(cfg.work_dir / \"gates\" / \"device_report.json\", report)\n        print(json.dumps(report, indent=2))\n        if not report[\"automatic_pass\"]:\n            raise SystemExit(2)\n    elif args.command == \"status\":\n        gate_path = cfg.work_dir / \"gates\" / \"gate_report.json\"\n        training_path = cfg.work_dir / \"training.json\"\n        print(\"gate report:\")\n        print(gate_path.read_text() if gate_path.is_file() else \"not created\")\n        print(\"training:\")\n        print(training_path.read_text() if training_path.is_file() else \"not started\")\n    elif args.command == \"package\":\n        destination = shutil.make_archive(\n            str(cfg.work_dir.parent / args.name), \"zip\", root_dir=cfg.work_dir\n        )\n        print(destination)\n\n\nif __name__ == \"__main__\":\n    main()\n", "config.py": "from __future__ import annotations\n\nimport hashlib\nimport json\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\n\n@dataclass\nclass PathsConfig:\n    data_dir: str\n    source_jsonl: str\n    qwen_model: str\n    kokoro_model: str = \"\"\n    kokoro_voices: str = \"\"\n    mimi_encoder: str = \"\"\n    mimi_decoder: str = \"\"\n    mimi_weights: str = \"\"\n    work_dir: str = \"/kaggle/working/streaming_speech\"\n\n\n@dataclass\nclass ModelConfig:\n    model_dim: int = 256\n    temporal_layers: int = 2\n    attention_heads: int = 4\n    ffn_dim: int = 512\n    text_context_size: int = 8\n    audio_cache_frames: int = 64\n    num_codebooks: int = 8\n    codec_vocab: int = 2048\n    dropout: float = 0.0\n\n\n@dataclass\nclass TrainConfig:\n    seed: int = 42\n    epochs: int = 20\n    learning_rate: float = 1.0e-4\n    weight_decay: float = 0.01\n    warmup_fraction: float = 0.05\n    min_lr_ratio: float = 0.05\n    grad_clip: float = 1.0\n    duration_weight: float = 0.1\n    cb0_weight: float = 4.0\n    residual_weight: float = 1.0\n    max_frames_per_gpu: int = 1200\n    max_batch_size: int = 8\n    grad_accum_steps: int = 1\n    num_workers: int = 2\n    val_fraction: float = 0.1\n    eval_samples: int = 64\n    eval_audio_samples: int = 2\n    save_every: int = 1\n    log_every: int = 25\n    resume: bool = True\n\n\n@dataclass\nclass GateConfig:\n    codec_samples: int = 20\n    alignment_samples: int = 50\n    alignment_pass_fraction: float = 0.95\n    overfit_teacher_forced_ce: float = 0.05\n    overfit_rollout_cb0_accuracy: float = 0.99\n    text_control_margin: float = 0.05\n    min_code_usage_ratio: float = 0.50\n    min_entropy_ratio: float = 0.75\n    max_duration_median_ape: float = 0.20\n    max_device_p50_ms: float = 20.0\n    max_device_p95_ms: float = 40.0\n    max_device_worst_ms: float = 80.0\n    max_first_audio_ms: float = 300.0\n    max_int8_weight_bytes: int = 12 * 1024 * 1024\n    max_working_memory_bytes: int = 32 * 1024 * 1024\n    max_cache_bytes: int = 1024 * 1024\n\n\n@dataclass\nclass RuntimeConfig:\n    \"\"\"Execution limits that do not change the adapter architecture.\"\"\"\n\n    profile: str = \"kaggle\"\n    device: str = \"auto\"\n    require_two_gpus: bool = True\n    manifest_record_limit: int = 0\n    cpu_threads: int = 2\n\n\n@dataclass\nclass AppConfig:\n    paths: PathsConfig\n    model: ModelConfig = field(default_factory=ModelConfig)\n    train: TrainConfig = field(default_factory=TrainConfig)\n    gates: GateConfig = field(default_factory=GateConfig)\n    runtime: RuntimeConfig = field(default_factory=RuntimeConfig)\n    feature_mode: str = \"legacy_answer\"\n    run_stage: str = \"preflight\"\n    confirm_full_training: str = \"\"\n\n    @classmethod\n    def load(cls, path: str | Path) -> \"AppConfig\":\n        source = Path(path)\n        raw = json.loads(source.read_text(encoding=\"utf-8\"))\n        cfg = cls(\n            paths=PathsConfig(**raw[\"paths\"]),\n            model=ModelConfig(**raw.get(\"model\", {})),\n            train=TrainConfig(**raw.get(\"train\", {})),\n            gates=GateConfig(**raw.get(\"gates\", {})),\n            runtime=RuntimeConfig(**raw.get(\"runtime\", {})),\n            feature_mode=str(raw.get(\"feature_mode\", \"legacy_answer\")),\n            run_stage=str(raw.get(\"run_stage\", \"preflight\")),\n            confirm_full_training=str(raw.get(\"confirm_full_training\", \"\")),\n        )\n        cfg.validate()\n        return cfg\n\n    def validate(self) -> None:\n        if self.run_stage not in {\"preflight\", \"prepare\", \"qualify\", \"train_full\", \"evaluate\", \"export\"}:\n            raise ValueError(f\"unsupported run_stage={self.run_stage!r}\")\n        if self.feature_mode not in {\n            \"legacy_answer\", \"answer_only_recompute\", \"deployment_context\"\n        }:\n            raise ValueError(\n                \"feature_mode must be 'legacy_answer', 'answer_only_recompute', \"\n                \"or 'deployment_context'\"\n            )\n        if self.runtime.profile not in {\"local_cpu\", \"kaggle\"}:\n            raise ValueError(\"runtime.profile must be 'local_cpu' or 'kaggle'\")\n        if self.runtime.device not in {\"auto\", \"cpu\", \"cuda\"}:\n            raise ValueError(\"runtime.device must be 'auto', 'cpu', or 'cuda'\")\n        if self.runtime.manifest_record_limit < 0:\n            raise ValueError(\"runtime.manifest_record_limit cannot be negative\")\n        if 0 < self.runtime.manifest_record_limit < 4:\n            raise ValueError(\"runtime.manifest_record_limit must be zero or at least four\")\n        if self.runtime.cpu_threads < 1:\n            raise ValueError(\"runtime.cpu_threads must be positive\")\n        if self.model.model_dim % self.model.attention_heads:\n            raise ValueError(\"model_dim must be divisible by attention_heads\")\n        if self.model.num_codebooks != 8 or self.model.codec_vocab != 2048:\n            raise ValueError(\"this dataset contract requires 8 codebooks with vocabulary 2048\")\n        if self.model.audio_cache_frames < 1 or self.model.text_context_size < 1:\n            raise ValueError(\"cache sizes must be positive\")\n        if not 0.0 < self.train.val_fraction < 1.0:\n            raise ValueError(\"val_fraction must be between 0 and 1\")\n        if self.train.max_frames_per_gpu < 1 or self.train.max_batch_size < 1:\n            raise ValueError(\"batch limits must be positive\")\n        if self.train.eval_samples < 1 or self.train.eval_audio_samples < 1:\n            raise ValueError(\"validation sample counts must be positive\")\n        if self.run_stage == \"train_full\" and self.confirm_full_training != \"TRAIN_FULL_DATASET\":\n            raise ValueError(\n                \"full training is locked; set confirm_full_training to TRAIN_FULL_DATASET\"\n            )\n\n    @property\n    def work_dir(self) -> Path:\n        return Path(self.paths.work_dir)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n    def fingerprint(self) -> str:\n        payload = json.dumps(self.to_dict(), sort_keys=True, separators=(\",\", \":\"))\n        return hashlib.sha256(payload.encode(\"utf-8\")).hexdigest()\n\n    def feature_fingerprint(self) -> str:\n        payload = {\n            \"feature_schema\": 2,\n            \"data_dir\": self.paths.data_dir,\n            \"source_jsonl\": self.paths.source_jsonl,\n            \"qwen_model\": self.paths.qwen_model,\n            \"feature_mode\": self.feature_mode,\n            \"manifest_record_limit\": self.runtime.manifest_record_limit,\n            \"seed\": self.train.seed,\n            \"val_fraction\": self.train.val_fraction,\n        }\n        dataset_report = self.work_dir / \"dataset_report.json\"\n        if dataset_report.is_file():\n            try:\n                payload[\"dataset_signature\"] = json.loads(\n                    dataset_report.read_text(encoding=\"utf-8\")\n                ).get(\"dataset_signature\")\n            except (OSError, json.JSONDecodeError):\n                payload[\"dataset_signature\"] = \"unreadable\"\n        raw = json.dumps(payload, sort_keys=True, separators=(\",\", \":\"))\n        return hashlib.sha256(raw.encode(\"utf-8\")).hexdigest()\n\n\ndef atomic_json(path: str | Path, value: Any) -> None:\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_suffix(target.suffix + \".tmp\")\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True), encoding=\"utf-8\")\n    temporary.replace(target)\n\n\ndef sha256_file(path: str | Path, chunk_size: int = 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open(\"rb\") as handle:\n        while True:\n            block = handle.read(chunk_size)\n            if not block:\n                break\n            digest.update(block)\n    return digest.hexdigest()\n", "data.py": "from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport random\nimport shutil\nimport time\nfrom collections import OrderedDict, defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterator, Sequence\n\nimport numpy as np\n\nfrom .config import AppConfig, atomic_json\n\nREQUIRED_SHARD_FILES = (\n    \"codes.npy\",\n    \"offsets.npy\",\n    \"hidden.npy\",\n    \"hidden_offsets.npy\",\n    \"token_ids.npy\",\n    \"token_char_offsets.npy\",\n    \"ids.jsonl\",\n    \"alignment.jsonl\",\n    \"pieces.jsonl\",\n)\n\n\ndef jsonl(path: str | Path) -> Iterator[Any]:\n    with Path(path).open(encoding=\"utf-8\") as handle:\n        for line_no, line in enumerate(handle, 1):\n            if not line.strip():\n                continue\n            try:\n                yield json.loads(line)\n            except json.JSONDecodeError as exc:\n                raise ValueError(f\"invalid JSON at {path}:{line_no}: {exc}\") from exc\n\n\ndef discover_shards(data_dir: str | Path) -> list[Path]:\n    root = Path(data_dir)\n    if not root.is_dir():\n        raise FileNotFoundError(f\"encoded shard directory does not exist: {root}\")\n    shards = sorted(path for path in root.glob(\"shard_*\") if path.is_dir())\n    complete = [s for s in shards if all((s / name).is_file() for name in REQUIRED_SHARD_FILES)]\n    if not complete:\n        missing = {\n            str(s): [name for name in REQUIRED_SHARD_FILES if not (s / name).is_file()]\n            for s in shards[:10]\n        }\n        raise FileNotFoundError(f\"no complete shards under {root}; missing={missing}\")\n    return complete\n\n\ndef _source_files(path: str | Path) -> list[Path]:\n    source = Path(path)\n    if source.is_file():\n        return [source]\n    if source.is_dir():\n        files = sorted(source.glob(\"*.jsonl\"))\n        if files:\n            return files\n    raise FileNotFoundError(f\"source JSONL file/directory does not exist or is empty: {source}\")\n\n\ndef source_rows(\n    path: str | Path, wanted_ids: set[str] | None = None\n) -> dict[str, dict[str, str]]:\n    rows: dict[str, dict[str, str]] = {}\n    fallback_id = 0\n    for source in _source_files(path):\n        for item in jsonl(source):\n            record_id = str(item.get(\"id\", fallback_id))\n            fallback_id += 1\n            if wanted_ids is not None and record_id not in wanted_ids:\n                continue\n            question = str(item.get(\"question\", \"\")).strip()\n            answer = str(item.get(\"answer\") or item.get(\"text\") or item.get(\"response\") or \"\").strip()\n            if question and answer:\n                rows[record_id] = {\"question\": question, \"answer\": answer}\n            if wanted_ids is not None and len(rows) == len(wanted_ids):\n                break\n        if wanted_ids is not None and len(rows) == len(wanted_ids):\n            break\n    if not rows:\n        raise RuntimeError(f\"no usable question/answer rows in {path}\")\n    return rows\n\n\ndef _split(record_id: str, seed: int, val_fraction: float) -> str:\n    token = hashlib.sha256(f\"{seed}:{record_id}\".encode()).digest()\n    value = int.from_bytes(token[:8], \"big\") / float(2**64)\n    return \"val\" if value < val_fraction else \"train\"\n\n\ndef _file_signature(path: Path) -> dict[str, Any]:\n    stat = path.stat()\n    return {\"path\": str(path), \"bytes\": stat.st_size, \"mtime_ns\": stat.st_mtime_ns}\n\n\ndef dataset_signature(shards: Sequence[Path], source_jsonl: str | Path) -> str:\n    payload = [_file_signature(path) for path in _source_files(source_jsonl)]\n    for shard in shards:\n        payload.extend(_file_signature(shard / name) for name in REQUIRED_SHARD_FILES)\n    raw = json.dumps(payload, sort_keys=True, separators=(\",\", \":\"))\n    return hashlib.sha256(raw.encode()).hexdigest()\n\n\ndef build_manifest(cfg: AppConfig, force: bool = False) -> dict[str, Any]:\n    \"\"\"Validate shards and write a compact, restart-safe record manifest.\"\"\"\n    work = cfg.work_dir\n    work.mkdir(parents=True, exist_ok=True)\n    manifest_path = work / \"manifest.jsonl\"\n    report_path = work / \"dataset_report.json\"\n    quarantine_path = work / \"quarantine.jsonl\"\n    shards = discover_shards(cfg.paths.data_dir)\n    signature = dataset_signature(shards, cfg.paths.source_jsonl)\n    record_limit = int(cfg.runtime.manifest_record_limit)\n    if not force and manifest_path.is_file() and report_path.is_file():\n        report = json.loads(report_path.read_text())\n        if (\n            report.get(\"dataset_signature\") == signature\n            and int(report.get(\"record_limit\", 0)) == record_limit\n            and report.get(\"complete\")\n        ):\n            print(f\"manifest already valid: {manifest_path}\")\n            return report\n\n    wanted_ids: set[str] | None = None\n    if record_limit:\n        wanted_ids = set()\n        candidate_budget = max(1024, record_limit * 20)\n        for shard in shards:\n            for raw_id in jsonl(shard / \"ids.jsonl\"):\n                wanted_ids.add(str(raw_id))\n                if len(wanted_ids) >= candidate_budget:\n                    break\n            if len(wanted_ids) >= candidate_budget:\n                break\n    sources = source_rows(cfg.paths.source_jsonl, wanted_ids=wanted_ids)\n    manifest_tmp = manifest_path.with_suffix(\".jsonl.tmp\")\n    quarantine_tmp = quarantine_path.with_suffix(\".jsonl.tmp\")\n    good = bad = frames = tokens = 0\n    split_counts = {\"train\": 0, \"val\": 0}\n    hidden_dim = None\n    validated_shards = 0\n    with manifest_tmp.open(\"w\", encoding=\"utf-8\") as output, quarantine_tmp.open(\n        \"w\", encoding=\"utf-8\"\n    ) as rejected:\n        for shard_no, shard in enumerate(shards, 1):\n            validated_shards += 1\n            offsets = np.load(shard / \"offsets.npy\", mmap_mode=\"r\")\n            hidden_offsets = np.load(shard / \"hidden_offsets.npy\", mmap_mode=\"r\")\n            codes = np.load(shard / \"codes.npy\", mmap_mode=\"r\")\n            hidden = np.load(shard / \"hidden.npy\", mmap_mode=\"r\")\n            token_ids = np.load(shard / \"token_ids.npy\", mmap_mode=\"r\")\n            char_offsets = np.load(shard / \"token_char_offsets.npy\", mmap_mode=\"r\")\n            ids = list(jsonl(shard / \"ids.jsonl\"))\n            alignments = list(jsonl(shard / \"alignment.jsonl\"))\n            pieces = list(jsonl(shard / \"pieces.jsonl\"))\n            n = len(ids)\n            errors = []\n            if offsets.ndim != 1 or len(offsets) != n + 1:\n                errors.append(f\"offsets shape {offsets.shape} for {n} rows\")\n            if hidden_offsets.ndim != 1 or len(hidden_offsets) != n + 1:\n                errors.append(f\"hidden_offsets shape {hidden_offsets.shape} for {n} rows\")\n            if len(alignments) != n or len(pieces) != n:\n                errors.append(\"JSONL row counts disagree\")\n            if codes.ndim != 2 or codes.shape[1] != cfg.model.num_codebooks:\n                errors.append(f\"codes must be [frames,8], got {codes.shape}\")\n            if hidden.ndim != 2:\n                errors.append(f\"hidden must be rank 2, got {hidden.shape}\")\n            if token_ids.ndim != 1 or char_offsets.ndim != 2 or char_offsets.shape[1] != 2:\n                errors.append(\"token arrays have invalid ranks\")\n            if errors:\n                raise RuntimeError(f\"invalid shard {shard}: {'; '.join(errors)}\")\n            if int(offsets[-1]) != codes.shape[0] or int(hidden_offsets[-1]) != hidden.shape[0]:\n                raise RuntimeError(f\"terminal offsets disagree with arrays in {shard}\")\n            if token_ids.shape[0] != hidden.shape[0] or char_offsets.shape[0] != hidden.shape[0]:\n                raise RuntimeError(f\"hidden/token arrays disagree in {shard}\")\n            if np.any(np.diff(offsets) < 0) or np.any(np.diff(hidden_offsets) < 0):\n                raise RuntimeError(f\"non-monotonic offsets in {shard}\")\n            if codes.size and (int(codes.min()) < 0 or int(codes.max()) >= cfg.model.codec_vocab):\n                raise RuntimeError(f\"code ID outside [0,{cfg.model.codec_vocab - 1}] in {shard}\")\n            hidden_dim = hidden_dim or int(hidden.shape[1])\n            if int(hidden.shape[1]) != hidden_dim:\n                raise RuntimeError(f\"hidden dimension changed in {shard}\")\n\n            for row, raw_id in enumerate(ids):\n                record_id = str(raw_id)\n                try:\n                    source = sources[record_id]\n                    lo, hi = int(offsets[row]), int(offsets[row + 1])\n                    hlo, hhi = int(hidden_offsets[row]), int(hidden_offsets[row + 1])\n                    alignment = alignments[row]\n                    words = alignment.get(\"words\", [])\n                    if hi <= lo or hhi <= hlo or not words:\n                        raise ValueError(\"empty codes, hidden states, or word alignment\")\n                    if int(words[0][\"start_frame\"]) != 0 or int(words[-1][\"end_frame\"]) != hi - lo:\n                        raise ValueError(\"word alignment does not cover all Mimi frames\")\n                    previous = 0\n                    for word in words:\n                        start, end = int(word[\"start_frame\"]), int(word[\"end_frame\"])\n                        if start != previous or end <= start:\n                            raise ValueError(\"word alignment is not gap-free and monotonic\")\n                        previous = end\n                    split = _split(record_id, cfg.train.seed, cfg.train.val_fraction)\n                    entry = {\n                        \"record_id\": record_id,\n                        \"shard\": str(shard),\n                        \"row\": row,\n                        \"code_lo\": lo,\n                        \"code_hi\": hi,\n                        \"frames\": hi - lo,\n                        \"legacy_tokens\": hhi - hlo,\n                        \"split\": split,\n                        \"question\": source[\"question\"],\n                        \"answer\": source[\"answer\"],\n                        \"words\": words,\n                    }\n                    output.write(json.dumps(entry, ensure_ascii=False) + \"\\n\")\n                    good += 1\n                    split_counts[split] += 1\n                    frames += hi - lo\n                    tokens += hhi - hlo\n                    if record_limit and good >= record_limit:\n                        break\n                except Exception as exc:\n                    bad += 1\n                    rejected.write(\n                        json.dumps({\"record_id\": record_id, \"shard\": str(shard), \"row\": row, \"reason\": str(exc)})\n                        + \"\\n\"\n                    )\n            print(f\"validated shard {shard_no}/{len(shards)}: {shard.name} ({n} rows)\", flush=True)\n            if record_limit and good >= record_limit:\n                print(f\"local manifest limit reached: {good} records\", flush=True)\n                break\n    if good < 4:\n        raise RuntimeError(f\"only {good} valid records found\")\n    manifest_tmp.replace(manifest_path)\n    quarantine_tmp.replace(quarantine_path)\n    report = {\n        \"complete\": True,\n        \"dataset_signature\": signature,\n        \"record_limit\": record_limit,\n        \"records\": good,\n        \"quarantined\": bad,\n        \"frames\": frames,\n        \"audio_hours\": frames / 12.5 / 3600.0,\n        \"legacy_tokens\": tokens,\n        \"hidden_dim\": hidden_dim,\n        \"shards\": validated_shards,\n        \"available_shards\": len(shards),\n        \"split_counts\": split_counts,\n        \"manifest\": str(manifest_path),\n        \"quarantine\": str(quarantine_path),\n    }\n    estimated_feature_bytes = tokens * (int(hidden_dim or 0) * 2 + 12)\n    report[\"estimated_feature_gib\"] = estimated_feature_bytes / 2**30\n    report[\"recommended_working_gib\"] = estimated_feature_bytes * 1.25 / 2**30 + 2.0\n    report[\"free_working_gib\"] = shutil.disk_usage(work).free / 2**30\n    atomic_json(report_path, report)\n    return report\n\n\ndef allocate_token_frames(\n    text: str, token_offsets: np.ndarray, words: Sequence[dict[str, Any]], total_frames: int\n) -> np.ndarray:\n    \"\"\"Allocate a gap-free frame span to every assistant token, including punctuation.\"\"\"\n    n = int(token_offsets.shape[0])\n    if n < 1 or total_frames < 1 or not words:\n        raise ValueError(\"alignment allocation received empty input\")\n    word_char_centers: list[float] = []\n    word_frame_centers: list[float] = []\n    cursor = 0\n    lower = text.lower()\n    for word in words:\n        raw = str(word[\"word\"])\n        found = text.find(raw, cursor)\n        if found < 0:\n            found = lower.find(raw.lower(), cursor)\n        if found < 0:\n            raise ValueError(f\"aligned word {raw!r} is absent from answer\")\n        word_char_centers.append(found + len(raw) / 2.0)\n        word_frame_centers.append((float(word[\"start_frame\"]) + float(word[\"end_frame\"])) / 2.0)\n        cursor = found + len(raw)\n    token_centers = []\n    for start, end in np.asarray(token_offsets, dtype=np.int64):\n        if end > start:\n            token_centers.append((float(start) + float(end)) / 2.0)\n        else:\n            token_centers.append(token_centers[-1] if token_centers else 0.0)\n    centers = np.interp(token_centers, word_char_centers, word_frame_centers)\n    boundaries = np.zeros(n + 1, dtype=np.int32)\n    boundaries[-1] = int(total_frames)\n    if n > 1:\n        boundaries[1:-1] = np.rint((centers[:-1] + centers[1:]) / 2.0).astype(np.int32)\n    boundaries = np.maximum.accumulate(np.clip(boundaries, 0, total_frames))\n    boundaries[-1] = total_frames\n    durations = np.diff(boundaries).astype(np.int32)\n    if durations.sum() != total_frames or np.any(durations < 0):\n        raise RuntimeError(\"token allocation is not gap-free\")\n    return durations\n\n\nclass MMapLRU:\n    def __init__(self, capacity: int = 8):\n        self.capacity = max(1, int(capacity))\n        self._items: OrderedDict[str, np.ndarray] = OrderedDict()\n\n    def get(self, path: str | Path, *, dtype: Any = None, shape: tuple[int, ...] | None = None) -> np.ndarray:\n        key = f\"{path}|{dtype}|{shape}\"\n        if key in self._items:\n            value = self._items.pop(key)\n            self._items[key] = value\n            return value\n        if dtype is None:\n            value = np.load(path, mmap_mode=\"r\")\n        else:\n            if shape is None:\n                raise ValueError(\"raw memmap requires shape\")\n            value = np.memmap(path, mode=\"r\", dtype=dtype, shape=shape)\n        self._items[key] = value\n        while len(self._items) > self.capacity:\n            self._items.popitem(last=False)\n        return value\n\n\ndef load_manifest(path: str | Path, split: str | None = None, limit: int = 0) -> list[dict[str, Any]]:\n    rows = []\n    for item in jsonl(path):\n        if split is None or item[\"split\"] == split:\n            rows.append(item)\n            if limit > 0 and len(rows) >= limit:\n                break\n    return rows\n\n\ndef load_feature_index(path: str | Path) -> dict[str, dict[str, Any]]:\n    return {str(row[\"record_id\"]): row for row in jsonl(path)}\n\n\ndef _qwen_embedding_rows(model_dir: str | Path, token_ids: np.ndarray) -> np.ndarray:\n    \"\"\"Read Qwen's embedding tensor without constructing the full language model.\"\"\"\n    from safetensors import safe_open\n\n    root = Path(model_dir)\n    keys = (\n        \"model.embed_tokens.weight\",\n        \"model.model.embed_tokens.weight\",\n        \"transformer.wte.weight\",\n        \"lm_head.weight\",\n    )\n    tensor_path: Path | None = None\n    tensor_key: str | None = None\n    indices = sorted(root.glob(\"*.safetensors.index.json\"))\n    if indices:\n        weight_map = json.loads(indices[0].read_text(encoding=\"utf-8\")).get(\"weight_map\", {})\n        for key in keys:\n            filename = weight_map.get(key)\n            if filename:\n                tensor_path, tensor_key = root / filename, key\n                break\n    if tensor_path is None:\n        for path in sorted(root.glob(\"*.safetensors\")):\n            with safe_open(path, framework=\"pt\", device=\"cpu\") as handle:\n                available = set(handle.keys())\n            tensor_key = next((key for key in keys if key in available), None)\n            if tensor_key is not None:\n                tensor_path = path\n                break\n    if tensor_path is None or tensor_key is None:\n        raise FileNotFoundError(\n            f\"could not find Qwen input embeddings in safetensors under {root}\"\n        )\n    print(f\"loading Qwen embedding only: {tensor_path.name}:{tensor_key}\", flush=True)\n    with safe_open(tensor_path, framework=\"pt\", device=\"cpu\") as handle:\n        table = handle.get_tensor(tensor_key)\n    ids = np.asarray(token_ids, dtype=np.int64)\n    if ids.size and (int(ids.min()) < 0 or int(ids.max()) >= int(table.shape[0])):\n        raise RuntimeError(\n            f\"token ID range [{ids.min()},{ids.max()}] exceeds {table.shape[0]} embedding rows\"\n        )\n    import torch\n\n    selected = table.index_select(0, torch.from_numpy(ids)).float().numpy().astype(np.float16)\n    del table\n    return selected\n\n\ndef _feature_worker(cfg: AppConfig) -> tuple[Any, Any, int, int, int, Any]:\n    import torch\n    import torch.distributed as dist\n\n    rank = int(os.environ.get(\"RANK\", \"0\"))\n    world = int(os.environ.get(\"WORLD_SIZE\", \"1\"))\n    local_rank = int(os.environ.get(\"LOCAL_RANK\", rank))\n    use_cuda = cfg.runtime.device == \"cuda\" or (\n        cfg.runtime.device == \"auto\" and torch.cuda.is_available()\n    )\n    if cfg.runtime.device == \"cuda\" and not torch.cuda.is_available():\n        raise RuntimeError(\"runtime.device='cuda' but CUDA is not available\")\n    if use_cuda:\n        torch.cuda.set_device(local_rank)\n    if world > 1:\n        dist.init_process_group(\"nccl\" if use_cuda else \"gloo\")\n    device = torch.device(f\"cuda:{local_rank}\" if use_cuda else \"cpu\")\n    if device.type == \"cpu\":\n        torch.set_num_threads(cfg.runtime.cpu_threads)\n    return torch, dist, rank, world, local_rank, device\n\n\ndef _ensure_manifest(cfg: AppConfig, rank: int, world: int, dist: Any) -> None:\n    \"\"\"Build the prerequisite once, then release all feature workers.\"\"\"\n    manifest = cfg.work_dir / \"manifest.jsonl\"\n    report = cfg.work_dir / \"dataset_report.json\"\n    if rank == 0:\n        if not manifest.is_file() or not report.is_file():\n            print(\"manifest is missing; building it before feature preparation\", flush=True)\n        build_manifest(cfg)\n    if world > 1:\n        dist.barrier()\n    if not manifest.is_file() or not report.is_file():\n        raise RuntimeError(\"manifest preparation did not produce its required outputs\")\n\n\ndef _feature_complete(\n    cfg: AppConfig, done_path: Path, selected: Sequence[dict[str, Any]], torch: Any,\n    dist: Any, world: int, device: Any,\n) -> tuple[bool, bool]:\n    local = False\n    final_path = done_path.parent / \"features.done.json\"\n    final_valid = False\n    if final_path.is_file():\n        final_marker = json.loads(final_path.read_text())\n        final_valid = final_marker.get(\"feature_fingerprint\") == cfg.feature_fingerprint()\n    if done_path.is_file():\n        marker = json.loads(done_path.read_text())\n        local = (\n            final_valid\n            and marker.get(\"feature_mode\") == cfg.feature_mode\n            and marker.get(\"feature_fingerprint\") == cfg.feature_fingerprint()\n            and marker.get(\"records\") == len(selected)\n        )\n    complete = local\n    if world > 1:\n        flag = torch.tensor(int(local), device=device, dtype=torch.int32)\n        dist.all_reduce(flag, op=dist.ReduceOp.MIN)\n        complete = bool(flag.item())\n    return local, complete\n\n\ndef _merge_feature_indices(\n    cfg: AppConfig, feature_dir: Path, world: int, hidden_dim: int,\n    embedding_values: np.ndarray, runtime_stats: dict[str, Any] | None = None,\n) -> None:\n    indices: list[dict[str, Any]] = []\n    all_ids = []\n    for worker in range(world):\n        indices.extend(jsonl(feature_dir / f\"index_rank{worker:02d}.jsonl\"))\n        all_ids.append(np.load(feature_dir / f\"used_ids_rank{worker:02d}.npy\"))\n    indices.sort(key=lambda item: str(item[\"record_id\"]))\n    temporary = feature_dir / \"index.jsonl.tmp\"\n    with temporary.open(\"w\", encoding=\"utf-8\") as handle:\n        for item in indices:\n            handle.write(json.dumps(item) + \"\\n\")\n    temporary.replace(feature_dir / \"index.jsonl\")\n    unique = np.unique(np.concatenate(all_ids)).astype(np.int32)\n    if int(embedding_values.shape[0]) != int(unique.size):\n        raise RuntimeError(\"embedding rows do not match the used token ID set\")\n    np.save(feature_dir / \"embedding_ids.npy\", unique)\n    np.save(feature_dir / \"embedding_values.npy\", embedding_values)\n    marker = {\n        \"feature_fingerprint\": cfg.feature_fingerprint(),\n        \"feature_mode\": cfg.feature_mode,\n        \"records\": len(indices),\n        \"vocabulary_rows\": int(unique.size),\n        \"hidden_dim\": hidden_dim,\n        \"embedding_dim\": int(embedding_values.shape[1]),\n    }\n    marker.update(runtime_stats or {})\n    atomic_json(feature_dir / \"features.done.json\", marker)\n\n\ndef _peak_rss_mib() -> float | None:\n    status = Path(\"/proc/self/status\")\n    if not status.is_file():\n        return None\n    for line in status.read_text(encoding=\"utf-8\").splitlines():\n        if line.startswith(\"VmHWM:\"):\n            return float(line.split()[1]) / 1024.0\n    return None\n\n\ndef _prepare_legacy_answer_features(cfg: AppConfig) -> None:\n    \"\"\"Index answer-only shard features; do not run the Qwen language model.\"\"\"\n    torch, dist, rank, world, local_rank, device = _feature_worker(cfg)\n    _ensure_manifest(cfg, rank, world, dist)\n    print(\n        f\"rank {rank}/{world}: reusing answer-only shard hidden/token features; \"\n        f\"local_rank={local_rank}\",\n        flush=True,\n    )\n    rows = load_manifest(cfg.work_dir / \"manifest.jsonl\")\n    selected = [row for index, row in enumerate(rows) if index % world == rank]\n    feature_dir = cfg.work_dir / \"features\"\n    feature_dir.mkdir(parents=True, exist_ok=True)\n    index_path = feature_dir / f\"index_rank{rank:02d}.jsonl\"\n    done_path = feature_dir / f\"rank{rank:02d}.done.json\"\n    local_complete, all_complete = _feature_complete(\n        cfg, done_path, selected, torch, dist, world, device\n    )\n    if all_complete:\n        print(f\"rank {rank}: answer-only features already complete\", flush=True)\n        if world > 1:\n            dist.barrier()\n            dist.destroy_process_group()\n        return\n    if local_complete:\n        print(f\"rank {rank}: rebuilding index because another rank is incomplete\", flush=True)\n    dataset_report = json.loads((cfg.work_dir / \"dataset_report.json\").read_text())\n    hidden_dim = int(dataset_report[\"hidden_dim\"])\n    cache = MMapLRU(12)\n    token_offset = 0\n    used_ids: set[int] = set()\n    temporary = Path(str(index_path) + \".tmp\")\n    with temporary.open(\"w\", encoding=\"utf-8\") as output:\n        for position, row in enumerate(selected, 1):\n            shard = Path(row[\"shard\"])\n            shard_row = int(row[\"row\"])\n            hidden_offsets = cache.get(shard / \"hidden_offsets.npy\")\n            hidden_lo = int(hidden_offsets[shard_row])\n            hidden_hi = int(hidden_offsets[shard_row + 1])\n            token_ids = np.asarray(\n                cache.get(shard / \"token_ids.npy\")[hidden_lo:hidden_hi], dtype=np.int32\n            )\n            char_offsets = np.asarray(\n                cache.get(shard / \"token_char_offsets.npy\")[hidden_lo:hidden_hi],\n                dtype=np.int32,\n            )\n            if token_ids.size < 1 or char_offsets.shape != (token_ids.size, 2):\n                raise RuntimeError(f\"invalid legacy answer features for {row['record_id']}\")\n            durations = allocate_token_frames(\n                row[\"answer\"], char_offsets, row[\"words\"], int(row[\"frames\"])\n            )\n            output.write(\n                json.dumps(\n                    {\n                        \"record_id\": row[\"record_id\"],\n                        \"source\": \"legacy_answer\",\n                        \"rank\": rank,\n                        \"token_offset\": token_offset,\n                        \"tokens\": int(token_ids.size),\n                        \"hidden_dim\": hidden_dim,\n                        \"shard\": str(shard),\n                        \"hidden_lo\": hidden_lo,\n                        \"hidden_hi\": hidden_hi,\n                        \"durations\": durations.tolist(),\n                    }\n                )\n                + \"\\n\"\n            )\n            token_offset += int(token_ids.size)\n            used_ids.update(int(value) for value in token_ids)\n            if position == 1 or position % 500 == 0 or position == len(selected):\n                print(f\"rank {rank}: indexed {position}/{len(selected)}\", flush=True)\n    temporary.replace(index_path)\n    np.save(\n        feature_dir / f\"used_ids_rank{rank:02d}.npy\",\n        np.asarray(sorted(used_ids), dtype=np.int32),\n    )\n    atomic_json(\n        done_path,\n        {\n            \"feature_fingerprint\": cfg.feature_fingerprint(),\n            \"feature_mode\": cfg.feature_mode,\n            \"records\": len(selected),\n            \"tokens\": token_offset,\n            \"hidden_dim\": hidden_dim,\n        },\n    )\n    if world > 1:\n        dist.barrier()\n    if rank == 0:\n        all_ids = [\n            np.load(feature_dir / f\"used_ids_rank{worker:02d}.npy\")\n            for worker in range(world)\n        ]\n        unique = np.unique(np.concatenate(all_ids)).astype(np.int32)\n        values = _qwen_embedding_rows(cfg.paths.qwen_model, unique)\n        _merge_feature_indices(cfg, feature_dir, world, hidden_dim, values)\n        print(\n            f\"answer-only feature preparation complete: {len(rows)} records; \"\n            f\"no Qwen forward passes; {unique.size} embedding rows\",\n            flush=True,\n        )\n    if world > 1:\n        dist.barrier()\n        dist.destroy_process_group()\n\n\ndef _prepare_recomputed_features(cfg: AppConfig, include_chat_context: bool) -> None:\n    \"\"\"Recompute answer states, optionally after the deployment chat prefix.\"\"\"\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    torch, dist, rank, world, local_rank, device = _feature_worker(cfg)\n    started = time.perf_counter()\n    _ensure_manifest(cfg, rank, world, dist)\n    source_mode = \"deployment_context\" if include_chat_context else \"answer_only_recompute\"\n    print(\n        f\"rank {rank}/{world}: {source_mode} feature worker; \"\n        f\"local_rank={local_rank}; Qwen={cfg.paths.qwen_model}\",\n        flush=True,\n    )\n    dataset_report = json.loads((cfg.work_dir / \"dataset_report.json\").read_text())\n    feature_dir = cfg.work_dir / \"features\"\n    existing_bytes = sum(\n        path.stat().st_size for path in feature_dir.glob(\"**/*\") if path.is_file()\n    ) if feature_dir.exists() else 0\n    available_bytes = shutil.disk_usage(cfg.work_dir).free + existing_bytes\n    required_bytes = int(dataset_report[\"estimated_feature_gib\"] * 2**30 * 1.25 + 2 * 2**30)\n    if available_bytes < required_bytes:\n        raise RuntimeError(\n            f\"deployment-context sidecars need about {required_bytes / 2**30:.1f} GiB including reserve; \"\n            f\"only {available_bytes / 2**30:.1f} GiB is available\"\n        )\n    rows = load_manifest(cfg.work_dir / \"manifest.jsonl\")\n    selected = [row for index, row in enumerate(rows) if index % world == rank]\n    feature_dir.mkdir(parents=True, exist_ok=True)\n    hidden_path = feature_dir / f\"hidden_rank{rank:02d}.f16\"\n    ids_path = feature_dir / f\"token_ids_rank{rank:02d}.i32\"\n    chars_path = feature_dir / f\"char_offsets_rank{rank:02d}.i32\"\n    index_path = feature_dir / f\"index_rank{rank:02d}.jsonl\"\n    done_path = feature_dir / f\"rank{rank:02d}.done.json\"\n    local_complete, all_complete = _feature_complete(\n        cfg, done_path, selected, torch, dist, world, device\n    )\n    if all_complete:\n        print(f\"rank {rank}: {source_mode} features already complete\", flush=True)\n        if world > 1:\n            dist.barrier()\n            dist.destroy_process_group()\n        return\n    if local_complete:\n        print(f\"rank {rank}: rebuilding sidecar because another rank is incomplete\", flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(cfg.paths.qwen_model, use_fast=True)\n    dtype = torch.float16 if device.type == \"cuda\" else torch.float32\n    print(f\"rank {rank}: loading full Qwen onto {device}\", flush=True)\n    model = AutoModelForCausalLM.from_pretrained(\n        cfg.paths.qwen_model, dtype=dtype, low_cpu_mem_usage=True\n    ).to(device)\n    print(f\"rank {rank}: Qwen loaded; beginning {len(selected)} records\", flush=True)\n    model.eval()\n    for parameter in model.parameters():\n        parameter.requires_grad_(False)\n    hidden_dim = int(model.config.hidden_size)\n    token_offset = 0\n    used_ids: set[int] = set()\n    suffix = \".tmp\"\n    with open(str(hidden_path) + suffix, \"wb\") as hidden_out, open(\n        str(ids_path) + suffix, \"wb\"\n    ) as ids_out, open(str(chars_path) + suffix, \"wb\") as chars_out, open(\n        str(index_path) + suffix, \"w\", encoding=\"utf-8\"\n    ) as index_out:\n        for position, row in enumerate(selected, 1):\n            answer = tokenizer(\n                row[\"answer\"], add_special_tokens=False, return_offsets_mapping=True\n            )\n            answer_ids = [int(value) for value in answer[\"input_ids\"]]\n            offsets = np.asarray(answer[\"offset_mapping\"], dtype=np.int32)\n            if not answer_ids or offsets.shape != (len(answer_ids), 2):\n                raise RuntimeError(f\"invalid assistant tokenization for {row['record_id']}\")\n            prefix_ids = []\n            if include_chat_context:\n                prefix_ids = tokenizer.apply_chat_template(\n                    [{\"role\": \"user\", \"content\": row[\"question\"]}],\n                    tokenize=True,\n                    add_generation_prompt=True,\n                )\n            full = torch.tensor([list(prefix_ids) + answer_ids], device=device, dtype=torch.long)\n            with torch.inference_mode(), torch.autocast(\n                device_type=device.type, enabled=device.type == \"cuda\"\n            ):\n                result = model(full, output_hidden_states=True, use_cache=False)\n                states = (\n                    result.hidden_states[-1][0, -len(answer_ids) :]\n                    .float().cpu().numpy().astype(np.float16)\n                )\n            states.tofile(hidden_out)\n            np.asarray(answer_ids, dtype=np.int32).tofile(ids_out)\n            offsets.tofile(chars_out)\n            durations = allocate_token_frames(\n                row[\"answer\"], offsets, row[\"words\"], int(row[\"frames\"])\n            )\n            index_out.write(\n                json.dumps(\n                    {\n                        \"record_id\": row[\"record_id\"],\n                        \"source\": source_mode,\n                        \"rank\": rank,\n                        \"token_offset\": token_offset,\n                        \"tokens\": len(answer_ids),\n                        \"hidden_dim\": hidden_dim,\n                        \"durations\": durations.tolist(),\n                    }\n                )\n                + \"\\n\"\n            )\n            token_offset += len(answer_ids)\n            used_ids.update(answer_ids)\n            if position == 1 or position % 100 == 0 or position == len(selected):\n                print(f\"rank {rank}: features {position}/{len(selected)}\", flush=True)\n    Path(str(hidden_path) + suffix).replace(hidden_path)\n    Path(str(ids_path) + suffix).replace(ids_path)\n    Path(str(chars_path) + suffix).replace(chars_path)\n    Path(str(index_path) + suffix).replace(index_path)\n    np.save(\n        feature_dir / f\"used_ids_rank{rank:02d}.npy\",\n        np.asarray(sorted(used_ids), dtype=np.int32),\n    )\n    atomic_json(\n        done_path,\n        {\n            \"feature_fingerprint\": cfg.feature_fingerprint(),\n            \"feature_mode\": cfg.feature_mode,\n            \"records\": len(selected),\n            \"tokens\": token_offset,\n            \"hidden_dim\": hidden_dim,\n        },\n    )\n    if world > 1:\n        dist.barrier()\n    if rank == 0:\n        all_ids = [\n            np.load(feature_dir / f\"used_ids_rank{worker:02d}.npy\")\n            for worker in range(world)\n        ]\n        unique = np.unique(np.concatenate(all_ids)).astype(np.int32)\n        embedding = model.get_input_embeddings().weight.detach()\n        values = (\n            embedding[torch.from_numpy(unique.astype(np.int64)).to(device)]\n            .float().cpu().numpy().astype(np.float16)\n        )\n        _merge_feature_indices(\n            cfg,\n            feature_dir,\n            world,\n            hidden_dim,\n            values,\n            {\n                \"preparation_device\": str(device),\n                \"preparation_seconds_rank0\": time.perf_counter() - started,\n                \"preparation_peak_rss_mib_rank0\": _peak_rss_mib(),\n            },\n        )\n    if world > 1:\n        dist.barrier()\n        dist.destroy_process_group()\n\n\ndef prepare_qwen_features(cfg: AppConfig) -> None:\n    \"\"\"Prepare the selected answer-only or deployment-context feature contract.\"\"\"\n    if cfg.feature_mode == \"legacy_answer\":\n        _prepare_legacy_answer_features(cfg)\n    elif cfg.feature_mode == \"answer_only_recompute\":\n        _prepare_recomputed_features(cfg, include_chat_context=False)\n    else:\n        _prepare_recomputed_features(cfg, include_chat_context=True)\n\n\nclass SpeechDataset:\n    def __init__(self, cfg: AppConfig, split: str, limit: int = 0):\n        import torch\n\n        self.torch = torch\n        self.cfg = cfg\n        self.rows = load_manifest(cfg.work_dir / \"manifest.jsonl\", split=split, limit=limit)\n        self.features = load_feature_index(cfg.work_dir / \"features\" / \"index.jsonl\")\n        feature_marker = json.loads(\n            (cfg.work_dir / \"features\" / \"features.done.json\").read_text()\n        )\n        if feature_marker.get(\"feature_fingerprint\") != cfg.feature_fingerprint():\n            raise RuntimeError(\"feature sidecars do not match the configured dataset/Qwen context\")\n        ids = np.load(cfg.work_dir / \"features\" / \"embedding_ids.npy\", mmap_mode=\"r\")\n        values = np.load(cfg.work_dir / \"features\" / \"embedding_values.npy\", mmap_mode=\"r\")\n        self.embedding_by_id = {int(token): index for index, token in enumerate(ids)}\n        self.embedding_values = values\n        self.cache = MMapLRU(12)\n        self.lengths = [int(row[\"frames\"]) for row in self.rows]\n        missing = [row[\"record_id\"] for row in self.rows if row[\"record_id\"] not in self.features]\n        if missing:\n            raise RuntimeError(f\"{len(missing)} records lack prepared features; first={missing[:3]}\")\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        torch = self.torch\n        row = self.rows[index]\n        ref = self.features[row[\"record_id\"]]\n        source = str(ref.get(\"source\", \"deployment_context\"))\n        rank = int(ref[\"rank\"])\n        offset, count, hidden_dim = int(ref[\"token_offset\"]), int(ref[\"tokens\"]), int(ref[\"hidden_dim\"])\n        feature_dir = self.cfg.work_dir / \"features\"\n        if source == \"legacy_answer\":\n            shard = Path(ref[\"shard\"])\n            hidden_lo, hidden_hi = int(ref[\"hidden_lo\"]), int(ref[\"hidden_hi\"])\n            hidden = self.cache.get(shard / \"hidden.npy\")[hidden_lo:hidden_hi]\n            token_ids = self.cache.get(shard / \"token_ids.npy\")[hidden_lo:hidden_hi]\n            char_offsets = self.cache.get(shard / \"token_char_offsets.npy\")[hidden_lo:hidden_hi]\n        elif source in {\"answer_only_recompute\", \"deployment_context\"}:\n            marker = json.loads((feature_dir / f\"rank{rank:02d}.done.json\").read_text())\n            total_tokens = int(marker[\"tokens\"])\n            hidden = self.cache.get(\n                feature_dir / f\"hidden_rank{rank:02d}.f16\",\n                dtype=np.float16,\n                shape=(total_tokens, hidden_dim),\n            )[offset : offset + count]\n            token_ids = self.cache.get(\n                feature_dir / f\"token_ids_rank{rank:02d}.i32\",\n                dtype=np.int32,\n                shape=(total_tokens,),\n            )[offset : offset + count]\n            char_offsets = self.cache.get(\n                feature_dir / f\"char_offsets_rank{rank:02d}.i32\",\n                dtype=np.int32,\n                shape=(total_tokens, 2),\n            )[offset : offset + count]\n        else:\n            raise RuntimeError(f\"unknown feature source {source!r} for {row['record_id']}\")\n        if int(hidden.shape[0]) != count or int(hidden.shape[1]) != hidden_dim:\n            raise RuntimeError(f\"hidden shape mismatch for {row['record_id']}: {hidden.shape}\")\n        embedding_rows = [self.embedding_by_id[int(token)] for token in token_ids]\n        embeddings = np.asarray(self.embedding_values[embedding_rows], dtype=np.float32)\n        codes_all = self.cache.get(Path(row[\"shard\"]) / \"codes.npy\")\n        codes = np.asarray(codes_all[int(row[\"code_lo\"]) : int(row[\"code_hi\"])], dtype=np.int64)\n        durations = np.asarray(ref[\"durations\"], dtype=np.int64)\n        if durations.shape[0] != count or int(durations.sum()) != codes.shape[0]:\n            raise RuntimeError(f\"duration/code mismatch for {row['record_id']}\")\n        return {\n            \"record_id\": row[\"record_id\"],\n            \"text\": row[\"answer\"],\n            \"hidden\": torch.from_numpy(np.asarray(hidden, dtype=np.float32)),\n            \"embeddings\": torch.from_numpy(embeddings),\n            \"durations\": torch.from_numpy(durations),\n            \"codes\": torch.from_numpy(codes.copy()),\n            \"char_offsets\": np.asarray(char_offsets, dtype=np.int32).copy(),\n            \"words\": row[\"words\"],\n        }\n\n\ndef collate_speech(items: Sequence[dict[str, Any]]) -> dict[str, Any]:\n    import torch\n\n    batch = len(items)\n    max_tokens = max(int(item[\"hidden\"].shape[0]) for item in items)\n    max_frames = max(int(item[\"codes\"].shape[0]) for item in items)\n    hidden_dim = int(items[0][\"hidden\"].shape[1])\n    embedding_dim = int(items[0][\"embeddings\"].shape[1])\n    hidden = torch.zeros(batch, max_tokens, hidden_dim)\n    embeddings = torch.zeros(batch, max_tokens, embedding_dim)\n    durations = torch.zeros(batch, max_tokens, dtype=torch.long)\n    token_mask = torch.zeros(batch, max_tokens, dtype=torch.bool)\n    codes = torch.zeros(batch, max_frames, 8, dtype=torch.long)\n    frame_mask = torch.zeros(batch, max_frames, dtype=torch.bool)\n    owners = torch.zeros(batch, max_frames, dtype=torch.long)\n    for b, item in enumerate(items):\n        n, frames = item[\"hidden\"].shape[0], item[\"codes\"].shape[0]\n        hidden[b, :n] = item[\"hidden\"]\n        embeddings[b, :n] = item[\"embeddings\"]\n        durations[b, :n] = item[\"durations\"]\n        token_mask[b, :n] = True\n        codes[b, :frames] = item[\"codes\"]\n        frame_mask[b, :frames] = True\n        owner = torch.repeat_interleave(torch.arange(n), item[\"durations\"])\n        if owner.numel() != frames:\n            raise RuntimeError(\"frame-owner construction failed\")\n        owners[b, :frames] = owner\n    return {\n        \"record_ids\": [item[\"record_id\"] for item in items],\n        \"texts\": [item[\"text\"] for item in items],\n        \"hidden\": hidden,\n        \"embeddings\": embeddings,\n        \"durations\": durations,\n        \"token_mask\": token_mask,\n        \"codes\": codes,\n        \"frame_mask\": frame_mask,\n        \"owners\": owners,\n    }\n\n\nclass DistributedFrameBatchSampler:\n    \"\"\"Deterministic variable-size batches with an equal step count on every rank.\"\"\"\n\n    def __init__(\n        self,\n        lengths: Sequence[int],\n        frame_budget: int,\n        max_batch_size: int,\n        rank: int,\n        world_size: int,\n        seed: int,\n        shuffle: bool,\n    ):\n        self.lengths = list(map(int, lengths))\n        self.frame_budget = int(frame_budget)\n        self.max_batch_size = int(max_batch_size)\n        self.rank = int(rank)\n        self.world_size = int(world_size)\n        self.seed = int(seed)\n        self.shuffle = bool(shuffle)\n        self.epoch = 0\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def _global_batches(self) -> list[list[int]]:\n        indices = list(range(len(self.lengths)))\n        rng = random.Random(self.seed + self.epoch)\n        if self.shuffle:\n            rng.shuffle(indices)\n        bucket = max(32, self.max_batch_size * 16)\n        ordered: list[int] = []\n        for start in range(0, len(indices), bucket):\n            part = indices[start : start + bucket]\n            part.sort(key=self.lengths.__getitem__)\n            if self.shuffle and rng.random() < 0.5:\n                part.reverse()\n            ordered.extend(part)\n        batches: list[list[int]] = []\n        current: list[int] = []\n        current_frames = 0\n        for index in ordered:\n            length = self.lengths[index]\n            if length > self.frame_budget:\n                raise RuntimeError(\n                    f\"sample {index} has {length} frames, above per-GPU budget {self.frame_budget}\"\n                )\n            if current and (current_frames + length > self.frame_budget or len(current) >= self.max_batch_size):\n                batches.append(current)\n                current, current_frames = [], 0\n            current.append(index)\n            current_frames += length\n        if current:\n            batches.append(current)\n        if self.shuffle:\n            rng.shuffle(batches)\n        if not batches:\n            return []\n        original = list(batches)\n        cursor = 0\n        while len(batches) % self.world_size:\n            batches.append(list(original[cursor % len(original)]))\n            cursor += 1\n        return batches\n\n    def __iter__(self) -> Iterator[list[int]]:\n        yield from self._global_batches()[self.rank :: self.world_size]\n\n    def __len__(self) -> int:\n        return len(self._global_batches()) // self.world_size\n", "diagnostics.py": "from __future__ import annotations\n\nimport html\nimport json\nimport math\nimport time\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Any, Sequence\n\nimport numpy as np\n\nfrom .config import AppConfig, atomic_json, sha256_file\nfrom .data import SpeechDataset, discover_shards, load_manifest\n\nSAMPLE_RATE = 24_000\nFRAME_RATE = 12.5\n\n\ndef _providers() -> list[str]:\n    import onnxruntime as ort\n\n    available = ort.get_available_providers()\n    preferred = [\"CUDAExecutionProvider\", \"CPUExecutionProvider\"]\n    return [provider for provider in preferred if provider in available]\n\n\nclass OnnxMimi:\n    def __init__(self, encoder: str, decoder: str, num_codebooks: int = 8):\n        import onnxruntime as ort\n\n        self.encoder = ort.InferenceSession(encoder, providers=_providers())\n        self.decoder = ort.InferenceSession(decoder, providers=_providers())\n        self.num_codebooks = int(num_codebooks)\n        decoder_shape = self.decoder.get_inputs()[0].shape\n        decoder_books = decoder_shape[1] if len(decoder_shape) > 1 else None\n        if decoder_books != self.num_codebooks:\n            raise RuntimeError(\n                f\"Mimi decoder expects {decoder_books!r} codebooks, but the adapter produces \"\n                f\"{self.num_codebooks}. Do not zero-pad missing residual codebooks: use a true \"\n                f\"{self.num_codebooks}-codebook decoder ONNX.\"\n            )\n\n    def encode(self, waveform: np.ndarray) -> np.ndarray:\n        model_input = self.encoder.get_inputs()[0]\n        audio = np.asarray(waveform, dtype=np.float32).reshape(1, 1, -1)\n        raw = np.asarray(self.encoder.run(None, {model_input.name: audio})[0])\n        if raw.ndim != 3 or raw.shape[0] != 1:\n            raise RuntimeError(f\"unexpected Mimi encoder output {raw.shape}\")\n        raw = raw[0]\n        if raw.shape[0] in (self.num_codebooks, 32):\n            codes = raw[: self.num_codebooks].T\n        elif raw.shape[1] in (self.num_codebooks, 32):\n            codes = raw[:, : self.num_codebooks]\n        else:\n            raise RuntimeError(f\"cannot locate codebook axis in Mimi output {raw.shape}\")\n        codes = np.asarray(codes, dtype=np.int64)\n        if codes.ndim != 2 or codes.shape[1] != self.num_codebooks or codes.size == 0:\n            raise RuntimeError(\n                f\"normalized Mimi codes must be [frames,{self.num_codebooks}], got {codes.shape}\"\n            )\n        if int(codes.min()) < 0 or int(codes.max()) >= 2048:\n            raise RuntimeError(\"Mimi encoder produced an invalid code ID\")\n        return codes\n\n    def decode(self, codes: np.ndarray) -> np.ndarray:\n        model_input = self.decoder.get_inputs()[0]\n        values = np.asarray(codes, dtype=np.int64)\n        if values.ndim != 2 or values.shape[1] != self.num_codebooks:\n            raise ValueError(\n                f\"decode expects [frames,{self.num_codebooks}], got {values.shape}\"\n            )\n        transposed = values.T[None]\n        output = self.decoder.run(None, {model_input.name: transposed})[0]\n        waveform = np.asarray(output, dtype=np.float32).reshape(-1)\n        if not waveform.size or not np.isfinite(waveform).all():\n            raise RuntimeError(\"Mimi decoder returned empty or non-finite audio\")\n        return waveform\n\n\ndef inspect_mimi_contract(cfg: AppConfig) -> dict[str, Any]:\n    \"\"\"Verify that codec targets and decoder agree on the eight-book contract.\"\"\"\n    import onnxruntime as ort\n\n    expected = int(cfg.model.num_codebooks)\n    required = {\n        \"encoder\": Path(cfg.paths.mimi_encoder),\n        \"decoder\": Path(cfg.paths.mimi_decoder),\n    }\n    missing = [f\"{name}={path}\" for name, path in required.items() if not path.is_file()]\n    if missing:\n        raise FileNotFoundError(\"missing Mimi ONNX assets: \" + \", \".join(missing))\n    encoder = ort.InferenceSession(str(required[\"encoder\"]), providers=_providers())\n    decoder = ort.InferenceSession(str(required[\"decoder\"]), providers=_providers())\n    encoder_shape = encoder.get_outputs()[0].shape\n    decoder_shape = decoder.get_inputs()[0].shape\n    encoder_books = encoder_shape[1] if len(encoder_shape) > 1 else None\n    decoder_books = decoder_shape[1] if len(decoder_shape) > 1 else None\n    encoder_ok = isinstance(encoder_books, int) and encoder_books >= expected\n    decoder_ok = decoder_books == expected\n    decoder_smoke = False\n    decoder_samples = 0\n    if decoder_ok:\n        test_codes = np.zeros((1, expected, 2), dtype=np.int64)\n        test_audio = np.asarray(\n            decoder.run(None, {decoder.get_inputs()[0].name: test_codes})[0]\n        )\n        decoder_samples = int(test_audio.size)\n        decoder_smoke = bool(test_audio.size and np.isfinite(test_audio).all())\n    report = {\n        \"automatic_pass\": bool(encoder_ok and decoder_ok and decoder_smoke),\n        \"expected_codebooks\": expected,\n        \"encoder_output_shape\": encoder_shape,\n        \"encoder_output_codebooks\": encoder_books,\n        \"encoder_contract\": (\n            f\"first {expected} sequential residual codebooks are used\"\n            if encoder_ok else \"incompatible\"\n        ),\n        \"decoder_input_shape\": decoder_shape,\n        \"decoder_input_codebooks\": decoder_books,\n        \"decoder_contract\": \"exact\" if decoder_ok else \"incompatible\",\n        \"decoder_smoke_pass\": decoder_smoke,\n        \"decoder_smoke_samples\": decoder_samples,\n        \"zero_padding_allowed\": False,\n        \"warning\": None if decoder_ok else (\n            f\"The adapter produces {expected} books but this decoder consumes {decoder_books}. \"\n            \"Padding the remaining books with token 0 adds real quantizer embeddings and corrupts audio.\"\n        ),\n        \"encoder\": str(required[\"encoder\"]),\n        \"decoder\": str(required[\"decoder\"]),\n    }\n    atomic_json(cfg.work_dir / \"gates\" / \"mimi_contract.json\", report)\n    return report\n\n\nclass StatefulMimi:\n    def __init__(self, weights: str, device: str = \"cuda:0\"):\n        import torch\n        from moshi.models import loaders\n\n        self.torch = torch\n        self.device = torch.device(device)\n        self.model = loaders.get_mimi(weights, device=self.device)\n        self.model.set_num_codebooks(8)\n\n    def decode(self, codes: np.ndarray) -> np.ndarray:\n        chunks = []\n        tensor = self.torch.from_numpy(np.asarray(codes, dtype=np.int64).T[None]).to(self.device)\n        with self.torch.no_grad(), self.model.streaming(batch_size=1):\n            for frame in range(tensor.shape[-1]):\n                chunk = self.model.decode(tensor[:, :, frame : frame + 1])\n                chunks.append(chunk.detach().float().cpu())\n        if not chunks:\n            raise RuntimeError(\"stateful Mimi produced no chunks\")\n        return self.torch.cat(chunks, dim=-1)[0, 0].numpy()\n\n\n@lru_cache(maxsize=2)\ndef _kokoro_resources(model_path: str, voices_path: str):\n    import onnxruntime as ort\n    from kokoro_onnx.tokenizer import Tokenizer\n\n    session = ort.InferenceSession(model_path, providers=_providers())\n    tokenizer = Tokenizer()\n    voices = np.fromfile(voices_path, dtype=np.float32).reshape(-1, 1, 256)\n    return session, tokenizer, voices\n\n\ndef synthesize_kokoro(text: str, model_path: str, voices_path: str) -> np.ndarray:\n    session, tokenizer, voices = _kokoro_resources(model_path, voices_path)\n    re = __import__(\"re\")\n    sentences = [part.strip() for part in re.split(r\"(?<=[.!?])\\s+\", text.strip()) if part.strip()]\n    chunks: list[str] = []\n    current = \"\"\n\n    def flush() -> None:\n        nonlocal current\n        if current:\n            chunks.append(current)\n            current = \"\"\n\n    def hard_split(phonemes: str) -> None:\n        for offset in range(0, len(phonemes), 500):\n            piece = phonemes[offset : offset + 500].strip()\n            if piece:\n                chunks.append(piece)\n\n    for sentence in sentences or [text.strip()]:\n        sentence_phonemes = tokenizer.phonemize(sentence, lang=\"en-us\")\n        if not sentence_phonemes:\n            continue\n        if len(sentence_phonemes) <= 500:\n            combined = f\"{current} {sentence_phonemes}\".strip() if current else sentence_phonemes\n            if len(combined) <= 500:\n                current = combined\n            else:\n                flush()\n                current = sentence_phonemes\n            continue\n        flush()\n        buffered_text = \"\"\n        buffered_phonemes = \"\"\n        for word in sentence.split():\n            trial_text = f\"{buffered_text} {word}\".strip()\n            trial_phonemes = tokenizer.phonemize(trial_text, lang=\"en-us\")\n            if len(trial_phonemes) <= 500:\n                buffered_text, buffered_phonemes = trial_text, trial_phonemes\n                continue\n            if buffered_phonemes:\n                chunks.append(buffered_phonemes)\n            word_phonemes = tokenizer.phonemize(word, lang=\"en-us\")\n            if len(word_phonemes) <= 500:\n                buffered_text, buffered_phonemes = word, word_phonemes\n            else:\n                hard_split(word_phonemes)\n                buffered_text = buffered_phonemes = \"\"\n        if buffered_phonemes:\n            chunks.append(buffered_phonemes)\n    flush()\n    audio_parts = []\n    for piece in chunks:\n        ids = tokenizer.tokenize(piece)\n        if not ids:\n            continue\n        style = voices[min(len(ids), voices.shape[0] - 1)]\n        tokens = np.asarray([[0, *ids, 0]], dtype=np.int64)\n        output = session.run(\n            None,\n            {\"input_ids\": tokens, \"style\": style, \"speed\": np.ones(1, np.float32)},\n        )[0]\n        audio_parts.append(np.asarray(output, dtype=np.float32).reshape(-1))\n    if not audio_parts:\n        raise RuntimeError(\"Kokoro produced no audio\")\n    audio = np.concatenate(audio_parts)\n    if not np.isfinite(audio).all():\n        raise RuntimeError(\"Kokoro produced non-finite audio\")\n    return audio\n\n\ndef _audio_metrics(reference: np.ndarray, candidate: np.ndarray) -> dict[str, float]:\n    length = min(reference.size, candidate.size)\n    ref = np.asarray(reference[:length], dtype=np.float64)\n    cand = np.asarray(candidate[:length], dtype=np.float64)\n    error = ref - cand\n    alpha = float(np.dot(cand, ref) / max(np.dot(ref, ref), 1e-12))\n    target = alpha * ref\n    noise = cand - target\n    si_sdr = 10.0 * math.log10(max(np.dot(target, target), 1e-12) / max(np.dot(noise, noise), 1e-12))\n    return {\n        \"samples\": length,\n        \"duration_seconds\": candidate.size / SAMPLE_RATE,\n        \"mse\": float(np.mean(error**2)),\n        \"si_sdr_db\": si_sdr,\n        \"rms_dbfs\": float(20 * math.log10(max(np.sqrt(np.mean(cand**2)), 1e-12))),\n        \"peak\": float(np.max(np.abs(cand))),\n        \"clipped_fraction\": float(np.mean(np.abs(cand) >= 0.999)),\n    }\n\n\ndef _write_wav(path: Path, waveform: np.ndarray) -> None:\n    import soundfile as sf\n\n    path.parent.mkdir(parents=True, exist_ok=True)\n    sf.write(path, np.asarray(waveform, dtype=np.float32), SAMPLE_RATE)\n\n\ndef run_gate_a(cfg: AppConfig) -> dict[str, Any]:\n    for label, path in {\n        \"Kokoro model\": cfg.paths.kokoro_model,\n        \"Kokoro voices\": cfg.paths.kokoro_voices,\n        \"Mimi encoder\": cfg.paths.mimi_encoder,\n        \"Mimi decoder\": cfg.paths.mimi_decoder,\n        \"stateful Mimi weights\": cfg.paths.mimi_weights,\n    }.items():\n        if not path or not Path(path).is_file():\n            raise FileNotFoundError(f\"Gate A requires {label}: {path!r}\")\n    rows = load_manifest(cfg.work_dir / \"manifest.jsonl\")\n    rows.sort(key=lambda item: item[\"record_id\"])\n    rows = rows[: cfg.gates.codec_samples]\n    offline = OnnxMimi(cfg.paths.mimi_encoder, cfg.paths.mimi_decoder)\n    streaming = StatefulMimi(cfg.paths.mimi_weights)\n    output_dir = cfg.work_dir / \"gates\" / \"A_codec\"\n    results = []\n    all_exact = True\n    for position, row in enumerate(rows, 1):\n        source = synthesize_kokoro(row[\"answer\"], cfg.paths.kokoro_model, cfg.paths.kokoro_voices)\n        fresh_codes = offline.encode(source)\n        codes = np.load(Path(row[\"shard\"]) / \"codes.npy\", mmap_mode=\"r\")[\n            int(row[\"code_lo\"]) : int(row[\"code_hi\"])\n        ].astype(np.int64)\n        exact = fresh_codes.shape == codes.shape and np.array_equal(fresh_codes, codes)\n        all_exact = all_exact and exact\n        offline_audio = offline.decode(codes)\n        streaming_audio = streaming.decode(codes)\n        sample_dir = output_dir / str(row[\"record_id\"])\n        _write_wav(sample_dir / \"source.wav\", source)\n        _write_wav(sample_dir / \"stored_offline.wav\", offline_audio)\n        _write_wav(sample_dir / \"stored_streaming.wav\", streaming_audio)\n        result = {\n            \"record_id\": row[\"record_id\"],\n            \"codes_exact\": exact,\n            \"frames\": int(codes.shape[0]),\n            \"expected_seconds\": codes.shape[0] / FRAME_RATE,\n            \"offline\": _audio_metrics(source, offline_audio),\n            \"streaming\": _audio_metrics(source, streaming_audio),\n        }\n        results.append(result)\n        print(f\"Gate A {position}/{len(rows)} id={row['record_id']} exact={exact}\", flush=True)\n    automatic = all_exact and all(\n        item[\"offline\"][\"clipped_fraction\"] < 0.01\n        and item[\"streaming\"][\"clipped_fraction\"] < 0.01\n        and item[\"offline\"][\"samples\"] > 0\n        and item[\"streaming\"][\"samples\"] > 0\n        for item in results\n    )\n    report = {\n        \"automatic_pass\": automatic,\n        \"manual_required\": True,\n        \"samples\": results,\n        \"asset_hashes\": {\n            \"kokoro\": sha256_file(cfg.paths.kokoro_model),\n            \"voices\": sha256_file(cfg.paths.kokoro_voices),\n            \"mimi_encoder\": sha256_file(cfg.paths.mimi_encoder),\n            \"mimi_decoder\": sha256_file(cfg.paths.mimi_decoder),\n            \"mimi_weights\": sha256_file(cfg.paths.mimi_weights),\n        },\n    }\n    atomic_json(output_dir / \"report.json\", report)\n    return report\n\n\ndef run_gate_b(cfg: AppConfig) -> dict[str, Any]:\n    import matplotlib.pyplot as plt\n\n    dataset = SpeechDataset(cfg, \"val\", limit=cfg.gates.alignment_samples)\n    decoder = OnnxMimi(cfg.paths.mimi_encoder, cfg.paths.mimi_decoder)\n    output_dir = cfg.work_dir / \"gates\" / \"B_alignment\"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    links = []\n    for index in range(len(dataset)):\n        item = dataset[index]\n        waveform = decoder.decode(item[\"codes\"].numpy())\n        fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)\n        times = np.arange(waveform.size) / SAMPLE_RATE\n        axes[0].plot(times, waveform, linewidth=0.5)\n        axes[0].set(title=f\"{item['record_id']} waveform\", xlabel=\"seconds\")\n        axes[0].grid(alpha=0.2)\n        axes[1].specgram(waveform, NFFT=1024, Fs=SAMPLE_RATE, noverlap=768)\n        axes[1].set(title=item[\"text\"], xlabel=\"seconds\")\n        frame_cursor = 0\n        for token, duration in enumerate(item[\"durations\"].tolist()):\n            if duration:\n                axes[0].axvline(frame_cursor / FRAME_RATE, color=\"tab:blue\", alpha=0.15)\n            frame_cursor += duration\n        for word in item[\"words\"]:\n            boundary = int(word[\"start_frame\"]) / FRAME_RATE\n            axes[0].axvline(boundary, color=\"tab:red\", alpha=0.35)\n            axes[0].text(boundary, 0.95, str(word[\"word\"]), rotation=90, transform=axes[0].get_xaxis_transform())\n        name = f\"{index:03d}_{item['record_id']}.png\"\n        fig.savefig(output_dir / name, dpi=120)\n        plt.close(fig)\n        links.append({\"record_id\": item[\"record_id\"], \"image\": name})\n    rows = \"\\n\".join(\n        f'<article><h2>{html.escape(str(item[\"record_id\"]))}</h2><img src=\"{html.escape(item[\"image\"])}\" style=\"max-width:100%\"></article>'\n        for item in links\n    )\n    (output_dir / \"index.html\").write_text(\n        \"<!doctype html><meta charset='utf-8'><title>Alignment review</title>\"\n        \"<h1>Gate B alignment review</h1><p>Review every panel, then use approve-gate.</p>\" + rows,\n        encoding=\"utf-8\",\n    )\n    report = {\"automatic_pass\": len(links) >= cfg.gates.alignment_samples, \"manual_required\": True, \"panels\": links}\n    atomic_json(output_dir / \"report.json\", report)\n    return report\n\n\ndef model_structural_checks(cfg: AppConfig) -> dict[str, Any]:\n    import torch\n\n    from .model import StreamingSpeechAdapter, adapter_loss\n\n    torch.manual_seed(cfg.train.seed)\n    local_cfg = cfg.model\n    feature_marker_path = cfg.work_dir / \"features\" / \"features.done.json\"\n    if feature_marker_path.is_file():\n        feature_marker = json.loads(feature_marker_path.read_text())\n        text_dim = int(feature_marker[\"hidden_dim\"])\n        embedding_dim = int(\n            np.load(cfg.work_dir / \"features\" / \"embedding_values.npy\", mmap_mode=\"r\").shape[1]\n        )\n    else:\n        text_dim = embedding_dim = 1536\n    model = StreamingSpeechAdapter(text_dim, embedding_dim, local_cfg)\n    model.assert_budget()\n    model.eval()\n    batch, tokens, frames = 1, 6, 12\n    durations = torch.tensor([[2, 2, 2, 2, 2, 2]])\n    owners = torch.repeat_interleave(torch.arange(tokens), durations[0]).unsqueeze(0)\n    sample = {\n        \"hidden\": torch.randn(batch, tokens, text_dim),\n        \"embeddings\": torch.randn(batch, tokens, embedding_dim),\n        \"durations\": durations,\n        \"token_mask\": torch.ones(batch, tokens, dtype=torch.bool),\n        \"codes\": torch.randint(0, 2048, (batch, frames, 8)),\n        \"frame_mask\": torch.ones(batch, frames, dtype=torch.bool),\n        \"owners\": owners,\n    }\n    logits, log_duration = model(sample)\n    changed = {key: value.clone() if hasattr(value, \"clone\") else value for key, value in sample.items()}\n    changed[\"codes\"][:, 8:] = torch.randint(0, 2048, changed[\"codes\"][:, 8:].shape)\n    logits_changed, _ = model(changed)\n    causal = torch.allclose(logits[:, :8], logits_changed[:, :8], atol=1e-5, rtol=1e-4)\n    sequence_steps = local_cfg.audio_cache_frames + 5\n    temporal_input = torch.randn(1, sequence_steps, local_cfg.model_dim)\n    valid_mask = torch.ones(1, sequence_steps, dtype=torch.bool)\n    full = temporal_input\n    for block in model.temporal:\n        full = block(full, valid_mask)\n    head_dim = local_cfg.model_dim // local_cfg.attention_heads\n    key_caches = [\n        torch.zeros(1, local_cfg.attention_heads, local_cfg.audio_cache_frames, head_dim)\n        for _ in model.temporal\n    ]\n    value_caches = [torch.zeros_like(cache) for cache in key_caches]\n    incremental = []\n    cache_valid = 0\n    for step in range(sequence_steps):\n        value = temporal_input[:, step : step + 1]\n        next_valid = cache_valid\n        for layer, block in enumerate(model.temporal):\n            value, key_caches[layer], value_caches[layer], next_valid = block.step(\n                value, key_caches[layer], value_caches[layer], cache_valid, step\n            )\n        cache_valid = next_valid\n        incremental.append(value)\n    incremental_tensor = torch.cat(incremental, dim=1)\n    cache_equivalent = torch.allclose(full, incremental_tensor, atol=2e-5, rtol=2e-4)\n    model.train()\n    loss, _ = adapter_loss(logits, log_duration, sample, 4.0, 0.0, 0.0)\n    loss.backward()\n    cb0_gradient = model.hidden_projection.weight.grad is not None and float(\n        model.hidden_projection.weight.grad.abs().sum()\n    ) > 0\n    report = {\n        \"parameter_count\": model.parameter_count(),\n        \"fp16_weight_mib\": model.parameter_count() * 2 / 2**20,\n        \"int8_weight_mib\": model.parameter_count() / 2**20,\n        \"kv_cache_kib\": (\n            local_cfg.temporal_layers * 2 * local_cfg.audio_cache_frames\n            * local_cfg.model_dim * 2 / 1024\n        ),\n        \"causality_pass\": bool(causal),\n        \"cache_equivalence_pass\": bool(cache_equivalent),\n        \"cb0_shared_gradient_pass\": bool(cb0_gradient),\n        \"automatic_pass\": bool(\n            causal and cache_equivalent and cb0_gradient\n            and model.parameter_count() <= 12_000_000\n        ),\n    }\n    atomic_json(cfg.work_dir / \"gates\" / \"structural.json\", report)\n    return report\n\n\ndef update_gate(\n    cfg: AppConfig,\n    gate: str,\n    automatic_pass: bool,\n    manual_pass: bool | None,\n    reviewed: int = 0,\n    passed: int = 0,\n    note: str = \"\",\n) -> dict[str, Any]:\n    gate = gate.upper()\n    if gate not in \"ABCDEF\":\n        raise ValueError(\"gate must be A, B, C, D, E, or F\")\n    path = cfg.work_dir / \"gates\" / \"gate_report.json\"\n    report = json.loads(path.read_text()) if path.is_file() else {\n        \"feature_fingerprint\": cfg.feature_fingerprint(), \"gates\": {}\n    }\n    if report.get(\"feature_fingerprint\") != cfg.feature_fingerprint():\n        raise RuntimeError(\"existing gate report belongs to different features\")\n    if gate == \"B\" and manual_pass:\n        if reviewed < cfg.gates.alignment_samples or passed / max(1, reviewed) < cfg.gates.alignment_pass_fraction:\n            raise ValueError(\"Gate B approval does not meet the configured review threshold\")\n    status = \"passed\" if automatic_pass and manual_pass is True else \"pending\"\n    if manual_pass is False or not automatic_pass:\n        status = \"failed\"\n    report[\"gates\"][gate] = {\n        \"status\": status,\n        \"automatic_pass\": bool(automatic_pass),\n        \"manual_pass\": manual_pass,\n        \"reviewed\": int(reviewed),\n        \"passed\": int(passed),\n        \"note\": note,\n        \"updated_unix\": time.time(),\n    }\n    atomic_json(path, report)\n    return report\n\n\ndef preflight(cfg: AppConfig, require_two_gpus: bool = True) -> dict[str, Any]:\n    import os\n    import shutil\n    import torch\n\n    required = {\"data_dir\": cfg.paths.data_dir, \"source_jsonl\": cfg.paths.source_jsonl, \"qwen_model\": cfg.paths.qwen_model}\n    missing = [f\"{name}={path}\" for name, path in required.items() if not Path(path).exists()]\n    if missing:\n        raise FileNotFoundError(\"missing required inputs: \" + \", \".join(missing))\n    shards = discover_shards(cfg.paths.data_dir)\n    gpu_count = torch.cuda.device_count()\n    if require_two_gpus and gpu_count != 2:\n        raise RuntimeError(f\"select Kaggle GPU T4 x2; detected {gpu_count} CUDA devices\")\n    gpus = []\n    for index in range(gpu_count):\n        props = torch.cuda.get_device_properties(index)\n        gpus.append({\"index\": index, \"name\": props.name, \"vram_gib\": props.total_memory / 2**30})\n    disk_probe = cfg.work_dir\n    while not disk_probe.exists() and disk_probe != disk_probe.parent:\n        disk_probe = disk_probe.parent\n    free = shutil.disk_usage(disk_probe)\n    page_size = os.sysconf(\"SC_PAGE_SIZE\")\n    physical_pages = os.sysconf(\"SC_PHYS_PAGES\")\n    report = {\n        \"profile\": cfg.runtime.profile,\n        \"selected_device\": cfg.runtime.device,\n        \"python\": __import__(\"sys\").version,\n        \"torch\": torch.__version__,\n        \"cuda\": torch.version.cuda,\n        \"gpus\": gpus,\n        \"shards\": len(shards),\n        \"manifest_record_limit\": cfg.runtime.manifest_record_limit,\n        \"cpu_threads_configured\": cfg.runtime.cpu_threads,\n        \"logical_cpu_count\": os.cpu_count(),\n        \"physical_ram_gib\": page_size * physical_pages / 2**30,\n        \"free_working_gib\": free.free / 2**30,\n        \"config_fingerprint\": cfg.fingerprint(),\n    }\n    print(json.dumps(report, indent=2))\n    return report\n", "evaluation.py": "from __future__ import annotations\n\nimport csv\nimport html\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nfrom safetensors.torch import load_file\n\nfrom .config import AppConfig, atomic_json\nfrom .data import SpeechDataset, collate_speech\nfrom .diagnostics import OnnxMimi, _write_wav\nfrom .metrics import write_codebook_csv\nfrom .train import build_model, evaluate, move_batch\n\n\ndef load_best(cfg: AppConfig, device: torch.device):\n    model = build_model(cfg, device)\n    path = cfg.work_dir / \"checkpoints\" / \"best.safetensors\"\n    if not path.is_file():\n        raise FileNotFoundError(f\"best checkpoint not found: {path}\")\n    model.load_state_dict(load_file(str(path), device=str(device)))\n    model.eval()\n    return model\n\n\n@torch.no_grad()\ndef generate_artifacts(cfg: AppConfig, limit: int = 20) -> dict[str, Any]:\n    device = torch.device(\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n    model = load_best(cfg, device)\n    dataset = SpeechDataset(cfg, \"val\", limit=limit)\n    decoder = OnnxMimi(cfg.paths.mimi_encoder, cfg.paths.mimi_decoder)\n    output_dir = cfg.work_dir / \"evaluation\"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    metrics = {\n        mode: evaluate(model, dataset, device, limit, text_mode=mode)\n        for mode in (\"correct\", \"zeroed\", \"swapped\")\n    }\n    csv_path = output_dir / \"per_codebook.csv\"\n    if csv_path.exists():\n        csv_path.unlink()\n    for text_mode, result in metrics.items():\n        write_codebook_csv(csv_path, result[\"teacher_forced\"], f\"{text_mode}_teacher_forced\")\n        write_codebook_csv(csv_path, result[\"rollout\"], f\"{text_mode}_rollout\")\n    duration_rows = []\n    links = []\n    cb0_confusion = np.zeros((model.cfg.codec_vocab, model.cfg.codec_vocab), dtype=np.int64)\n    predicted_usage = np.zeros((model.cfg.num_codebooks, model.cfg.codec_vocab), dtype=np.int64)\n    for index in range(len(dataset)):\n        item = dataset[index]\n        batch = move_batch(collate_speech([item]), device)\n        logits, _ = model(batch)\n        frames = int(batch[\"frame_mask\"][0].sum().item())\n        teacher = logits[0, :frames].argmax(dim=-1)\n        hidden = batch[\"hidden\"][0, batch[\"token_mask\"][0]]\n        embeddings = batch[\"embeddings\"][0, batch[\"token_mask\"][0]]\n        oracle_duration = batch[\"durations\"][0, batch[\"token_mask\"][0]]\n        rollout_oracle, predicted_duration, _ = model.rollout_with_logits(\n            hidden, embeddings, oracle_duration\n        )\n        rollout_predicted, _, _ = model.rollout_with_logits(hidden, embeddings, None)\n        oracle = batch[\"codes\"][0, :frames]\n        oracle_np = oracle.detach().cpu().numpy()\n        rollout_np = rollout_oracle.detach().cpu().numpy()\n        np.add.at(cb0_confusion, (oracle_np[:, 0], rollout_np[:, 0]), 1)\n        for book in range(model.cfg.num_codebooks):\n            predicted_usage[book] += np.bincount(\n                rollout_np[:, book], minlength=model.cfg.codec_vocab\n            )\n        cb0_pred_residual_oracle = oracle.clone()\n        cb0_pred_residual_oracle[:, 0] = rollout_oracle[:, 0]\n        cb0_oracle_residual_pred = rollout_oracle.clone()\n        cb0_oracle_residual_pred[:, 0] = oracle[:, 0]\n        modes = {\n            \"oracle\": oracle,\n            \"teacher_forced\": teacher,\n            \"rollout_oracle_duration\": rollout_oracle,\n            \"rollout_predicted_duration\": rollout_predicted,\n            \"cb0_predicted_residual_oracle\": cb0_pred_residual_oracle,\n            \"cb0_oracle_residual_predicted\": cb0_oracle_residual_pred,\n        }\n        sample_dir = output_dir / \"audio\" / str(item[\"record_id\"])\n        sample_links = []\n        for name, codes in modes.items():\n            wav = decoder.decode(codes.detach().cpu().numpy())\n            path = sample_dir / f\"{name}.wav\"\n            _write_wav(path, wav)\n            sample_links.append((name, path.relative_to(output_dir)))\n        true = oracle_duration.float()\n        error = (predicted_duration.float() - true).abs()\n        for token_index in range(true.numel()):\n            duration_rows.append(\n                {\n                    \"record_id\": item[\"record_id\"],\n                    \"token_index\": token_index,\n                    \"true_frames\": int(true[token_index].item()),\n                    \"predicted_frames\": int(predicted_duration[token_index].item()),\n                    \"absolute_error\": float(error[token_index].item()),\n                    \"absolute_percentage_error\": float(\n                        error[token_index].item() / max(1.0, true[token_index].item())\n                    ),\n                }\n            )\n        duration_rows.append(\n            {\n                \"record_id\": item[\"record_id\"],\n                \"token_index\": \"ALL\",\n                \"true_frames\": int(true.sum().item()),\n                \"predicted_frames\": int(predicted_duration.sum().item()),\n                \"absolute_error\": float(abs(predicted_duration.sum().item() - true.sum().item())),\n                \"absolute_percentage_error\": float(\n                    abs(predicted_duration.sum().item() - true.sum().item())\n                    / max(1.0, true.sum().item())\n                ),\n            }\n        )\n        links.append({\"record_id\": item[\"record_id\"], \"text\": item[\"text\"], \"files\": sample_links})\n    with (output_dir / \"duration.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as handle:\n        fields = list(duration_rows[0]) if duration_rows else [\"record_id\"]\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(duration_rows)\n    page = [\"<!doctype html><meta charset='utf-8'><h1>Streaming speech listening index</h1>\"]\n    for sample in links:\n        page.append(f\"<h2>{html.escape(str(sample['record_id']))}</h2><p>{html.escape(sample['text'])}</p>\")\n        for mode, relative in sample[\"files\"]:\n            source = html.escape(str(relative))\n            page.append(f\"<p>{html.escape(mode)}<br><audio controls src='{source}'></audio></p>\")\n    (output_dir / \"listening.html\").write_text(\"\\n\".join(page), encoding=\"utf-8\")\n    np.save(output_dir / \"cb0_confusion.npy\", cb0_confusion)\n    np.save(output_dir / \"predicted_code_usage.npy\", predicted_usage)\n    try:\n        import matplotlib.pyplot as plt\n\n        top = np.argsort(cb0_confusion.sum(axis=1))[-64:]\n        fig, axis = plt.subplots(figsize=(9, 8))\n        image = axis.imshow(np.log1p(cb0_confusion[np.ix_(top, top)]), aspect=\"auto\")\n        axis.set(title=\"CB0 confusion (top oracle codes, log counts)\", xlabel=\"predicted\", ylabel=\"oracle\")\n        fig.colorbar(image, ax=axis)\n        fig.tight_layout()\n        fig.savefig(output_dir / \"cb0_confusion.png\", dpi=140)\n        plt.close(fig)\n        fig, axes = plt.subplots(4, 2, figsize=(14, 12), constrained_layout=True)\n        for book, axis in enumerate(axes.flat):\n            used = np.flatnonzero(predicted_usage[book])\n            axis.bar(used, predicted_usage[book, used], width=1.0)\n            axis.set(title=f\"CB{book} predicted usage\", xlabel=\"code\", ylabel=\"count\")\n        fig.savefig(output_dir / \"code_usage.png\", dpi=140)\n        plt.close(fig)\n        training_path = cfg.work_dir / \"training.json\"\n        if training_path.is_file():\n            history = json.loads(training_path.read_text()).get(\"history\", [])\n            if history:\n                fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)\n                axes[0].plot([row[\"epoch\"] for row in history], [row[\"train_loss\"] for row in history])\n                axes[0].set(title=\"Training loss\", xlabel=\"epoch\")\n                axes[1].plot(\n                    [row[\"epoch\"] for row in history],\n                    [row[\"validation\"][\"rollout_cb0_ce\"] for row in history],\n                )\n                axes[1].set(title=\"Validation rollout CB0 CE\", xlabel=\"epoch\")\n                fig.savefig(output_dir / \"training_curves.png\", dpi=140)\n                plt.close(fig)\n    except Exception as exc:\n        (output_dir / \"plot_warning.txt\").write_text(str(exc), encoding=\"utf-8\")\n    atomic_json(output_dir / \"metrics.json\", metrics)\n    conclusion = [\"# Experiment conclusion\", \"\", f\"Evaluated {len(dataset)} validation samples.\", \"\"]\n    for mode, result in metrics.items():\n        conclusion.append(f\"- {mode}: rollout CB0 CE = {result['rollout_cb0_ce']:.4f}\")\n    (output_dir / \"conclusion.md\").write_text(\"\\n\".join(conclusion), encoding=\"utf-8\")\n    return metrics\n", "export.py": "from __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom safetensors.torch import load_file\n\nfrom .config import AppConfig, atomic_json\nfrom .train import build_model\n\n\nclass TextStepExport(nn.Module):\n    def __init__(self, model):\n        super().__init__()\n        self.model = model\n\n    def forward(\n        self,\n        hidden: torch.Tensor,\n        embedding: torch.Tensor,\n        cache: torch.Tensor,\n        valid_mask: torch.Tensor,\n    ):\n        fused = self.model.fuse_text(hidden, embedding).unsqueeze(1)\n        next_cache = torch.cat((cache[:, 1:], fused), dim=1)\n        next_mask = torch.cat(\n            (valid_mask[:, 1:], torch.ones_like(valid_mask[:, :1])), dim=1\n        )\n        all_context = self.model.text_context(next_cache, next_mask)\n        context = all_context[:, -1]\n        duration = self.model.duration_head(context).squeeze(-1)\n        return context, duration, next_cache, next_mask\n\n\nclass AudioStepExport(nn.Module):\n    def __init__(self, model):\n        super().__init__()\n        self.model = model\n\n    def forward(\n        self,\n        text_context: torch.Tensor,\n        previous_codes: torch.Tensor,\n        first_frame: torch.Tensor,\n        key_caches: torch.Tensor,\n        value_caches: torch.Tensor,\n        valid_mask: torch.Tensor,\n        position: torch.Tensor,\n    ):\n        previous = self.model.embed_frame(previous_codes[:, None, :])[:, 0]\n        bos = self.model.audio_bos.view(1, -1).expand_as(previous)\n        previous = torch.where(first_frame.reshape(-1, 1), bos, previous)\n        value = self.model.audio_input(torch.cat((text_context, previous), dim=-1)).unsqueeze(1)\n        next_keys = []\n        next_values = []\n        next_mask = torch.cat((valid_mask[:, 1:], torch.ones_like(valid_mask[:, :1])), dim=1)\n        for layer, block in enumerate(self.model.temporal):\n            normalized = block.attn_norm(value)\n            q, k, v = block.attn.qkv(normalized).chunk(3, dim=-1)\n            q, k, v = block.attn._split(q), block.attn._split(k), block.attn._split(v)\n            angles = position.float().reshape(1, 1) * block.attn.rope.inv_freq.float().reshape(1, -1)\n            cos = torch.cat((angles.cos(), angles.cos()), dim=-1)[None].to(q.dtype)\n            sin = torch.cat((angles.sin(), angles.sin()), dim=-1)[None].to(q.dtype)\n            q = q * cos + torch.cat((-q[..., q.shape[-1] // 2 :], q[..., : q.shape[-1] // 2]), -1) * sin\n            k = k * cos + torch.cat((-k[..., k.shape[-1] // 2 :], k[..., : k.shape[-1] // 2]), -1) * sin\n            layer_k = torch.cat((key_caches[layer, :, :, 1:], k), dim=2)\n            layer_v = torch.cat((value_caches[layer, :, :, 1:], v), dim=2)\n            scores = torch.matmul(q, layer_k.transpose(-1, -2)) / math.sqrt(block.attn.head_dim)\n            scores = scores.masked_fill(\n                ~next_mask[:, None, None], torch.finfo(scores.dtype).min\n            )\n            weights = torch.softmax(scores.float(), dim=-1).to(scores.dtype)\n            update = torch.matmul(weights, layer_v).transpose(1, 2).reshape(value.shape)\n            value = value + block.attn.out(update)\n            value = value + block.ffn(block.ffn_norm(value))\n            next_keys.append(layer_k)\n            next_values.append(layer_v)\n        hidden = self.model.temporal_norm(value[:, 0])\n        prefix_sum = torch.zeros_like(hidden)\n        codes = []\n        logits = []\n        for book in range(self.model.cfg.num_codebooks):\n            prefix = prefix_sum / math.sqrt(max(book, 1))\n            conditioned = self.model.depth_norm(hidden + self.model.depth_mix(prefix))\n            book_logits = conditioned @ self.model.codec_weight[book] + self.model.codec_bias[book]\n            code = book_logits.argmax(dim=-1)\n            logits.append(book_logits)\n            codes.append(code)\n            prefix_sum = prefix_sum + self.model.code_embedding(\n                code + book * self.model.cfg.codec_vocab\n            )\n        return (\n            torch.stack(codes, dim=-1),\n            torch.stack(logits, dim=1),\n            torch.stack(next_keys),\n            torch.stack(next_values),\n            next_mask,\n            position + 1,\n        )\n\n\ndef _ort_session(path: Path):\n    import onnxruntime as ort\n\n    return ort.InferenceSession(str(path), providers=[\"CPUExecutionProvider\"])\n\n\ndef export_adapter(\n    cfg: AppConfig,\n    parity_steps: int = 100,\n    checkpoint_path: str = \"\",\n    output_dir_path: str = \"\",\n) -> dict[str, Any]:\n    device = torch.device(\"cpu\")\n    model = build_model(cfg, device)\n    checkpoint = (\n        Path(checkpoint_path) if checkpoint_path\n        else cfg.work_dir / \"checkpoints\" / \"best.safetensors\"\n    )\n    model.load_state_dict(load_file(str(checkpoint), device=\"cpu\"))\n    model.eval()\n    output_dir = Path(output_dir_path) if output_dir_path else cfg.work_dir / \"export\"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    text_model, audio_model = TextStepExport(model).eval(), AudioStepExport(model).eval()\n    text_dim = model.hidden_projection.in_features\n    embedding_dim = model.embedding_projection.in_features\n    dim, context = model.cfg.model_dim, model.cfg.text_context_size\n    heads = model.cfg.attention_heads\n    head_dim = dim // heads\n    layers, window = model.cfg.temporal_layers, model.cfg.audio_cache_frames\n    text_inputs = (\n        torch.randn(1, text_dim),\n        torch.randn(1, embedding_dim),\n        torch.zeros(1, context, dim),\n        torch.zeros(1, context, dtype=torch.bool),\n    )\n    audio_inputs = (\n        torch.randn(1, dim),\n        torch.zeros(1, 8, dtype=torch.long),\n        torch.ones(1, dtype=torch.bool),\n        torch.zeros(layers, 1, heads, window, head_dim),\n        torch.zeros(layers, 1, heads, window, head_dim),\n        torch.zeros(1, window, dtype=torch.bool),\n        torch.zeros(1, dtype=torch.long),\n    )\n    text_path, audio_path = output_dir / \"text_step.onnx\", output_dir / \"audio_step.onnx\"\n    torch.onnx.export(\n        text_model,\n        text_inputs,\n        text_path,\n        input_names=[\"hidden\", \"embedding\", \"text_cache\", \"text_valid\"],\n        output_names=[\"context\", \"log_duration\", \"next_text_cache\", \"next_text_valid\"],\n        opset_version=18,\n        dynamo=False,\n    )\n    torch.onnx.export(\n        audio_model,\n        audio_inputs,\n        audio_path,\n        input_names=[\n            \"text_context\", \"previous_codes\", \"first_frame\", \"key_caches\",\n            \"value_caches\", \"audio_valid\", \"position\",\n        ],\n        output_names=[\n            \"codes\", \"logits\", \"next_key_caches\", \"next_value_caches\",\n            \"next_audio_valid\", \"next_position\",\n        ],\n        opset_version=18,\n        dynamo=False,\n    )\n    text_session, audio_session = _ort_session(text_path), _ort_session(audio_path)\n    rng = np.random.default_rng(cfg.train.seed)\n    text_cache = np.zeros((1, context, dim), np.float32)\n    text_valid = np.zeros((1, context), bool)\n    key_cache = np.zeros((layers, 1, heads, window, head_dim), np.float32)\n    value_cache = np.zeros_like(key_cache)\n    audio_valid = np.zeros((1, window), bool)\n    position = np.zeros(1, np.int64)\n    previous = np.zeros((1, 8), np.int64)\n    max_text_error = max_audio_error = 0.0\n    cb0_agreement = []\n    context_value = None\n    for step in range(parity_steps):\n        hidden = rng.standard_normal((1, text_dim)).astype(np.float32)\n        embedding = rng.standard_normal((1, embedding_dim)).astype(np.float32)\n        torch_text = text_model(\n            torch.from_numpy(hidden), torch.from_numpy(embedding),\n            torch.from_numpy(text_cache), torch.from_numpy(text_valid),\n        )\n        ort_text = text_session.run(None, {\n            \"hidden\": hidden, \"embedding\": embedding,\n            \"text_cache\": text_cache, \"text_valid\": text_valid,\n        })\n        max_text_error = max(max_text_error, float(np.max(np.abs(torch_text[0].detach().numpy() - ort_text[0]))))\n        context_value, text_cache, text_valid = ort_text[0], ort_text[2], ort_text[3]\n        first = np.asarray([step == 0], dtype=bool)\n        torch_audio = audio_model(\n            torch.from_numpy(context_value), torch.from_numpy(previous), torch.from_numpy(first),\n            torch.from_numpy(key_cache), torch.from_numpy(value_cache),\n            torch.from_numpy(audio_valid), torch.from_numpy(position),\n        )\n        ort_audio = audio_session.run(None, {\n            \"text_context\": context_value, \"previous_codes\": previous,\n            \"first_frame\": first, \"key_caches\": key_cache,\n            \"value_caches\": value_cache, \"audio_valid\": audio_valid,\n            \"position\": position,\n        })\n        torch_logits = torch_audio[1].detach().numpy()\n        max_audio_error = max(max_audio_error, float(np.max(np.abs(torch_logits - ort_audio[1]))))\n        cb0_agreement.append(bool(torch_logits[:, 0].argmax(-1)[0] == ort_audio[1][:, 0].argmax(-1)[0]))\n        previous, key_cache, value_cache, audio_valid, position = (\n            ort_audio[0], ort_audio[2], ort_audio[3], ort_audio[4], ort_audio[5]\n        )\n    quantized = output_dir / \"audio_step.int8.onnx\"\n    quantization_error = None\n    int8_cb0_agreement = None\n    int8_max_logit_error = None\n    try:\n        from onnxruntime.quantization import QuantType, quantize_dynamic\n\n        quantize_dynamic(str(audio_path), str(quantized), weight_type=QuantType.QInt8)\n        int8_session = _ort_session(quantized)\n        rng = np.random.default_rng(cfg.train.seed + 1)\n        fp_keys = np.zeros((layers, 1, heads, window, head_dim), np.float32)\n        fp_values = np.zeros_like(fp_keys)\n        q_keys = fp_keys.copy()\n        q_values = fp_values.copy()\n        fp_mask = np.zeros((1, window), bool)\n        q_mask = fp_mask.copy()\n        fp_pos = np.zeros(1, np.int64)\n        q_pos = fp_pos.copy()\n        fp_previous = np.zeros((1, 8), np.int64)\n        q_previous = fp_previous.copy()\n        agreements = []\n        errors = []\n        for step in range(parity_steps):\n            context_input = rng.standard_normal((1, dim)).astype(np.float32)\n            first = np.asarray([step == 0], dtype=bool)\n            fp = audio_session.run(None, {\n                \"text_context\": context_input, \"previous_codes\": fp_previous,\n                \"first_frame\": first, \"key_caches\": fp_keys,\n                \"value_caches\": fp_values, \"audio_valid\": fp_mask,\n                \"position\": fp_pos,\n            })\n            quant = int8_session.run(None, {\n                \"text_context\": context_input, \"previous_codes\": q_previous,\n                \"first_frame\": first, \"key_caches\": q_keys,\n                \"value_caches\": q_values, \"audio_valid\": q_mask,\n                \"position\": q_pos,\n            })\n            agreements.append(bool(fp[1][:, 0].argmax(-1)[0] == quant[1][:, 0].argmax(-1)[0]))\n            errors.append(float(np.max(np.abs(fp[1] - quant[1]))))\n            fp_previous, fp_keys, fp_values, fp_mask, fp_pos = fp[0], fp[2], fp[3], fp[4], fp[5]\n            q_previous, q_keys, q_values, q_mask, q_pos = (\n                quant[0], quant[2], quant[3], quant[4], quant[5]\n            )\n        int8_cb0_agreement = float(np.mean(agreements))\n        int8_max_logit_error = float(max(errors))\n    except Exception as exc:\n        quantization_error = f\"quantization failed: {exc}\"\n    report = {\n        \"steps\": parity_steps,\n        \"max_text_absolute_error\": max_text_error,\n        \"max_audio_logit_absolute_error\": max_audio_error,\n        \"cb0_argmax_agreement\": float(np.mean(cb0_agreement)),\n        \"pytorch_onnx_pass\": max_text_error < 1e-3 and max_audio_error < 1e-3 and all(cb0_agreement),\n        \"text_onnx_bytes\": text_path.stat().st_size,\n        \"audio_onnx_bytes\": audio_path.stat().st_size,\n        \"int8_audio_onnx_bytes\": quantized.stat().st_size if quantized.is_file() else None,\n        \"int8_cb0_argmax_agreement\": int8_cb0_agreement,\n        \"int8_max_logit_error\": int8_max_logit_error,\n        \"int8_pass\": bool(int8_cb0_agreement is not None and int8_cb0_agreement >= 0.99),\n        \"quantization_note\": quantization_error,\n    }\n    atomic_json(output_dir / \"export_report.json\", report)\n    return report\n", "metrics.py": "from __future__ import annotations\n\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\nclass CodebookMetrics:\n    def __init__(self, books: int = 8, vocabulary: int = 2048):\n        self.books = books\n        self.vocabulary = vocabulary\n        self.frames = np.zeros(books, dtype=np.int64)\n        self.loss = np.zeros(books, dtype=np.float64)\n        self.top1 = np.zeros(books, dtype=np.int64)\n        self.top5 = np.zeros(books, dtype=np.int64)\n        self.entropy = np.zeros(books, dtype=np.float64)\n        self.histogram = np.zeros((books, vocabulary), dtype=np.int64)\n        self.repeat_frames = np.zeros(books, dtype=np.int64)\n        self.longest_repeat = np.zeros(books, dtype=np.int64)\n\n    @torch.no_grad()\n    def update(self, logits: torch.Tensor, labels: torch.Tensor, mask: torch.Tensor) -> None:\n        for b in range(logits.shape[0]):\n            valid = int(mask[b].sum().item())\n            if valid < 1:\n                continue\n            sample_logits = logits[b, :valid].float()\n            sample_labels = labels[b, :valid]\n            losses = F.cross_entropy(\n                sample_logits.reshape(-1, self.vocabulary), sample_labels.reshape(-1), reduction=\"none\"\n            ).view(valid, self.books)\n            probabilities = torch.softmax(sample_logits, dim=-1)\n            predicted = sample_logits.argmax(dim=-1)\n            top5 = sample_logits.topk(5, dim=-1).indices\n            entropy = -(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=-1)\n            for book in range(self.books):\n                pred = predicted[:, book].cpu().numpy()\n                self.frames[book] += valid\n                self.loss[book] += float(losses[:, book].sum().item())\n                self.top1[book] += int((predicted[:, book] == sample_labels[:, book]).sum().item())\n                self.top5[book] += int(\n                    (top5[:, book] == sample_labels[:, book, None]).any(dim=-1).sum().item()\n                )\n                self.entropy[book] += float(entropy[:, book].sum().item())\n                self.histogram[book] += np.bincount(pred, minlength=self.vocabulary)\n                run = 1\n                longest = 1\n                repeated = 0\n                for left, right in zip(pred, pred[1:]):\n                    if left == right:\n                        run += 1\n                        repeated += 1\n                        longest = max(longest, run)\n                    else:\n                        run = 1\n                self.repeat_frames[book] += repeated\n                self.longest_repeat[book] = max(self.longest_repeat[book], longest)\n\n    def report(self) -> list[dict[str, Any]]:\n        rows = []\n        for book in range(self.books):\n            count = max(1, int(self.frames[book]))\n            ce = float(self.loss[book] / count)\n            used = int(np.count_nonzero(self.histogram[book]))\n            rows.append(\n                {\n                    \"codebook\": book,\n                    \"frames\": int(self.frames[book]),\n                    \"cross_entropy\": ce,\n                    \"perplexity\": float(math.exp(min(ce, 30.0))),\n                    \"top1_accuracy\": float(self.top1[book] / count),\n                    \"top5_accuracy\": float(self.top5[book] / count),\n                    \"predicted_entropy\": float(self.entropy[book] / count),\n                    \"codes_used\": used,\n                    \"codes_used_fraction\": float(used / self.vocabulary),\n                    \"repeated_fraction\": float(self.repeat_frames[book] / max(1, count - 1)),\n                    \"longest_repeat\": int(self.longest_repeat[book]),\n                }\n            )\n        return rows\n\n\ndef sequence_accuracy(predicted: torch.Tensor, target: torch.Tensor) -> dict[str, Any]:\n    frames = min(int(predicted.shape[0]), int(target.shape[0]))\n    if frames == 0:\n        return {\"frames\": 0, \"per_codebook_accuracy\": [0.0] * target.shape[-1]}\n    matches = predicted[:frames] == target[:frames]\n    cb0 = matches[:, 0]\n    incorrect = torch.nonzero(~cb0)\n    first_error = int(incorrect[0].item()) if incorrect.numel() else frames\n    after = cb0[first_error + 1 :] if first_error + 1 < frames else cb0[:0]\n    buckets = []\n    for start in range(0, frames, 25):\n        buckets.append(\n            {\n                \"start\": start,\n                \"end\": min(frames, start + 25),\n                \"cb0_accuracy\": float(cb0[start : start + 25].float().mean().item()),\n            }\n        )\n    return {\n        \"frames\": frames,\n        \"per_codebook_accuracy\": matches.float().mean(dim=0).cpu().tolist(),\n        \"first_cb0_error\": first_error if first_error < frames else None,\n        \"cb0_accuracy_after_first_error\": float(after.float().mean().item()) if after.numel() else None,\n        \"position_buckets\": buckets,\n    }\n\n\ndef write_codebook_csv(path: str | Path, rows: list[dict[str, Any]], mode: str) -> None:\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    fields = [\"mode\"] + list(rows[0].keys()) if rows else [\"mode\"]\n    exists = target.is_file()\n    with target.open(\"a\", newline=\"\", encoding=\"utf-8\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        if not exists:\n            writer.writeheader()\n        for row in rows:\n            writer.writerow({\"mode\": mode, **row})\n\n\ndef _loss_trend(current: float, previous: float | None) -> tuple[str, float | None]:\n    \"\"\"Classify CE movement while ignoring normal sub-percent validation noise.\"\"\"\n    if previous is None or not math.isfinite(previous):\n        return \"baseline\", None\n    delta = current - previous\n    tolerance = max(0.01, abs(previous) * 0.005)\n    if delta < -tolerance:\n        return \"improving\", delta\n    if delta > tolerance:\n        return \"worsening\", delta\n    return \"flat\", delta\n\n\ndef training_health_signals(\n    validation: dict[str, Any],\n    previous_validation: dict[str, Any] | None = None,\n    vocabulary: int = 2048,\n) -> dict[str, Any]:\n    \"\"\"Turn teacher-forced and autoregressive validation metrics into clear signals.\n\n    Autoregressive metrics use the model's own previous Mimi codes while retaining\n    oracle token durations. Duration quality is reported independently.\n    \"\"\"\n    teacher = validation.get(\"teacher_forced\", [])\n    rollout = validation.get(\"rollout\", [])\n    if not teacher or len(teacher) != len(rollout):\n        raise ValueError(\"validation must contain matching teacher-forced and rollout codebooks\")\n    previous_teacher = (previous_validation or {}).get(\"teacher_forced\", [])\n    previous_rollout = (previous_validation or {}).get(\"rollout\", [])\n    oracle_usage = validation.get(\"oracle_codes_used\", [])\n    oracle_entropy = validation.get(\"oracle_entropy\", [])\n    random_ce = math.log(vocabulary)\n    books: list[dict[str, Any]] = []\n    warnings: list[str] = []\n    improving = worsening = collapsed = 0\n\n    for index, (tf, ar) in enumerate(zip(teacher, rollout)):\n        tf_ce = float(tf[\"cross_entropy\"])\n        ar_ce = float(ar[\"cross_entropy\"])\n        prior_tf = (\n            float(previous_teacher[index][\"cross_entropy\"])\n            if index < len(previous_teacher) else None\n        )\n        prior_ar = (\n            float(previous_rollout[index][\"cross_entropy\"])\n            if index < len(previous_rollout) else None\n        )\n        tf_trend, tf_delta = _loss_trend(tf_ce, prior_tf)\n        ar_trend, ar_delta = _loss_trend(ar_ce, prior_ar)\n        used_reference = int(oracle_usage[index]) if index < len(oracle_usage) else 0\n        entropy_reference = float(oracle_entropy[index]) if index < len(oracle_entropy) else 0.0\n        usage_ratio = float(ar[\"codes_used\"]) / max(1, used_reference)\n        entropy_ratio = float(ar[\"predicted_entropy\"]) / max(1e-9, entropy_reference)\n        collapse = bool(usage_ratio < 0.25 or entropy_ratio < 0.50)\n        gap = ar_ce - tf_ce\n\n        if ar_trend == \"improving\":\n            improving += 1\n        elif ar_trend == \"worsening\":\n            worsening += 1\n        if collapse:\n            collapsed += 1\n\n        if ar_trend == \"worsening\" or collapse or gap > 1.0:\n            signal = \"red\"\n        elif ar_trend in {\"baseline\", \"flat\"} or gap > 0.35:\n            signal = \"yellow\"\n        else:\n            signal = \"green\"\n        books.append(\n            {\n                \"codebook\": int(tf.get(\"codebook\", index)),\n                \"signal\": signal,\n                \"teacher_ce\": tf_ce,\n                \"teacher_ce_vs_random\": tf_ce - random_ce,\n                \"teacher_trend\": tf_trend,\n                \"teacher_delta\": tf_delta,\n                \"teacher_top1\": float(tf[\"top1_accuracy\"]),\n                \"ar_ce\": ar_ce,\n                \"ar_ce_vs_random\": ar_ce - random_ce,\n                \"ar_trend\": ar_trend,\n                \"ar_delta\": ar_delta,\n                \"ar_top1\": float(ar[\"top1_accuracy\"]),\n                \"ar_teacher_ce_gap\": gap,\n                \"codes_used\": int(ar[\"codes_used\"]),\n                \"oracle_codes_used\": used_reference,\n                \"code_usage_ratio\": usage_ratio,\n                \"entropy_ratio\": entropy_ratio,\n                \"repeated_fraction\": float(ar[\"repeated_fraction\"]),\n                \"longest_repeat\": int(ar[\"longest_repeat\"]),\n                \"collapse_warning\": collapse,\n            }\n        )\n\n    cb0 = books[0]\n    if cb0[\"ar_trend\"] == \"worsening\":\n        warnings.append(\"CB0 autoregressive loss worsened; intelligibility may be regressing\")\n    if cb0[\"collapse_warning\"]:\n        warnings.append(\"CB0 code usage/entropy is collapsing\")\n    if cb0[\"ar_teacher_ce_gap\"] > 1.0:\n        warnings.append(\"CB0 teacher-to-AR gap is severe; exposure error is dominating\")\n    elif cb0[\"ar_teacher_ce_gap\"] > 0.35:\n        warnings.append(\"CB0 teacher-to-AR gap is growing\")\n    if collapsed:\n        warnings.append(f\"{collapsed}/{len(books)} codebooks show possible code collapse\")\n    duration_ape = validation.get(\"duration_median_ape\")\n    if duration_ape is not None and float(duration_ape) > 0.50:\n        warnings.append(\"median duration error exceeds 50%\")\n\n    has_previous = bool(previous_teacher and previous_rollout)\n    if not all(math.isfinite(float(row[\"teacher_ce\"])) and math.isfinite(float(row[\"ar_ce\"])) for row in books):\n        overall = \"red\"\n        warnings.insert(0, \"non-finite validation loss detected\")\n    elif cb0[\"signal\"] == \"red\" or worsening >= max(2, len(books) // 2) or collapsed >= max(2, len(books) // 2):\n        overall = \"red\"\n    elif has_previous and cb0[\"ar_trend\"] == \"improving\" and improving >= max(1, len(books) // 2) and not warnings:\n        overall = \"green\"\n    else:\n        overall = \"yellow\"\n\n    if overall == \"green\":\n        headline = \"teacher-forced and autoregressive validation are improving together\"\n    elif overall == \"red\":\n        headline = warnings[0] if warnings else \"validation shows a material regression\"\n    elif not has_previous:\n        headline = \"baseline epoch recorded; trend signals begin next epoch\"\n    else:\n        headline = \"training needs observation; improvement is not yet consistent\"\n    return {\n        \"overall_signal\": overall,\n        \"headline\": headline,\n        \"evaluation_protocol\": \"teacher-forced plus autoregressive Mimi codes with oracle token durations\",\n        \"random_guess_cross_entropy\": random_ce,\n        \"samples\": int(validation.get(\"samples\", 0)),\n        \"improving_codebooks\": improving,\n        \"flat_codebooks\": len(books) - improving - worsening,\n        \"worsening_codebooks\": worsening,\n        \"collapsed_codebooks\": collapsed,\n        \"cb0\": cb0,\n        \"duration_mae_frames\": validation.get(\"duration_mae_frames\"),\n        \"duration_median_ape\": duration_ape,\n        \"warnings\": warnings,\n        \"codebooks\": books,\n    }\n\n\ndef write_epoch_signal_csv(path: str | Path, history: list[dict[str, Any]]) -> None:\n    \"\"\"Rewrite a resume-safe epoch/codebook table from canonical training history.\"\"\"\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    fields = [\n        \"epoch\", \"overall_signal\", \"codebook\", \"signal\", \"teacher_ce\",\n        \"teacher_trend\", \"teacher_delta\", \"teacher_top1\", \"ar_ce\", \"ar_trend\",\n        \"ar_delta\", \"ar_top1\", \"ar_teacher_ce_gap\", \"code_usage_ratio\",\n        \"entropy_ratio\", \"repeated_fraction\", \"longest_repeat\", \"collapse_warning\",\n    ]\n    temporary = target.with_suffix(target.suffix + \".tmp\")\n    with temporary.open(\"w\", newline=\"\", encoding=\"utf-8\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        for epoch_row in history:\n            signal = epoch_row.get(\"signals\", {})\n            for book in signal.get(\"codebooks\", []):\n                writer.writerow(\n                    {\n                        \"epoch\": epoch_row[\"epoch\"],\n                        \"overall_signal\": signal.get(\"overall_signal\"),\n                        **{field: book.get(field) for field in fields if field not in {\"epoch\", \"overall_signal\"}},\n                    }\n                )\n    temporary.replace(target)\n\n\ndef atomic_metrics(path: str | Path, metrics: dict[str, Any]) -> None:\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_suffix(target.suffix + \".tmp\")\n    temporary.write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding=\"utf-8\")\n    temporary.replace(target)\n", "model.py": "from __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\nfrom typing import Any\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .config import ModelConfig\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim: int, eps: float = 1.0e-6):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n        self.eps = eps\n\n    def forward(self, value: torch.Tensor) -> torch.Tensor:\n        scale = value.float().pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()\n        return (value * scale.to(value.dtype)) * self.weight\n\n\ndef _rotate_half(value: torch.Tensor) -> torch.Tensor:\n    left, right = value.chunk(2, dim=-1)\n    return torch.cat((-right, left), dim=-1)\n\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, head_dim: int, base: float = 10_000.0):\n        super().__init__()\n        if head_dim % 2:\n            raise ValueError(\"rotary head dimension must be even\")\n        inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))\n        self.register_buffer(\"inv_freq\", inv, persistent=False)\n\n    def apply(\n        self, query: torch.Tensor, key: torch.Tensor, positions: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        angles = torch.outer(positions.float(), self.inv_freq.float())\n        cos = torch.cat((angles.cos(), angles.cos()), dim=-1)[None, None].to(query.dtype)\n        sin = torch.cat((angles.sin(), angles.sin()), dim=-1)[None, None].to(query.dtype)\n        return query * cos + _rotate_half(query) * sin, key * cos + _rotate_half(key) * sin\n\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, dim: int, heads: int, window: int):\n        super().__init__()\n        if dim % heads:\n            raise ValueError(\"attention dimension must be divisible by heads\")\n        self.dim = dim\n        self.heads = heads\n        self.head_dim = dim // heads\n        self.window = int(window)\n        self.qkv = nn.Linear(dim, dim * 3, bias=False)\n        self.out = nn.Linear(dim, dim, bias=False)\n        self.rope = RotaryEmbedding(self.head_dim)\n\n    def _split(self, value: torch.Tensor) -> torch.Tensor:\n        batch, steps, _ = value.shape\n        return value.view(batch, steps, self.heads, self.head_dim).transpose(1, 2)\n\n    def forward(self, value: torch.Tensor, valid_mask: torch.Tensor) -> torch.Tensor:\n        batch, steps, _ = value.shape\n        q, k, v = self.qkv(value).chunk(3, dim=-1)\n        q, k, v = self._split(q), self._split(k), self._split(v)\n        positions = torch.arange(steps, device=value.device)\n        q, k = self.rope.apply(q, k, positions)\n        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)\n        row = positions[:, None]\n        col = positions[None, :]\n        causal = (col <= row) & (col >= row - self.window + 1)\n        allowed = causal[None, None] & valid_mask[:, None, None, :]\n        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)\n        weights = torch.softmax(scores.float(), dim=-1).to(scores.dtype)\n        weights = torch.nan_to_num(weights)\n        output = torch.matmul(weights, v).transpose(1, 2).reshape(batch, steps, self.dim)\n        return self.out(output) * valid_mask.unsqueeze(-1)\n\n    def step(\n        self,\n        value: torch.Tensor,\n        key_cache: torch.Tensor,\n        value_cache: torch.Tensor,\n        valid: int,\n        position: int,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, int]:\n        \"\"\"One fixed-cache step. Caches are chronological with newest at the end.\"\"\"\n        if value.shape[1] != 1:\n            raise ValueError(\"incremental attention expects exactly one step\")\n        q, k, v = self.qkv(value).chunk(3, dim=-1)\n        q, k, v = self._split(q), self._split(k), self._split(v)\n        pos = torch.tensor([position], device=value.device)\n        q, k = self.rope.apply(q, k, pos)\n        next_k = torch.roll(key_cache, shifts=-1, dims=2)\n        next_v = torch.roll(value_cache, shifts=-1, dims=2)\n        next_k[:, :, -1:, :] = k\n        next_v[:, :, -1:, :] = v\n        next_valid = min(self.window, int(valid) + 1)\n        scores = torch.matmul(q, next_k.transpose(-1, -2)) / math.sqrt(self.head_dim)\n        slots = torch.arange(self.window, device=value.device)\n        allowed = slots >= self.window - next_valid\n        scores = scores.masked_fill(~allowed[None, None, None], torch.finfo(scores.dtype).min)\n        weights = torch.softmax(scores.float(), dim=-1).to(scores.dtype)\n        output = torch.matmul(weights, next_v).transpose(1, 2).reshape(value.shape[0], 1, self.dim)\n        return self.out(output), next_k, next_v, next_valid\n\n\nclass CausalBlock(nn.Module):\n    def __init__(self, dim: int, heads: int, ffn_dim: int, window: int, dropout: float):\n        super().__init__()\n        self.attn_norm = RMSNorm(dim)\n        self.attn = CausalSelfAttention(dim, heads, window)\n        self.ffn_norm = RMSNorm(dim)\n        self.ffn = nn.Sequential(\n            nn.Linear(dim, ffn_dim),\n            nn.SiLU(),\n            nn.Dropout(dropout),\n            nn.Linear(ffn_dim, dim),\n        )\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, value: torch.Tensor, valid_mask: torch.Tensor) -> torch.Tensor:\n        value = value + self.dropout(self.attn(self.attn_norm(value), valid_mask))\n        value = value + self.dropout(self.ffn(self.ffn_norm(value))) * valid_mask.unsqueeze(-1)\n        return value\n\n    def step(\n        self,\n        value: torch.Tensor,\n        key_cache: torch.Tensor,\n        value_cache: torch.Tensor,\n        valid: int,\n        position: int,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, int]:\n        update, next_k, next_v, next_valid = self.attn.step(\n            self.attn_norm(value), key_cache, value_cache, valid, position\n        )\n        value = value + update\n        value = value + self.ffn(self.ffn_norm(value))\n        return value, next_k, next_v, next_valid\n\n\nclass TextContextAttention(nn.Module):\n    def __init__(self, dim: int, heads: int, window: int):\n        super().__init__()\n        self.window = int(window)\n        self.attn = CausalSelfAttention(dim, heads, window)\n        self.norm = RMSNorm(dim)\n\n    def forward(self, value: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n        return self.norm(value + self.attn(value, mask))\n\n    def step(\n        self, value: torch.Tensor, cache: torch.Tensor, valid: int\n    ) -> tuple[torch.Tensor, torch.Tensor, int]:\n        next_cache = torch.roll(cache, shifts=-1, dims=1)\n        next_cache[:, -1:, :] = value\n        next_valid = min(self.window, int(valid) + 1)\n        q = self.attn._split(self.attn.qkv(value).chunk(3, dim=-1)[0])\n        projected = self.attn.qkv(next_cache)\n        _, keys, values = projected.chunk(3, dim=-1)\n        keys, values = self.attn._split(keys), self.attn._split(values)\n        positions = torch.arange(self.window, device=value.device)\n        query_pos = torch.tensor([self.window - 1], device=value.device)\n        q, _ = self.attn.rope.apply(q, q, query_pos)\n        dummy = keys\n        _, keys = self.attn.rope.apply(dummy, keys, positions)\n        scores = torch.matmul(q, keys.transpose(-1, -2)) / math.sqrt(self.attn.head_dim)\n        allowed = positions >= self.window - next_valid\n        scores = scores.masked_fill(~allowed[None, None, None], torch.finfo(scores.dtype).min)\n        weights = torch.softmax(scores.float(), dim=-1).to(scores.dtype)\n        context = torch.matmul(weights, values).transpose(1, 2).reshape(value.shape)\n        context = self.attn.out(context)\n        return self.norm(value + context), next_cache, next_valid\n\n\n@dataclass\nclass StreamingState:\n    text_cache: torch.Tensor\n    text_valid: int\n    key_caches: list[torch.Tensor]\n    value_caches: list[torch.Tensor]\n    audio_valid: int\n    audio_position: int\n    previous_codes: torch.Tensor | None\n\n\nclass StreamingSpeechAdapter(nn.Module):\n    def __init__(self, text_dim: int, embedding_dim: int, cfg: ModelConfig):\n        super().__init__()\n        self.cfg = cfg\n        dim = cfg.model_dim\n        self.hidden_norm = nn.LayerNorm(text_dim)\n        self.embedding_norm = nn.LayerNorm(embedding_dim)\n        self.hidden_projection = nn.Linear(text_dim, dim)\n        self.embedding_projection = nn.Linear(embedding_dim, dim)\n        self.fusion_norm = RMSNorm(dim)\n        self.text_context = TextContextAttention(dim, cfg.attention_heads, cfg.text_context_size)\n        self.duration_head = nn.Sequential(nn.Linear(dim, dim // 2), nn.SiLU(), nn.Linear(dim // 2, 1))\n        self.code_embedding = nn.Embedding(cfg.num_codebooks * cfg.codec_vocab, dim)\n        self.audio_bos = nn.Parameter(torch.randn(dim) * (dim ** -0.5))\n        self.audio_input = nn.Linear(dim * 2, dim)\n        self.temporal = nn.ModuleList(\n            CausalBlock(dim, cfg.attention_heads, cfg.ffn_dim, cfg.audio_cache_frames, cfg.dropout)\n            for _ in range(cfg.temporal_layers)\n        )\n        self.temporal_norm = RMSNorm(dim)\n        self.depth_mix = nn.Linear(dim, dim, bias=False)\n        self.depth_norm = RMSNorm(dim)\n        self.codec_weight = nn.Parameter(\n            torch.randn(cfg.num_codebooks, dim, cfg.codec_vocab) * (dim ** -0.5)\n        )\n        self.codec_bias = nn.Parameter(torch.zeros(cfg.num_codebooks, cfg.codec_vocab))\n\n    def parameter_count(self) -> int:\n        return sum(parameter.numel() for parameter in self.parameters())\n\n    def assert_budget(self, maximum: int = 12_000_000) -> None:\n        count = self.parameter_count()\n        if count > maximum:\n            raise RuntimeError(f\"adapter has {count:,} parameters, above budget {maximum:,}\")\n\n    def fuse_text(self, hidden: torch.Tensor, embeddings: torch.Tensor) -> torch.Tensor:\n        fused = self.hidden_projection(self.hidden_norm(hidden))\n        fused = fused + self.embedding_projection(self.embedding_norm(embeddings))\n        return self.fusion_norm(fused)\n\n    def encode_text(\n        self, hidden: torch.Tensor, embeddings: torch.Tensor, token_mask: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        fused = self.fuse_text(hidden, embeddings)\n        context = self.text_context(fused, token_mask)\n        duration = self.duration_head(context).squeeze(-1)\n        return context * token_mask.unsqueeze(-1), duration\n\n    def embed_frame(self, codes: torch.Tensor) -> torch.Tensor:\n        books = torch.arange(self.cfg.num_codebooks, device=codes.device).view(1, 1, -1)\n        values = self.code_embedding(codes.long() + books * self.cfg.codec_vocab)\n        return values.sum(dim=-2) / math.sqrt(self.cfg.num_codebooks)\n\n    def _previous_frame_embeddings(self, codes: torch.Tensor) -> torch.Tensor:\n        batch, frames, _ = codes.shape\n        embedded = self.embed_frame(codes)\n        bos = self.audio_bos.view(1, 1, -1).expand(batch, 1, -1)\n        return torch.cat((bos, embedded[:, :-1]), dim=1) if frames else embedded\n\n    def frame_logits(self, frame_hidden: torch.Tensor, teacher_codes: torch.Tensor) -> torch.Tensor:\n        books = torch.arange(self.cfg.num_codebooks, device=teacher_codes.device).view(1, 1, -1)\n        values = self.code_embedding(teacher_codes.long() + books * self.cfg.codec_vocab)\n        inclusive = values.cumsum(dim=2)\n        prefix = torch.cat((torch.zeros_like(inclusive[:, :, :1]), inclusive[:, :, :-1]), dim=2)\n        counts = torch.arange(self.cfg.num_codebooks, device=teacher_codes.device).float().sqrt().clamp(min=1)\n        prefix = prefix / counts.view(1, 1, -1, 1)\n        conditioned = self.depth_norm(frame_hidden[:, :, None, :] + self.depth_mix(prefix))\n        return torch.einsum(\"btkd,kdv->btkv\", conditioned, self.codec_weight) + self.codec_bias\n\n    def forward(self, batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:\n        text_context, log_duration = self.encode_text(\n            batch[\"hidden\"], batch[\"embeddings\"], batch[\"token_mask\"]\n        )\n        gather = batch[\"owners\"].unsqueeze(-1).expand(-1, -1, self.cfg.model_dim)\n        frame_text = torch.gather(text_context, 1, gather)\n        previous = self._previous_frame_embeddings(batch[\"codes\"])\n        frame_hidden = self.audio_input(torch.cat((frame_text, previous), dim=-1))\n        for block in self.temporal:\n            frame_hidden = block(frame_hidden, batch[\"frame_mask\"])\n        frame_hidden = self.temporal_norm(frame_hidden)\n        return self.frame_logits(frame_hidden, batch[\"codes\"]), log_duration\n\n    def make_streaming_state(self, batch_size: int, device: torch.device, dtype: torch.dtype) -> StreamingState:\n        head_dim = self.cfg.model_dim // self.cfg.attention_heads\n        shape = (batch_size, self.cfg.attention_heads, self.cfg.audio_cache_frames, head_dim)\n        return StreamingState(\n            text_cache=torch.zeros(batch_size, self.cfg.text_context_size, self.cfg.model_dim, device=device, dtype=dtype),\n            text_valid=0,\n            key_caches=[torch.zeros(shape, device=device, dtype=dtype) for _ in self.temporal],\n            value_caches=[torch.zeros(shape, device=device, dtype=dtype) for _ in self.temporal],\n            audio_valid=0,\n            audio_position=0,\n            previous_codes=None,\n        )\n\n    def text_step(\n        self, hidden: torch.Tensor, embedding: torch.Tensor, state: StreamingState\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        fused = self.fuse_text(hidden, embedding).unsqueeze(1)\n        context, state.text_cache, state.text_valid = self.text_context.step(\n            fused, state.text_cache, state.text_valid\n        )\n        duration = self.duration_head(context[:, 0]).squeeze(-1)\n        return context[:, 0], duration\n\n    def audio_step(\n        self, text_context: torch.Tensor, state: StreamingState\n    ) -> tuple[torch.Tensor, list[torch.Tensor]]:\n        batch = text_context.shape[0]\n        if state.previous_codes is None:\n            previous = self.audio_bos.view(1, -1).expand(batch, -1)\n        else:\n            previous = self.embed_frame(state.previous_codes[:, None, :])[:, 0]\n        value = self.audio_input(torch.cat((text_context, previous), dim=-1)).unsqueeze(1)\n        next_keys: list[torch.Tensor] = []\n        next_values: list[torch.Tensor] = []\n        next_valid = state.audio_valid\n        for index, block in enumerate(self.temporal):\n            value, key, val, layer_valid = block.step(\n                value,\n                state.key_caches[index],\n                state.value_caches[index],\n                state.audio_valid,\n                state.audio_position,\n            )\n            next_keys.append(key)\n            next_values.append(val)\n            next_valid = layer_valid\n        state.key_caches, state.value_caches = next_keys, next_values\n        state.audio_valid = next_valid\n        state.audio_position += 1\n        hidden = self.temporal_norm(value[:, 0])\n        generated: list[torch.Tensor] = []\n        prefix_sum = torch.zeros(batch, self.cfg.model_dim, device=hidden.device, dtype=hidden.dtype)\n        logits_by_book: list[torch.Tensor] = []\n        for book in range(self.cfg.num_codebooks):\n            prefix = prefix_sum / math.sqrt(max(book, 1))\n            conditioned = self.depth_norm(hidden + self.depth_mix(prefix))\n            logits = conditioned @ self.codec_weight[book] + self.codec_bias[book]\n            code = logits.argmax(dim=-1)\n            logits_by_book.append(logits)\n            generated.append(code)\n            embedded = self.code_embedding(code + book * self.cfg.codec_vocab)\n            prefix_sum = prefix_sum + embedded\n        codes = torch.stack(generated, dim=-1)\n        state.previous_codes = codes\n        return codes, logits_by_book\n\n    @torch.no_grad()\n    def rollout(\n        self,\n        hidden: torch.Tensor,\n        embeddings: torch.Tensor,\n        durations: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        self.eval()\n        device, dtype = hidden.device, next(self.parameters()).dtype\n        state = self.make_streaming_state(1, device, dtype)\n        contexts = []\n        predicted_durations = []\n        for index in range(hidden.shape[0]):\n            context, log_duration = self.text_step(hidden[index : index + 1], embeddings[index : index + 1], state)\n            contexts.append(context)\n            predicted_durations.append(torch.expm1(log_duration.float()).clamp(0, 250).round().long())\n        predicted = torch.cat(predicted_durations)\n        selected = durations.long() if durations is not None else predicted\n        frames = []\n        for context, count in zip(contexts, selected.tolist()):\n            for _ in range(int(count)):\n                codes, _ = self.audio_step(context, state)\n                frames.append(codes[0])\n        output = torch.stack(frames) if frames else torch.empty(0, self.cfg.num_codebooks, dtype=torch.long, device=device)\n        return output, predicted\n\n    @torch.no_grad()\n    def rollout_with_logits(\n        self,\n        hidden: torch.Tensor,\n        embeddings: torch.Tensor,\n        durations: torch.Tensor | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        self.eval()\n        device, dtype = hidden.device, next(self.parameters()).dtype\n        state = self.make_streaming_state(1, device, dtype)\n        contexts: list[torch.Tensor] = []\n        predicted_durations: list[torch.Tensor] = []\n        for index in range(hidden.shape[0]):\n            context, log_duration = self.text_step(\n                hidden[index : index + 1], embeddings[index : index + 1], state\n            )\n            contexts.append(context)\n            predicted_durations.append(\n                torch.expm1(log_duration.float()).clamp(0, 250).round().long()\n            )\n        predicted = torch.cat(predicted_durations)\n        selected = durations.long() if durations is not None else predicted\n        frames: list[torch.Tensor] = []\n        frame_logits: list[torch.Tensor] = []\n        for context, count in zip(contexts, selected.tolist()):\n            for _ in range(int(count)):\n                codes, logits = self.audio_step(context, state)\n                frames.append(codes[0])\n                frame_logits.append(torch.stack([book[0] for book in logits]))\n        codes_out = (\n            torch.stack(frames)\n            if frames\n            else torch.empty(0, self.cfg.num_codebooks, dtype=torch.long, device=device)\n        )\n        logits_out = (\n            torch.stack(frame_logits)\n            if frame_logits\n            else torch.empty(\n                0, self.cfg.num_codebooks, self.cfg.codec_vocab, device=device, dtype=dtype\n            )\n        )\n        return codes_out, predicted, logits_out\n\n\ndef adapter_loss(\n    logits: torch.Tensor,\n    log_duration: torch.Tensor,\n    batch: dict[str, torch.Tensor],\n    cb0_weight: float,\n    residual_weight: float,\n    duration_weight: float,\n) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:\n    vocabulary = logits.shape[-1]\n    raw = F.cross_entropy(\n        logits.float().reshape(-1, vocabulary), batch[\"codes\"].reshape(-1), reduction=\"none\"\n    ).view_as(batch[\"codes\"])\n    weights = torch.tensor(\n        [cb0_weight] + [residual_weight] * (batch[\"codes\"].shape[-1] - 1),\n        device=logits.device,\n    )\n    valid = batch[\"frame_mask\"].unsqueeze(-1)\n    codec = (raw * weights.view(1, 1, -1) * valid).sum() / (\n        valid.sum().clamp(min=1) * weights.sum()\n    )\n    target = torch.log1p(batch[\"durations\"].float())\n    duration = ((log_duration - target).square() * batch[\"token_mask\"]).sum() / batch[\n        \"token_mask\"\n    ].sum().clamp(min=1)\n    total = codec + duration_weight * duration\n    per_book = (raw * valid).sum(dim=(0, 1)) / valid.sum().clamp(min=1)\n    return total, {\"codec\": codec.detach(), \"duration\": duration.detach(), \"per_book\": per_book.detach()}\n", "smoke.py": "from __future__ import annotations\n\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .config import AppConfig, atomic_json\n\n\ndef _peak_rss_mib() -> float | None:\n    status = Path(\"/proc/self/status\")\n    if not status.is_file():\n        return None\n    for line in status.read_text(encoding=\"utf-8\").splitlines():\n        if line.startswith(\"VmHWM:\"):\n            return float(line.split()[1]) / 1024.0\n    return None\n\n\ndef _small_token_slice(item: dict[str, Any], maximum_frames: int = 8) -> dict[str, Any]:\n    \"\"\"Keep one real aligned token so CPU and CUDA smoke steps stay bounded.\"\"\"\n    import torch\n\n    durations = item[\"durations\"].long()\n    positive = torch.nonzero(durations > 0, as_tuple=False).flatten()\n    if positive.numel() == 0:\n        raise RuntimeError(f\"record {item['record_id']} has no token with audio frames\")\n    token = int(positive[0].item())\n    frame_start = int(durations[:token].sum().item())\n    frame_count = min(int(durations[token].item()), int(maximum_frames))\n    return {\n        \"record_id\": item[\"record_id\"],\n        \"text\": item[\"text\"],\n        \"hidden\": item[\"hidden\"][token : token + 1],\n        \"embeddings\": item[\"embeddings\"][token : token + 1],\n        \"durations\": torch.tensor([frame_count], dtype=torch.long),\n        \"codes\": item[\"codes\"][frame_start : frame_start + frame_count],\n        \"char_offsets\": item[\"char_offsets\"][token : token + 1],\n        \"words\": item[\"words\"],\n    }\n\n\ndef run_smoke(cfg: AppConfig) -> dict[str, Any]:\n    \"\"\"Exercise real prepared data on the configured CPU or CUDA device.\"\"\"\n    import torch\n\n    from .data import SpeechDataset, collate_speech\n    from .diagnostics import model_structural_checks\n    from .model import adapter_loss\n    from .train import build_model, move_batch\n\n    requested_device = cfg.runtime.device\n    if requested_device == \"cuda\" and not torch.cuda.is_available():\n        raise RuntimeError(\"smoke requires CUDA, but torch.cuda.is_available() is false\")\n    use_cuda = requested_device == \"cuda\" or (\n        requested_device == \"auto\" and torch.cuda.is_available()\n    )\n    if use_cuda:\n        local_rank = int(os.environ.get(\"LOCAL_RANK\", \"0\"))\n        if local_rank >= torch.cuda.device_count():\n            raise RuntimeError(\n                f\"LOCAL_RANK={local_rank} but only {torch.cuda.device_count()} CUDA devices exist\"\n            )\n        device = torch.device(f\"cuda:{local_rank}\")\n        torch.cuda.set_device(device)\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats(device)\n    else:\n        device = torch.device(\"cpu\")\n        torch.set_num_threads(cfg.runtime.cpu_threads)\n    structural = model_structural_checks(cfg)\n\n    dataset = SpeechDataset(cfg, \"train\", limit=1)\n    selected_split = \"train\"\n    if len(dataset) == 0:\n        dataset = SpeechDataset(cfg, \"val\", limit=1)\n        selected_split = \"val\"\n    if len(dataset) == 0:\n        raise RuntimeError(\"the limited manifest contains no train or validation record\")\n\n    item = _small_token_slice(dataset[0])\n    batch = move_batch(collate_speech([item]), device)\n    model = build_model(cfg, device)\n    model.train()\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.train.learning_rate)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_cuda)\n    optimizer.zero_grad(set_to_none=True)\n    started = time.perf_counter()\n    with torch.autocast(device_type=\"cuda\", dtype=torch.float16, enabled=use_cuda):\n        logits, log_duration = model(batch)\n        loss, parts = adapter_loss(\n            logits,\n            log_duration,\n            batch,\n            cfg.train.cb0_weight,\n            cfg.train.residual_weight,\n            cfg.train.duration_weight,\n        )\n    scaler.scale(loss).backward()\n    scaler.unscale_(optimizer)\n    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]\n    gradients_finite = bool(gradients) and all(\n        bool(torch.isfinite(gradient).all()) for gradient in gradients\n    )\n    gradient_norm = math.sqrt(\n        sum(float(gradient.detach().float().square().sum().item()) for gradient in gradients)\n    )\n    scaler.step(optimizer)\n    scaler.update()\n\n    model.eval()\n    rollout_duration = torch.tensor(\n        [min(2, int(item[\"durations\"][0]))], dtype=torch.long, device=device\n    )\n    generated, predicted_duration = model.rollout(\n        batch[\"hidden\"][0, :1], batch[\"embeddings\"][0, :1], rollout_duration\n    )\n    if use_cuda:\n        torch.cuda.synchronize(device)\n    elapsed = time.perf_counter() - started\n    feature_marker = json.loads(\n        (cfg.work_dir / \"features\" / \"features.done.json\").read_text(encoding=\"utf-8\")\n    )\n    report = {\n        \"automatic_pass\": bool(\n            structural[\"automatic_pass\"]\n            and torch.isfinite(loss)\n            and gradients_finite\n            and generated.shape == (int(rollout_duration[0]), cfg.model.num_codebooks)\n        ),\n        \"requested_device\": requested_device,\n        \"device\": str(device),\n        \"device_type\": device.type,\n        \"amp_enabled\": use_cuda,\n        \"elapsed_seconds\": elapsed,\n        \"record_id\": item[\"record_id\"],\n        \"split\": selected_split,\n        \"bounded_tokens\": 1,\n        \"bounded_frames\": int(item[\"codes\"].shape[0]),\n        \"loss\": float(loss.detach()),\n        \"codec_loss\": float(parts[\"codec\"]),\n        \"duration_loss\": float(parts[\"duration\"]),\n        \"loss_per_codebook\": [float(value) for value in parts[\"per_book\"]],\n        \"cb0_loss\": float(parts[\"per_book\"][0]),\n        \"gradient_norm\": gradient_norm,\n        \"gradients_finite\": gradients_finite,\n        \"rollout_frames\": int(generated.shape[0]),\n        \"rollout_codebooks\": int(generated.shape[1]),\n        \"predicted_duration\": int(predicted_duration[0]),\n        \"peak_process_rss_mib\": _peak_rss_mib(),\n        \"cuda\": {\n            \"available\": torch.cuda.is_available(),\n            \"device_count\": torch.cuda.device_count(),\n            \"name\": torch.cuda.get_device_name(device) if use_cuda else None,\n            \"capability\": list(torch.cuda.get_device_capability(device)) if use_cuda else None,\n            \"peak_allocated_mib\": (\n                torch.cuda.max_memory_allocated(device) / 2**20 if use_cuda else 0.0\n            ),\n            \"peak_reserved_mib\": (\n                torch.cuda.max_memory_reserved(device) / 2**20 if use_cuda else 0.0\n            ),\n        },\n        \"feature_preparation\": {\n            \"device\": feature_marker.get(\"preparation_device\"),\n            \"seconds_rank0\": feature_marker.get(\"preparation_seconds_rank0\"),\n            \"peak_rss_mib_rank0\": feature_marker.get(\"preparation_peak_rss_mib_rank0\"),\n        },\n        \"structural\": structural,\n    }\n    atomic_json(cfg.work_dir / \"smoke_report.json\", report)\n    atomic_json(cfg.work_dir / f\"{device.type}_smoke_report.json\", report)\n    if device.type == \"cpu\":\n        atomic_json(cfg.work_dir / \"local_smoke_report.json\", report)\n    return report\n\n\ndef run_local_smoke(cfg: AppConfig) -> dict[str, Any]:\n    \"\"\"Backward-compatible alias for callers using the old function name.\"\"\"\n    return run_smoke(cfg)\n", "train.py": "from __future__ import annotations\n\nimport contextlib\nimport datetime as dt\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom safetensors.torch import load_file, save_file\nfrom torch.nn.parallel import DistributedDataParallel\nfrom torch.utils.data import DataLoader\n\nfrom .config import AppConfig, atomic_json\nfrom .data import (\n    DistributedFrameBatchSampler,\n    SpeechDataset,\n    collate_speech,\n)\nfrom .metrics import (\n    CodebookMetrics,\n    sequence_accuracy,\n    training_health_signals,\n    write_epoch_signal_csv,\n)\nfrom .model import StreamingSpeechAdapter, adapter_loss\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef distributed(cfg: AppConfig) -> tuple[int, int, int]:\n    rank = int(os.environ.get(\"RANK\", \"0\"))\n    world = int(os.environ.get(\"WORLD_SIZE\", \"1\"))\n    local = int(os.environ.get(\"LOCAL_RANK\", rank))\n    if world > 1 and not dist.is_initialized():\n        use_cuda = cfg.runtime.device == \"cuda\" or (\n            cfg.runtime.device == \"auto\" and torch.cuda.is_available()\n        )\n        dist.init_process_group(\"nccl\" if use_cuda else \"gloo\", timeout=dt.timedelta(hours=2))\n    return rank, world, local\n\n\ndef move_batch(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:\n    return {\n        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value\n        for key, value in batch.items()\n    }\n\n\ndef build_model(cfg: AppConfig, device: torch.device) -> StreamingSpeechAdapter:\n    marker = json.loads((cfg.work_dir / \"features\" / \"features.done.json\").read_text())\n    embeddings = np.load(cfg.work_dir / \"features\" / \"embedding_values.npy\", mmap_mode=\"r\")\n    model = StreamingSpeechAdapter(\n        int(marker[\"hidden_dim\"]), int(embeddings.shape[1]), cfg.model\n    ).to(device)\n    model.assert_budget()\n    return model\n\n\ndef _candidate_batch(dataset: SpeechDataset, frame_budget: int, maximum: int) -> dict[str, Any]:\n    chosen = []\n    total = 0\n    for index in sorted(range(len(dataset)), key=dataset.lengths.__getitem__, reverse=True):\n        length = dataset.lengths[index]\n        if length > frame_budget:\n            continue\n        if chosen and (total + length > frame_budget or len(chosen) >= maximum):\n            continue\n        chosen.append(dataset[index])\n        total += length\n        if total >= int(frame_budget * 0.8) or len(chosen) >= maximum:\n            break\n    if not chosen:\n        longest = max(dataset.lengths)\n        raise RuntimeError(f\"frame budget {frame_budget} is below every sample; longest={longest}\")\n    return collate_speech(chosen)\n\n\ndef calibrate_budget(\n    model: StreamingSpeechAdapter,\n    dataset: SpeechDataset,\n    cfg: AppConfig,\n    device: torch.device,\n) -> int:\n    candidate = int(cfg.train.max_frames_per_gpu)\n    attempts = 0\n    while True:\n        attempts += 1\n        model.zero_grad(set_to_none=True)\n        try:\n            batch = move_batch(_candidate_batch(dataset, candidate, cfg.train.max_batch_size), device)\n            with torch.autocast(device_type=\"cuda\", dtype=torch.float16, enabled=device.type == \"cuda\"):\n                logits, duration = model(batch)\n                loss, _ = adapter_loss(\n                    logits,\n                    duration,\n                    batch,\n                    cfg.train.cb0_weight,\n                    cfg.train.residual_weight,\n                    cfg.train.duration_weight,\n                )\n            loss.backward()\n            model.zero_grad(set_to_none=True)\n            del batch, logits, duration, loss\n            if device.type == \"cuda\":\n                torch.cuda.empty_cache()\n            break\n        except torch.cuda.OutOfMemoryError:\n            model.zero_grad(set_to_none=True)\n            torch.cuda.empty_cache()\n            if attempts >= 2:\n                raise RuntimeError(\n                    f\"calibration OOM after retry; set max_frames_per_gpu below {candidate}\"\n                )\n            candidate = max(64, candidate // 2)\n            print(f\"CUDA OOM during calibration; retrying with {candidate} frames/GPU\", flush=True)\n    if dist.is_initialized():\n        tensor = torch.tensor(candidate, device=device, dtype=torch.long)\n        dist.all_reduce(tensor, op=dist.ReduceOp.MIN)\n        candidate = int(tensor.item())\n    return candidate\n\n\ndef _atomic_weights(model: StreamingSpeechAdapter, path: Path) -> None:\n    target = model.module if isinstance(model, DistributedDataParallel) else model\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \".tmp\")\n    save_file({key: value.detach().cpu() for key, value in target.state_dict().items()}, str(temporary))\n    temporary.replace(path)\n\n\ndef save_checkpoint(\n    model: StreamingSpeechAdapter,\n    optimizer: torch.optim.Optimizer,\n    scheduler: Any,\n    scaler: torch.amp.GradScaler,\n    epoch: int,\n    best: float,\n    cfg: AppConfig,\n    name: str,\n    run_dir: Path,\n) -> None:\n    directory = run_dir / \"checkpoints\"\n    directory.mkdir(parents=True, exist_ok=True)\n    _atomic_weights(model, directory / f\"{name}.safetensors\")\n    state = {\n        \"epoch\": epoch,\n        \"best_rollout_cb0_ce\": best,\n        \"optimizer\": optimizer.state_dict(),\n        \"scheduler\": scheduler.state_dict(),\n        \"scaler\": scaler.state_dict(),\n        \"python_rng\": random.getstate(),\n        \"numpy_rng\": np.random.get_state(),\n        \"torch_rng\": torch.get_rng_state(),\n        \"cuda_rng\": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n        \"config_fingerprint\": cfg.fingerprint(),\n        \"feature_fingerprint\": cfg.feature_fingerprint(),\n    }\n    temporary = directory / f\"{name}.state.pt.tmp\"\n    torch.save(state, temporary)\n    temporary.replace(directory / f\"{name}.state.pt\")\n\n\ndef maybe_resume(\n    model: StreamingSpeechAdapter,\n    optimizer: torch.optim.Optimizer,\n    scheduler: Any,\n    scaler: torch.amp.GradScaler,\n    cfg: AppConfig,\n    device: torch.device,\n    run_dir: Path,\n) -> tuple[int, float]:\n    directory = run_dir / \"checkpoints\"\n    weights = directory / \"last.safetensors\"\n    state_path = directory / \"last.state.pt\"\n    if not cfg.train.resume or not weights.is_file() or not state_path.is_file():\n        return 0, math.inf\n    state = torch.load(state_path, map_location=\"cpu\", weights_only=False)\n    if state.get(\"config_fingerprint\") != cfg.fingerprint():\n        raise RuntimeError(\"checkpoint configuration differs from the current configuration\")\n    if state.get(\"feature_fingerprint\") != cfg.feature_fingerprint():\n        raise RuntimeError(\"checkpoint feature dataset differs from current sidecars\")\n    model.load_state_dict(load_file(str(weights), device=str(device)))\n    optimizer.load_state_dict(state[\"optimizer\"])\n    scheduler.load_state_dict(state[\"scheduler\"])\n    scaler.load_state_dict(state[\"scaler\"])\n    random.setstate(state[\"python_rng\"])\n    np.random.set_state(state[\"numpy_rng\"])\n    torch.set_rng_state(state[\"torch_rng\"])\n    if torch.cuda.is_available() and state[\"cuda_rng\"]:\n        torch.cuda.set_rng_state_all(state[\"cuda_rng\"])\n    return int(state[\"epoch\"]), float(state[\"best_rollout_cb0_ce\"])\n\n\n@torch.no_grad()\ndef evaluate(\n    model: StreamingSpeechAdapter,\n    dataset: SpeechDataset,\n    device: torch.device,\n    limit: int,\n    text_mode: str = \"correct\",\n) -> dict[str, Any]:\n    model.eval()\n    teacher = CodebookMetrics(model.cfg.num_codebooks, model.cfg.codec_vocab)\n    rollout = CodebookMetrics(model.cfg.num_codebooks, model.cfg.codec_vocab)\n    rollout_sequences = []\n    duration_abs = []\n    duration_ape = []\n    oracle_histogram = np.zeros((model.cfg.num_codebooks, model.cfg.codec_vocab), dtype=np.int64)\n    count = min(len(dataset), limit if limit > 0 else len(dataset))\n    for index in range(count):\n        item = dataset[index]\n        if text_mode == \"zeroed\":\n            item = dict(item)\n            item[\"hidden\"] = torch.zeros_like(item[\"hidden\"])\n            item[\"embeddings\"] = torch.zeros_like(item[\"embeddings\"])\n        elif text_mode == \"swapped\":\n            donor = dataset[(index + 1) % len(dataset)]\n            if donor[\"hidden\"].shape[0] != item[\"hidden\"].shape[0]:\n                slots = torch.arange(item[\"hidden\"].shape[0]) % donor[\"hidden\"].shape[0]\n                hidden, embeddings = donor[\"hidden\"][slots], donor[\"embeddings\"][slots]\n            else:\n                hidden, embeddings = donor[\"hidden\"], donor[\"embeddings\"]\n            item = dict(item, hidden=hidden, embeddings=embeddings)\n        batch = move_batch(collate_speech([item]), device)\n        logits, log_duration = model(batch)\n        teacher.update(logits, batch[\"codes\"], batch[\"frame_mask\"])\n        generated, predicted_duration, rollout_logits = model.rollout_with_logits(\n            batch[\"hidden\"][0, batch[\"token_mask\"][0]],\n            batch[\"embeddings\"][0, batch[\"token_mask\"][0]],\n            batch[\"durations\"][0, batch[\"token_mask\"][0]],\n        )\n        target = batch[\"codes\"][0, batch[\"frame_mask\"][0]]\n        target_np = target.detach().cpu().numpy()\n        for book in range(model.cfg.num_codebooks):\n            oracle_histogram[book] += np.bincount(\n                target_np[:, book], minlength=model.cfg.codec_vocab\n            )\n        rollout.update(\n            rollout_logits.unsqueeze(0), target[: rollout_logits.shape[0]].unsqueeze(0),\n            torch.ones(1, rollout_logits.shape[0], dtype=torch.bool, device=device),\n        )\n        rollout_sequences.append(\n            {\"record_id\": item[\"record_id\"], **sequence_accuracy(generated, target)}\n        )\n        true_duration = batch[\"durations\"][0, batch[\"token_mask\"][0]].float()\n        difference = (predicted_duration.float() - true_duration).abs()\n        duration_abs.extend(difference.cpu().tolist())\n        duration_ape.extend((difference / true_duration.clamp(min=1)).cpu().tolist())\n    teacher_rows, rollout_rows = teacher.report(), rollout.report()\n    oracle_usage = [int(np.count_nonzero(row)) for row in oracle_histogram]\n    oracle_entropy = []\n    for row in oracle_histogram:\n        probabilities = row / max(1, int(row.sum()))\n        nonzero = probabilities[probabilities > 0]\n        oracle_entropy.append(float(-(nonzero * np.log(nonzero)).sum()))\n    return {\n        \"text_mode\": text_mode,\n        \"samples\": count,\n        \"teacher_forced\": teacher_rows,\n        \"rollout\": rollout_rows,\n        \"rollout_sequences\": rollout_sequences,\n        \"rollout_cb0_ce\": rollout_rows[0][\"cross_entropy\"],\n        \"duration_mae_frames\": float(np.mean(duration_abs)) if duration_abs else None,\n        \"duration_median_ape\": float(np.median(duration_ape)) if duration_ape else None,\n        \"oracle_codes_used\": oracle_usage,\n        \"oracle_entropy\": oracle_entropy,\n    }\n\n\n@torch.no_grad()\ndef write_epoch_validation_audio(\n    model: StreamingSpeechAdapter,\n    dataset: SpeechDataset,\n    device: torch.device,\n    cfg: AppConfig,\n    run_dir: Path,\n    epoch: int,\n) -> dict[str, Any]:\n    \"\"\"Decode fixed validation examples after an epoch using the exact 8-CB codec.\"\"\"\n    from .diagnostics import OnnxMimi, _write_wav\n\n    model.eval()\n    decoder = OnnxMimi(\n        cfg.paths.mimi_encoder,\n        cfg.paths.mimi_decoder,\n        num_codebooks=model.cfg.num_codebooks,\n    )\n    output_dir = run_dir / \"epoch_validation\" / f\"epoch_{epoch:04d}\"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    count = min(len(dataset), cfg.train.eval_audio_samples)\n    samples: list[dict[str, Any]] = []\n    generation_warnings: list[str] = []\n    for index in range(count):\n        item = dataset[index]\n        batch = move_batch(collate_speech([item]), device)\n        logits, _ = model(batch)\n        frames = int(batch[\"frame_mask\"][0].sum().item())\n        oracle = batch[\"codes\"][0, :frames]\n        teacher = logits[0, :frames].argmax(dim=-1)\n        hidden = batch[\"hidden\"][0, batch[\"token_mask\"][0]]\n        embeddings = batch[\"embeddings\"][0, batch[\"token_mask\"][0]]\n        oracle_duration = batch[\"durations\"][0, batch[\"token_mask\"][0]]\n        autoregressive, predicted_duration, _ = model.rollout_with_logits(\n            hidden, embeddings, oracle_duration\n        )\n        fully_autoregressive, _, _ = model.rollout_with_logits(hidden, embeddings, None)\n\n        safe_id = re.sub(r\"[^A-Za-z0-9_.-]+\", \"_\", str(item[\"record_id\"]))[:80]\n        sample_dir = output_dir / f\"sample_{index:03d}_{safe_id or 'record'}\"\n        modes = {\n            \"oracle\": oracle,\n            \"teacher_forced\": teacher,\n            \"autoregressive_oracle_duration\": autoregressive,\n            \"autoregressive_predicted_duration\": fully_autoregressive,\n        }\n        files: dict[str, str] = {}\n        for name, codes in modes.items():\n            if codes.ndim != 2 or codes.shape[1] != model.cfg.num_codebooks:\n                raise RuntimeError(\n                    f\"epoch {epoch} {name} produced invalid Mimi shape {tuple(codes.shape)}\"\n                )\n            if codes.shape[0] == 0:\n                warning = (\n                    f\"{item['record_id']} {name}: duration head generated zero audio frames\"\n                )\n                generation_warnings.append(warning)\n                files[name] = \"\"\n                continue\n            path = sample_dir / f\"{name}.wav\"\n            _write_wav(path, decoder.decode(codes.detach().cpu().numpy()))\n            files[name] = str(path.relative_to(run_dir))\n        samples.append(\n            {\n                \"record_id\": item[\"record_id\"],\n                \"text\": item[\"text\"],\n                \"target_frames\": frames,\n                \"predicted_duration_frames\": int(predicted_duration.sum().item()),\n                \"teacher_forced_accuracy\": sequence_accuracy(teacher, oracle),\n                \"autoregressive_accuracy\": sequence_accuracy(autoregressive, oracle),\n                \"files\": files,\n            }\n        )\n    report = {\n        \"automatic_pass\": len(samples) == count and count > 0 and not generation_warnings,\n        \"epoch\": epoch,\n        \"samples\": samples,\n        \"num_codebooks\": model.cfg.num_codebooks,\n        \"decoder\": cfg.paths.mimi_decoder,\n        \"generation_warnings\": generation_warnings,\n        \"teacher_forced_definition\": \"parallel forward pass conditioned on oracle code history\",\n        \"autoregressive_definition\": \"generated code history; oracle token durations\",\n        \"fully_autoregressive_definition\": \"generated code history and predicted token durations\",\n    }\n    atomic_json(output_dir / \"report.json\", report)\n    atomic_json(run_dir / \"latest_epoch_audio.json\", report)\n    return report\n\n\ndef require_qualification(cfg: AppConfig) -> None:\n    path = cfg.work_dir / \"gates\" / \"gate_report.json\"\n    if not path.is_file():\n        raise RuntimeError(\"full training requires gates/gate_report.json; run qualification first\")\n    report = json.loads(path.read_text())\n    if report.get(\"feature_fingerprint\") != cfg.feature_fingerprint():\n        raise RuntimeError(\"qualification report belongs to different features\")\n    for gate in \"ABCDEF\":\n        if report.get(\"gates\", {}).get(gate, {}).get(\"status\") != \"passed\":\n            raise RuntimeError(f\"Gate {gate} has not passed\")\n    context_path = cfg.work_dir / \"gates\" / \"context_selection.json\"\n    if not context_path.is_file():\n        raise RuntimeError(\"full training requires gates/context_selection.json\")\n    selection = json.loads(context_path.read_text())\n    if int(selection[\"selected_context_size\"]) != cfg.model.text_context_size:\n        raise RuntimeError(\n            \"configured text_context_size does not match the qualified context selection\"\n        )\n    device_path = cfg.work_dir / \"gates\" / \"device_report.json\"\n    if not device_path.is_file() or not json.loads(device_path.read_text()).get(\"automatic_pass\"):\n        raise RuntimeError(\"full training requires a passing imported Android device report\")\n    export_path = cfg.work_dir / \"gates\" / \"edge_export\" / \"export_report.json\"\n    if not export_path.is_file():\n        raise RuntimeError(\"full training requires the qualified incremental ONNX export\")\n    export_report = json.loads(export_path.read_text())\n    if not export_report.get(\"pytorch_onnx_pass\") or not export_report.get(\"int8_pass\"):\n        raise RuntimeError(\"qualified ONNX or INT8 parity did not pass\")\n\n\ndef run_training(\n    cfg: AppConfig,\n    limit: int = 0,\n    epochs: int | None = None,\n    experiment: str = \"full\",\n    init_weights: str = \"\",\n) -> dict[str, Any]:\n    rank, world, local = distributed(cfg)\n    if experiment == \"full\" and cfg.run_stage == \"train_full\":\n        require_qualification(cfg)\n    run_dir = cfg.work_dir if experiment == \"full\" else cfg.work_dir / \"experiments\" / experiment\n    run_dir.mkdir(parents=True, exist_ok=True)\n    seed_everything(cfg.train.seed + rank)\n    use_cuda = cfg.runtime.device == \"cuda\" or (\n        cfg.runtime.device == \"auto\" and torch.cuda.is_available()\n    )\n    if cfg.runtime.device == \"cuda\" and not torch.cuda.is_available():\n        raise RuntimeError(\"runtime.device='cuda' but CUDA is not available\")\n    device = torch.device(f\"cuda:{local}\" if use_cuda else \"cpu\")\n    if device.type == \"cuda\":\n        torch.cuda.set_device(device)\n        torch.backends.cuda.matmul.allow_tf32 = True\n    else:\n        torch.set_num_threads(cfg.runtime.cpu_threads)\n    train_data = SpeechDataset(cfg, \"train\", limit=limit)\n    val_limit = max(2, min(cfg.train.eval_samples, limit // 10 if limit else cfg.train.eval_samples))\n    val_data = SpeechDataset(cfg, \"val\", limit=val_limit)\n    model = build_model(cfg, device)\n    if init_weights and not (run_dir / \"checkpoints\" / \"last.safetensors\").is_file():\n        model.load_state_dict(load_file(init_weights, device=str(device)))\n        if rank == 0:\n            print(f\"initialized {experiment} from {init_weights}\", flush=True)\n    safe_budget = calibrate_budget(model, train_data, cfg, device)\n    sampler = DistributedFrameBatchSampler(\n        train_data.lengths,\n        safe_budget,\n        cfg.train.max_batch_size,\n        rank,\n        world,\n        cfg.train.seed,\n        True,\n    )\n    loader = DataLoader(\n        train_data,\n        batch_sampler=sampler,\n        collate_fn=collate_speech,\n        num_workers=cfg.train.num_workers,\n        pin_memory=device.type == \"cuda\",\n        persistent_workers=cfg.train.num_workers > 0,\n    )\n    if world > 1:\n        model = DistributedDataParallel(model, device_ids=[local], broadcast_buffers=False)\n    base_model = model.module if isinstance(model, DistributedDataParallel) else model\n    qualification = experiment != \"full\"\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=cfg.train.learning_rate,\n        weight_decay=0.0 if qualification else cfg.train.weight_decay,\n    )\n    run_epochs = int(epochs or cfg.train.epochs)\n    updates_per_epoch = max(1, math.ceil(len(loader) / cfg.train.grad_accum_steps))\n    total_updates = max(1, updates_per_epoch * run_epochs)\n    warmup = max(1, int(total_updates * cfg.train.warmup_fraction))\n\n    def lr_factor(step: int) -> float:\n        if step < warmup:\n            return max(1.0e-3, step / warmup)\n        progress = (step - warmup) / max(1, total_updates - warmup)\n        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))\n        return cfg.train.min_lr_ratio + (1.0 - cfg.train.min_lr_ratio) * cosine\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=device.type == \"cuda\")\n    start_epoch, best = maybe_resume(\n        base_model, optimizer, scheduler, scaler, cfg, device, run_dir\n    )\n    training_path = run_dir / \"training.json\"\n    if start_epoch and training_path.is_file():\n        history = json.loads(training_path.read_text()).get(\"history\", [])\n    else:\n        history = []\n    validation: dict[str, Any] | None = None\n    log_path = run_dir / \"train_steps.jsonl\"\n    for epoch in range(start_epoch, run_epochs):\n        sampler.set_epoch(epoch)\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        started = time.perf_counter()\n        seen_frames = 0\n        running = 0.0\n        running_codebook_loss = np.zeros(cfg.model.num_codebooks, dtype=np.float64)\n        running_codebook_frames = 0\n        for step, raw_batch in enumerate(loader, 1):\n            batch = move_batch(raw_batch, device)\n            final_accum = step % cfg.train.grad_accum_steps == 0 or step == len(loader)\n            sync = contextlib.nullcontext()\n            if isinstance(model, DistributedDataParallel) and not final_accum:\n                sync = model.no_sync()\n            with sync, torch.autocast(\n                device_type=\"cuda\", dtype=torch.float16, enabled=device.type == \"cuda\"\n            ):\n                logits, duration = model(batch)\n                loss, parts = adapter_loss(\n                    logits,\n                    duration,\n                    batch,\n                    cfg.train.cb0_weight,\n                    cfg.train.residual_weight,\n                    cfg.train.duration_weight,\n                )\n                loss = loss / cfg.train.grad_accum_steps\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f\"non-finite loss at epoch={epoch + 1} step={step}\")\n            scaler.scale(loss).backward()\n            running += float(loss.item()) * cfg.train.grad_accum_steps\n            batch_frames = int(batch[\"frame_mask\"].sum().item())\n            seen_frames += batch_frames\n            running_codebook_loss += (\n                parts[\"per_book\"].detach().float().cpu().numpy() * batch_frames\n            )\n            running_codebook_frames += batch_frames\n            if final_accum:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                scheduler.step()\n            if rank == 0 and (step == 1 or step % cfg.train.log_every == 0 or step == len(loader)):\n                elapsed = max(time.perf_counter() - started, 1e-6)\n                message = {\n                    \"epoch\": epoch + 1,\n                    \"step\": step,\n                    \"steps\": len(loader),\n                    \"loss\": running / step,\n                    \"codec_loss\": float(parts[\"codec\"].item()),\n                    \"batch_loss_per_codebook\": [\n                        float(value) for value in parts[\"per_book\"].cpu()\n                    ],\n                    \"batch_cb0_loss\": float(parts[\"per_book\"][0].item()),\n                    \"duration_loss\": float(parts[\"duration\"].item()),\n                    \"learning_rate\": scheduler.get_last_lr()[0],\n                    \"frames_per_second_rank0\": seen_frames / elapsed,\n                    \"peak_vram_gib\": torch.cuda.max_memory_allocated(device) / 2**30 if device.type == \"cuda\" else 0,\n                }\n                print(json.dumps(message), flush=True)\n                with log_path.open(\"a\", encoding=\"utf-8\") as handle:\n                    handle.write(json.dumps(message) + \"\\n\")\n        epoch_codebook_sums = torch.tensor(\n            running_codebook_loss, device=device, dtype=torch.float64\n        )\n        epoch_codebook_frames = torch.tensor(\n            running_codebook_frames, device=device, dtype=torch.float64\n        )\n        if dist.is_initialized():\n            dist.all_reduce(epoch_codebook_sums, op=dist.ReduceOp.SUM)\n            dist.all_reduce(epoch_codebook_frames, op=dist.ReduceOp.SUM)\n            dist.barrier()\n        train_codebook_loss = (\n            epoch_codebook_sums / epoch_codebook_frames.clamp(min=1)\n        ).cpu().tolist()\n        validation = None\n        if rank == 0:\n            score_data = train_data if experiment.startswith((\"C_\", \"D_\")) else val_data\n            validation = evaluate(base_model, score_data, device, cfg.train.eval_samples)\n            audio_report = write_epoch_validation_audio(\n                base_model, score_data, device, cfg, run_dir, epoch + 1\n            )\n            score = float(validation[\"rollout_cb0_ce\"])\n            previous_validation = history[-1].get(\"validation\") if history else None\n            signals = training_health_signals(\n                validation,\n                previous_validation=previous_validation,\n                vocabulary=cfg.model.codec_vocab,\n            )\n            atomic_json(\n                run_dir / \"epoch_validation\" / f\"epoch_{epoch + 1:04d}\" / \"metrics.json\",\n                {\n                    \"epoch\": epoch + 1,\n                    \"validation\": validation,\n                    \"signals\": signals,\n                    \"audio\": audio_report,\n                },\n            )\n            record = {\n                \"epoch\": epoch + 1,\n                \"train_loss\": running / max(1, len(loader)),\n                \"train_loss_per_codebook\": train_codebook_loss,\n                \"train_cb0_loss\": train_codebook_loss[0],\n                \"validation\": validation,\n                \"validation_audio\": {\n                    \"automatic_pass\": audio_report[\"automatic_pass\"],\n                    \"samples\": len(audio_report[\"samples\"]),\n                    \"report\": str(\n                        Path(\"epoch_validation\") / f\"epoch_{epoch + 1:04d}\" / \"report.json\"\n                    ),\n                },\n                \"signals\": signals,\n                \"safe_frames_per_gpu\": safe_budget,\n            }\n            history.append(record)\n            compact_books = [\n                {\n                    \"cb\": row[\"codebook\"],\n                    \"signal\": row[\"signal\"],\n                    \"tf_ce\": round(row[\"teacher_ce\"], 4),\n                    \"tf_top1\": round(row[\"teacher_top1\"], 4),\n                    \"ar_ce\": round(row[\"ar_ce\"], 4),\n                    \"ar_top1\": round(row[\"ar_top1\"], 4),\n                    \"gap\": round(row[\"ar_teacher_ce_gap\"], 4),\n                    \"ar_trend\": row[\"ar_trend\"],\n                    \"usage_ratio\": round(row[\"code_usage_ratio\"], 3),\n                }\n                for row in signals[\"codebooks\"]\n            ]\n            print(\n                json.dumps(\n                    {\n                        \"epoch_health\": epoch + 1,\n                        \"signal\": signals[\"overall_signal\"],\n                        \"headline\": signals[\"headline\"],\n                        \"warnings\": signals[\"warnings\"],\n                        \"codebooks\": compact_books,\n                        \"duration_median_ape\": signals[\"duration_median_ape\"],\n                        \"validation_audio_pass\": audio_report[\"automatic_pass\"],\n                        \"validation_audio_warnings\": audio_report[\"generation_warnings\"],\n                        \"validation_audio_report\": str(\n                            Path(\"epoch_validation\") / f\"epoch_{epoch + 1:04d}\" / \"report.json\"\n                        ),\n                    }\n                ),\n                flush=True,\n            )\n            atomic_json(run_dir / \"latest_training_signal.json\", signals)\n            write_epoch_signal_csv(run_dir / \"epoch_codebook_signals.csv\", history)\n            save_checkpoint(\n                base_model, optimizer, scheduler, scaler, epoch + 1,\n                min(best, score), cfg, \"last\", run_dir,\n            )\n            if score < best:\n                best = score\n                save_checkpoint(\n                    base_model, optimizer, scheduler, scaler, epoch + 1,\n                    best, cfg, \"best\", run_dir,\n                )\n            atomic_json(\n                run_dir / \"training.json\",\n                {\n                    \"complete\": epoch + 1 == run_epochs,\n                    \"config_fingerprint\": cfg.fingerprint(),\n                    \"feature_fingerprint\": cfg.feature_fingerprint(),\n                    \"world_size\": world,\n                    \"parameters\": base_model.parameter_count(),\n                    \"best_rollout_cb0_ce\": best,\n                    \"history\": history,\n                },\n            )\n        if dist.is_initialized():\n            dist.barrier()\n    if rank == 0 and experiment != \"full\":\n        best_path = run_dir / \"checkpoints\" / \"best.safetensors\"\n        if best_path.is_file():\n            base_model.load_state_dict(load_file(str(best_path), device=str(device)))\n            validation = None\n        if validation is None:\n            score_data = train_data if experiment.startswith((\"C_\", \"D_\")) else val_data\n            validation = evaluate(base_model, score_data, device, cfg.train.eval_samples)\n        controls = None\n        if experiment.startswith((\"D_\", \"E_\")):\n            score_data = train_data if experiment.startswith(\"D_\") else val_data\n            controls = {\n                mode: evaluate(base_model, score_data, device, cfg.train.eval_samples, text_mode=mode)\n                for mode in (\"zeroed\", \"swapped\")\n            }\n        if experiment.startswith(\"smoke_\"):\n            atomic_json(\n                run_dir / \"smoke_metrics.json\",\n                {\"automatic_pass\": True, \"experiment\": experiment, \"metrics\": validation},\n            )\n        else:\n            _write_experiment_gate(cfg, experiment, validation, run_dir, controls)\n    if dist.is_initialized():\n        dist.destroy_process_group()\n    return {\"best_rollout_cb0_ce\": best, \"epochs\": run_epochs} if rank == 0 else {}\n\n\ndef _write_experiment_gate(\n    cfg: AppConfig,\n    experiment: str,\n    result: dict[str, Any],\n    run_dir: Path,\n    controls: dict[str, dict[str, Any]] | None,\n) -> None:\n    from .diagnostics import update_gate\n\n    gate = experiment[:1].upper()\n    automatic = False\n    details: dict[str, Any] = {\n        \"experiment\": experiment, \"metrics\": result, \"controls\": controls\n    }\n    if gate == \"C\":\n        teacher_ce = float(result[\"teacher_forced\"][0][\"cross_entropy\"])\n        accuracies = [\n            float(row[\"per_codebook_accuracy\"][0])\n            for row in result[\"rollout_sequences\"]\n        ]\n        rollout_accuracy = float(np.mean(accuracies)) if accuracies else 0.0\n        details.update({\"teacher_cb0_ce\": teacher_ce, \"rollout_cb0_accuracy\": rollout_accuracy})\n        automatic = (\n            teacher_ce <= cfg.gates.overfit_teacher_forced_ce\n            and rollout_accuracy >= cfg.gates.overfit_rollout_cb0_accuracy\n        )\n    elif gate == \"D\":\n        correct = float(result[\"rollout_cb0_ce\"])\n        control_pass = bool(controls) and all(\n            correct <= float(controls[mode][\"rollout_cb0_ce\"]) * (1.0 - cfg.gates.text_control_margin)\n            for mode in (\"zeroed\", \"swapped\")\n        )\n        training_record = json.loads((run_dir / \"training.json\").read_text())\n        history = training_record.get(\"history\", [])\n        gaps = []\n        for item in history:\n            val = item[\"validation\"]\n            gaps.append(\n                float(val[\"rollout\"][0][\"cross_entropy\"])\n                - float(val[\"teacher_forced\"][0][\"cross_entropy\"])\n            )\n        shrinking = len(gaps) < 2 or gaps[-1] <= gaps[0]\n        automatic = result[\"samples\"] >= 32 and control_pass and shrinking and all(\n            row[\"per_codebook_accuracy\"][0] >= 0.95\n            for row in result[\"rollout_sequences\"]\n        )\n    elif gate == \"E\":\n        cb0 = result[\"rollout\"][0]\n        correct = float(result[\"rollout_cb0_ce\"])\n        control_pass = bool(controls) and all(\n            correct <= float(controls[mode][\"rollout_cb0_ce\"]) * (1.0 - cfg.gates.text_control_margin)\n            for mode in (\"zeroed\", \"swapped\")\n        )\n        usage_ratio = cb0[\"codes_used\"] / max(1, result[\"oracle_codes_used\"][0])\n        entropy_ratio = cb0[\"predicted_entropy\"] / max(1e-9, result[\"oracle_entropy\"][0])\n        automatic = (\n            result[\"samples\"] >= min(20, cfg.train.eval_samples)\n            and control_pass\n            and usage_ratio >= cfg.gates.min_code_usage_ratio\n            and entropy_ratio >= cfg.gates.min_entropy_ratio\n        )\n    elif gate == \"F\":\n        automatic = (\n            result.get(\"duration_median_ape\") is not None\n            and result[\"duration_median_ape\"] <= cfg.gates.max_duration_median_ape\n        )\n    atomic_json(run_dir / \"gate_metrics.json\", {\"automatic_pass\": automatic, **details})\n    update_gate(cfg, gate, automatic, None, note=f\"automatic results: {run_dir / 'gate_metrics.json'}\")\n"}
if 'def run_smoke(cfg' not in SOURCES.get("smoke.py", ""):
    raise RuntimeError(
        "Stale speech_adapter source bundle: smoke.py does not support CPU/CUDA selection. "
        "Upload the newly generated production notebook and rerun this source-install cell."
    )
if 'args.command in {"smoke", "local-smoke"}' not in SOURCES.get("cli.py", ""):
    raise RuntimeError(
        "Stale speech_adapter source bundle: cli.py is missing the device-aware smoke command. "
        "Upload the newly generated production notebook and rerun this source-install cell."
    )
PACKAGE_DIR = Path("/kaggle/working/Streaming_TTS/speech_adapter")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in SOURCES.items():
    compile(source, name, "exec")
    target = PACKAGE_DIR / name
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(source, encoding="utf-8")
    temporary.replace(target)
print(f"Wrote and compiled {len(SOURCES)} modules under {PACKAGE_DIR}")

Wrote and compiled 12 modules under /kaggle/working/Streaming_TTS/speech_adapter


In [6]:
import os
import subprocess
import sys

WORKING = "/kaggle/working/Streaming_TTS"
if WORKING not in sys.path:
    sys.path.insert(0, WORKING)
env = dict(os.environ)
env["PYTHONPATH"] = WORKING + os.pathsep + env.get("PYTHONPATH", "")

def run(command, *, check=True):
    print("+", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(list(map(str, command)), env=env, text=True)
    if check and completed.returncode:
        raise RuntimeError(f"command failed with exit code {completed.returncode}: {command}")
    return completed

run([sys.executable, "-m", "speech_adapter", "--config", CONFIG_PATH, "--help"])
run([sys.executable, "-m", "speech_adapter", "--config", CONFIG_PATH, "doctor"])
run([sys.executable, "-m", "speech_adapter", "--config", CONFIG_PATH, "structural-checks"])
print("Import and CLI smoke tests passed.")

+ /usr/bin/python3 -m speech_adapter --config /kaggle/working/streaming_speech/config.json --help
usage: speech-adapter [-h] --config CONFIG
                      {doctor,prepare-manifest,prepare-features,structural-checks,smoke,local-smoke,codec-contract,gate-a,gate-b,approve-gate,train,evaluate,export,compare-context,import-device-report,status,package}
                      ...

Local/Kaggle runner for the causal Qwen-to-Mimi speech adapter

positional arguments:
  {doctor,prepare-manifest,prepare-features,structural-checks,smoke,local-smoke,codec-contract,gate-a,gate-b,approve-gate,train,evaluate,export,compare-context,import-device-report,status,package}
    doctor              validate configured hardware and data/model inputs
    prepare-manifest    validate shards and build manifest
    prepare-features    prepare indexed or recomputed Qwen features
    structural-checks   run causal and CB0-gradient tests
    smoke               run one bounded CPU/CUDA forward/backward/cache 

# Unit Testing

In [7]:
!chmod +x /kaggle/working/Streaming_TTS/scripts/*

In [8]:
%%writefile /kaggle/working/Streaming_TTS/scripts/requirements.txt
numpy>=1.26,<3
torch>=2.5,<3
transformers>=4.56,<6
safetensors>=0.4.5,<1
accelerate>=1.2,<2
onnx>=1.17,<2
onnxruntime>=1.20,<2
soundfile>=0.13,<1
kokoro-onnx==0.6.1
moshi==0.2.13

Writing /kaggle/working/Streaming_TTS/scripts/requirements.txt


In [9]:
!pip install -r /kaggle/working/Streaming_TTS/scripts/requirements.txt

In [10]:
!bash /kaggle/working/Streaming_TTS/scripts/run_local_smoke.sh /kaggle/working/streaming_speech/config.json

[1/7] Preflight
{
  "profile": "kaggle",
  "selected_device": "auto",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "torch": "2.9.1+cu128",
  "cuda": "12.8",
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "vram_gib": 14.56219482421875
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "vram_gib": 14.56219482421875
    }
  ],
  "shards": 345,
  "manifest_record_limit": 0,
  "cpu_threads_configured": 2,
  "logical_cpu_count": 4,
  "physical_ram_gib": 31.346752166748047,
  "free_working_gib": 19.501590728759766,
  "config_fingerprint": "2065a10cb4424ba9d474c894911313749204b926a33790deafac3c4030c322a3"
}
[2/7] Mimi eight-codebook contract
{
  "automatic_pass": true,
  "expected_codebooks": 8,
  "encoder_output_shape": [
    "Pad_970_o0__d0",
    8,
    "floor(Pad_970_o0__d2/2) - 1"
  ],
  "encoder_output_codebooks": 8,
  "encoder_contract": "first 8 sequential residual codebooks are used",
  "decoder_input_shape": [
    "batch_size",
 

In [11]:
from pathlib import Path
import json
import os

# ---------------------------------------------------------------------------
# EDIT ONLY THIS CELL BEFORE RUNNING THE NOTEBOOK.
# ---------------------------------------------------------------------------
RUN_STAGE = "qualify"  # preflight | prepare | qualify | train_full | evaluate | export
CONFIRM_FULL_TRAINING = ""  # set exactly to TRAIN_FULL_DATASET only for train_full
# First prove the adapter with the answer-only hidden states already stored in
# the encoded shards. Switch to "deployment_context" only in a later ablation.
FEATURE_MODE = "legacy_answer"  # legacy_answer | answer_only_recompute | deployment_context


INPUT_ROOT = Path("/kaggle/input/datasets/raja25/mimi-codes/dataset_question_answer_codes")
WORK_DIR = Path("/kaggle/working/streaming_speech")

def first_match(patterns, *, directory=False, required=True):
    for pattern in patterns:
        matches = sorted(
            path.resolve() for path in INPUT_ROOT.glob(pattern)
            if path.is_dir() if directory
        ) if directory else sorted(
            path.resolve() for path in INPUT_ROOT.glob(pattern) if path.is_file()
        )
        if matches:
            return str(matches[0])
    if required:
        raise FileNotFoundError(
            "Could not auto-discover a required Kaggle input. Attach the dataset, "
            f"then edit the matching path below. Patterns: {patterns}"
        )
    return ""

def discover_complete_data_dir():
    candidates = []
    for shard in INPUT_ROOT.glob("**/shard_*"):
        if shard.is_dir() and all((shard / name).is_file() for name in (
            "codes.npy", "offsets.npy", "hidden.npy", "hidden_offsets.npy",
            "token_ids.npy", "token_char_offsets.npy", "ids.jsonl",
            "alignment.jsonl", "pieces.jsonl",
        )):
            candidates.append(shard.parent.resolve())
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            "No complete encoded shard dataset found below /kaggle/input. "
            "Codes-only shards are not sufficient."
        )
    return str(candidates[0])

# Auto-discovery is intentionally deterministic. If more than one candidate exists,
# replace a value here with the exact /kaggle/input/... path shown by Kaggle.
DATA_DIR = discover_complete_data_dir()
SOURCE_JSONL = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/dataset_question_answer_pair.jsonl"
QWEN_MODEL = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Qwen2.5-1.5B-Instruct"
KOKORO_MODEL = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/onnx/model.onnx"
KOKORO_VOICES = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/voices/af_jessica.bin"
MIMI_ENCODER = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/encoder_model_8cb.onnx"
MIMI_DECODER = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/decoder_model_8cb.onnx"
MIMI_STREAMING_WEIGHTS = "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/moshiko_pt_model/moshiko-pytorch-bf16/tokenizer-e351c8d8-checkpoint125.safetensors"


CONFIG = {
    "run_stage": RUN_STAGE,
    "confirm_full_training": CONFIRM_FULL_TRAINING,
    "feature_mode": FEATURE_MODE,
    "runtime": {
        "profile": "kaggle",
        "device": "auto",
        "require_two_gpus": True,
        "manifest_record_limit": 0,
        "cpu_threads": 2,
    },
    "paths": {
        "data_dir": DATA_DIR,
        "source_jsonl": SOURCE_JSONL,
        "qwen_model": QWEN_MODEL,
        "kokoro_model": KOKORO_MODEL,
        "kokoro_voices": KOKORO_VOICES,
        "mimi_encoder": MIMI_ENCODER,
        "mimi_decoder": MIMI_DECODER,
        "mimi_weights": MIMI_STREAMING_WEIGHTS,
        "work_dir": str(WORK_DIR),
    },
    "model": {
        "model_dim": 256,
        "temporal_layers": 2,
        "attention_heads": 4,
        "ffn_dim": 512,
        "text_context_size": 8,
        "audio_cache_frames": 64,
        "num_codebooks": 8,
        "codec_vocab": 2048,
        "dropout": 0.0,
    },
    "train": {
        "seed": 42,
        "epochs": 20,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "warmup_fraction": 0.05,
        "min_lr_ratio": 0.05,
        "grad_clip": 1.0,
        "duration_weight": 0.1,
        "cb0_weight": 4.0,
        "residual_weight": 1.0,
        "max_frames_per_gpu": 2500,
        "max_batch_size": 8,
        "grad_accum_steps": 1,
        "num_workers": 2,
        "val_fraction": 0.1,
        "eval_samples": 64,
        "eval_audio_samples": 2,
        "save_every": 1,
        "log_every": 25,
        "resume": True,
    },
    "gates": {
        "codec_samples": 20,
        "alignment_samples": 50,
        "alignment_pass_fraction": 0.95,
        "overfit_teacher_forced_ce": 0.05,
        "overfit_rollout_cb0_accuracy": 0.99,
        "text_control_margin": 0.05,
        "min_code_usage_ratio": 0.50,
        "min_entropy_ratio": 0.75,
        "max_duration_median_ape": 0.20,
        "max_device_p50_ms": 20.0,
        "max_device_p95_ms": 40.0,
        "max_device_worst_ms": 80.0,
        "max_first_audio_ms": 300.0,
        "max_int8_weight_bytes": 12582912,
        "max_working_memory_bytes": 33554432,
        "max_cache_bytes": 1048576,
    },
}

WORK_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK_DIR / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(json.dumps(CONFIG, indent=2))

{
  "run_stage": "qualify",
  "confirm_full_training": "",
  "feature_mode": "legacy_answer",
  "runtime": {
    "profile": "kaggle",
    "device": "auto",
    "require_two_gpus": true,
    "manifest_record_limit": 0,
    "cpu_threads": 2
  },
  "paths": {
    "data_dir": "/kaggle/input/datasets/raja25/mimi-codes/dataset_question_answer_codes",
    "source_jsonl": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/dataset_question_answer_pair.jsonl",
    "qwen_model": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Qwen2.5-1.5B-Instruct",
    "kokoro_model": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/onnx/model.onnx",
    "kokoro_voices": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/Kokoro-82M-v1.0-ONNX/voices/af_jessica.bin",
    "mimi_encoder": "/kaggle/input/datasets/raja25/lllama-qwen-qa-generation-dataset/encoder_model_8cb.onnx",
    "mimi_decoder": "/kaggle/input/datasets/raja25/lllama-q

In [ ]:
# This cell is restart-safe. Completed manifest/features/checkpoints are verified and reused.
BASE = [sys.executable, "-m", "speech_adapter", "--config", str(CONFIG_PATH)]

def cli(*arguments):
    return run([*BASE, *arguments])

def torchrun(*arguments):
    return run([
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        "--nproc_per_node=2", "-m", "speech_adapter", "--config", str(CONFIG_PATH),
        *arguments,
    ])

def ensure_prepared():
    cli("prepare-manifest")
    if FEATURE_MODE == "legacy_answer":
        # The shard states already exist. One CPU-side index pass avoids NCCL,
        # duplicate workers, and full Qwen model loading.
        cli("prepare-features")
    else:
        torchrun("prepare-features")
    cli("structural-checks")
    # Keep the established command spelling for compatibility. In the current
    # package it is an alias of the device-aware CPU/CUDA run_smoke function.
    cli("local-smoke")

if RUN_STAGE == "preflight":
    print("Preflight passed. Change RUN_STAGE to 'prepare' when ready.")
elif RUN_STAGE == "prepare":
    ensure_prepared()
    print("Preparation passed. Change RUN_STAGE to 'qualify' for Gates A-F.")
elif RUN_STAGE == "qualify":
    ensure_prepared()
    cli("codec-contract")
    cli("gate-a")
    cli("gate-b")
    # Qualification presets deliberately favor proof over regularization.
    torchrun("train", "--experiment", "C_one", "--limit", "1", "--epochs", "300", "--context-size", "3")
    torchrun("train", "--experiment", "D_32", "--limit", "32", "--epochs", "100", "--context-size", "3")
    torchrun("train", "--experiment", "E_k3", "--limit", "2048", "--epochs", "20", "--context-size", "3")
    torchrun("train", "--experiment", "E_k8", "--limit", "2048", "--epochs", "20", "--context-size", "8")
    cli("compare-context", "--small", "E_k3", "--large", "E_k8", "--small-k", "3", "--large-k", "8")
    selection = json.loads((WORK_DIR / "gates" / "context_selection.json").read_text())
    if selection["test_larger_context"]:
        torchrun("train", "--experiment", "E_k16", "--limit", "2048", "--epochs", "20", "--context-size", "16")
        cli("compare-context", "--small", "E_k8", "--large", "E_k16", "--small-k", "8", "--large-k", "16")
        selection = json.loads((WORK_DIR / "gates" / "context_selection.json").read_text())
        if selection["test_larger_context"]:
            torchrun("train", "--experiment", "E_k32", "--limit", "2048", "--epochs", "20", "--context-size", "32")
            cli("compare-context", "--small", "E_k16", "--large", "E_k32", "--small-k", "16", "--large-k", "32")
            selection = json.loads((WORK_DIR / "gates" / "context_selection.json").read_text())
    selected_k = int(selection["selected_context_size"])
    source_experiment = f"E_k{selected_k}"
    source_weights = WORK_DIR / "experiments" / source_experiment / "checkpoints" / "best.safetensors"
    edge_export = WORK_DIR / "gates" / "edge_export"
    cli(
        "export", "--parity-steps", "100", "--context-size", str(selected_k),
        "--checkpoint", str(source_weights), "--output-dir", str(edge_export),
    )
    torchrun(
        "train", "--experiment", "F_duration", "--limit", "2048", "--epochs", "5",
        "--context-size", str(selected_k), "--init-weights", str(source_weights),
    )
    cli("status")
    print("Automatic qualification finished. Review audio/alignment artifacts and use the next cell.")
elif RUN_STAGE == "train_full":
    selection_path = WORK_DIR / "gates" / "context_selection.json"
    if not selection_path.is_file():
        raise RuntimeError("Run qualification before full training")
    selection = json.loads(selection_path.read_text())
    CONFIG["model"]["text_context_size"] = int(selection["selected_context_size"])
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
    torchrun("train", "--experiment", "full")
elif RUN_STAGE == "evaluate":
    cli("evaluate", "--limit", "20")
    cli("package")
elif RUN_STAGE == "export":
    cli("export", "--parity-steps", "100")
    cli("package")
else:
    raise ValueError(RUN_STAGE)

+ /usr/bin/python3 -m speech_adapter --config /kaggle/working/streaming_speech/config.json prepare-manifest
manifest already valid: /kaggle/working/streaming_speech/manifest.jsonl
{
  "audio_hours": 207.03524444444446,
  "available_shards": 345,
  "complete": true,
  "dataset_signature": "961b09f643a45d0593cbcc4531842edf95e5ac57bcc5229ac52230fbcd988f3e",
  "estimated_feature_gib": 6.123315341770649,
  "frames": 9316586,
  "free_working_gib": 19.266586303710938,
  "hidden_dim": 1536,
  "legacy_tokens": 2131926,
  "manifest": "/kaggle/working/streaming_speech/manifest.jsonl",
  "quarantine": "/kaggle/working/streaming_speech/quarantine.jsonl",
  "quarantined": 0,
  "recommended_working_gib": 9.654144177213311,
  "record_limit": 0,
  "records": 88320,
  "shards": 345,
  "split_counts": {
    "train": 79396,
    "val": 8924
  }
}
+ /usr/bin/python3 -m speech_adapter --config /kaggle/working/streaming_speech/config.json prepare-features
manifest already valid: /kaggle/working/streaming_spee


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W1004 09:11:59.830748510 socket.cpp:209] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1004 09:12:01.933881733 socket.cpp:209] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1004 09:12:01.945577549 socket.cpp:209] [c10d] The hostname of the client socket cannot be retrieved. err=-3


{"epoch": 1, "step": 1, "steps": 1, "loss": 8.322295188903809, "codec_loss": 8.080169677734375, "batch_loss_per_codebook": [7.896210670471191, 8.091974258422852, 8.227667808532715, 8.332813262939453, 8.2218017578125, 8.146409034729004, 8.349720001220703, 7.926643371582031], "batch_cb0_loss": 7.896210670471191, "duration_loss": 2.421257495880127, "learning_rate": 6.666666666666667e-06, "frames_per_second_rank0": 237.37558947529243, "peak_vram_gib": 0.2635016441345215}


/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4876: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  warnings.warn(  # warn only once


{"epoch_health": 1, "signal": "yellow", "headline": "baseline epoch recorded; trend signals begin next epoch", "warnings": ["median duration error exceeds 50%"], "codebooks": [{"cb": 0, "signal": "yellow", "tf_ce": 7.8955, "tf_top1": 0.0, "ar_ce": 8.0123, "ar_top1": 0.0, "gap": 0.1168, "ar_trend": "baseline", "usage_ratio": 0.86}, {"cb": 1, "signal": "yellow", "tf_ce": 8.0916, "tf_top1": 0.0, "ar_ce": 7.9798, "ar_top1": 0.0, "gap": -0.1119, "ar_trend": "baseline", "usage_ratio": 1.065}, {"cb": 2, "signal": "yellow", "tf_ce": 8.2275, "tf_top1": 0.0, "ar_ce": 8.3167, "ar_top1": 0.0, "gap": 0.0892, "ar_trend": "baseline", "usage_ratio": 0.935}, {"cb": 3, "signal": "yellow", "tf_ce": 8.3325, "tf_top1": 0.0, "ar_ce": 8.2157, "ar_top1": 0.0, "gap": -0.1168, "ar_trend": "baseline", "usage_ratio": 1.068}, {"cb": 4, "signal": "yellow", "tf_ce": 8.2215, "tf_top1": 0.0, "ar_ce": 8.3966, "ar_top1": 0.0, "gap": 0.175, "ar_trend": "baseline", "usage_ratio": 1.205}, {"cb": 5, "signal": "yellow", "tf_